#Ideal Trigger >>

**T.v.3.3**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

SIM_DURATION = 80
NUM_STEPS = 600
THRESHOLD = 6
PT = 0.3

time_ns = np.linspace(0, SIM_DURATION, NUM_STEPS)

signal_values = np.random.uniform(0, 7, size=NUM_STEPS)


clk_40M = (np.floor((time_ns + 6.25) / 12.5) % 2).astype(int)
clk_400M = (np.floor(time_ns / 1.25) % 2).astype(int)

GATE = ((time_ns % 25) < 2.5).astype(int)

trig_in = (signal_values > THRESHOLD).astype(int)
ext_output_pulse = ((GATE == 1) & (trig_in == 1)).astype(int)

triggered_count = np.sum(trig_in)
idle_count = NUM_STEPS - triggered_count

print(" Running ideal trigger simulation...\n")
time.sleep(0.5)
print(" Simulation complete.\n")
print(f" Total Samples: {NUM_STEPS}")
print(f" Triggered States: {triggered_count}")
print(f" Idle States: {idle_count}\n")

plt.figure(figsize=(12, 6))
plt.step(time_ns, clk_40M * 2 + 8, where='post', color='black', label='clk_40M (rising start)')
plt.step(time_ns, clk_400M * 2 + 6, where='post', color='skyblue', label='clk_400M')
plt.step(time_ns, GATE * 2 + 4, where='post', color='violet', label='GATE (2.5 ns, repeats)')
plt.step(time_ns, trig_in * 2 + 2, where='post', color='orange', label='trig_in')
plt.step(time_ns, ext_output_pulse * 2, where='post', color='red', label='ext_output_pulse')

gate_period = 25
gate_high = 2.5
for start in np.arange(0, SIM_DURATION, gate_period):
    plt.axvspan(start, start + gate_high, color='violet', alpha=0.2)

plt.title("Ideal Trigger Simulation – Repeating GATE (2.5 ns, Highlighted Windows)")
plt.xlabel("Time (ns)")
plt.yticks([])
plt.xlim(0, SIM_DURATION)
plt.ylim(-1, 10)
plt.legend(loc='upper right')
plt.grid(axis='x', linestyle='--', alpha=0.4)
plt.show()


#Gausian_Jitter >>

**J.v.3.2**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

SIM_DURATION = 50
NUM_STEPS = 500
THRESHOLD = 9
PT = 0.3
JITTER_STD = 0.1

t = np.linspace(0, SIM_DURATION, NUM_STEPS)
signal = np.random.uniform(0, 10, NUM_STEPS) + np.random.normal(0, JITTER_STD, NUM_STEPS)

clk_40M = 1 - (np.floor(t / 12.5) % 2).astype(int)
clk_400M = 1 - (np.floor(t / 1.25) % 2).astype(int)

GATE = ((t % 25) < 2.5).astype(int)

ext_input_pulse = (signal > THRESHOLD).astype(int)
ext_output_pulse = ((GATE == 1) & (ext_input_pulse == 1)).astype(int)

print("Running ideal trigger simulation with Gaussian jitter...\n")
time.sleep(0.3)
print("Simulation complete.\n")
print(f"Total Samples: {NUM_STEPS}")
print(f"Triggered States: {np.sum(ext_input_pulse)}")
print(f"Idle States: {NUM_STEPS - np.sum(ext_input_pulse)}\n")

plt.figure(figsize=(10, 6))
plt.step(t, clk_40M * 2 + 8, where='post', color='black', label='clk_40M (starts high)')
plt.step(t, clk_400M * 2 + 6, where='post', color='skyblue', label='clk_400M')
plt.step(t, GATE * 2 + 4, where='post', color='violet', label='GATE (repeats each 40MHz cycle)')
plt.step(t, ext_input_pulse * 2 + 2, where='post', color='orange', label='ext_input_pulse (jittered)')
plt.step(t, ext_output_pulse * 2, where='post', color='red', label='ext_output_pulse')

for n in range(int(SIM_DURATION // 25) + 1):
    start = n * 25
    plt.axvspan(start, start + 2.5, color='red', alpha=0.2)

plt.title("Ideal Trigger Simulation with Repeating Gate (Synchronized to 40MHz Clock)")
plt.xlabel("Time (ns)")
plt.yticks([])
plt.xlim(0, SIM_DURATION)
plt.ylim(-1, 10)
plt.legend(loc='upper right')
plt.grid(axis='x', linestyle='--', alpha=0.4)
plt.show()


# Side_by_side comparision of Jitter and no Jitter >>

**J.v.3.3.2**

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

#Simulation parameters
SIM_DURATION = 100
NUM_STEPS = 500
THRESHOLD = 7.9
JITTER_STD = 0.5  # Gaussian jitter (for signal)

#Time vector
t = np.linspace(0, SIM_DURATION, NUM_STEPS)

#Signal generation
signal_ideal = np.random.uniform(0, 8, NUM_STEPS)
signal_jittered = signal_ideal + np.random.normal(0, JITTER_STD, NUM_STEPS)

#Clock signals
clk_40M = 1 - ((t // 12.5) % 2).astype(int)   # starts high, ends low
clk_400M = 1 - ((t // 1.25) % 2).astype(int)  # 400 MHz reference

#Gate signal: 2.5 ns pulse every 40 MHz cycle
gate_period = 25.0  # ns (40 MHz)
gate_width = 2.5    # ns pulse width
GATE = ((t % gate_period) < gate_width).astype(int)

#Trigger logic
def trigger_logic(signal):
    ext_input_pulse = (signal > THRESHOLD).astype(int)
    return ext_input_pulse, ext_input_pulse & GATE

ext_input_pulse_ideal, ext_output_pulse_ideal = trigger_logic(signal_ideal)
ext_input_pulse_jittered, ext_output_pulse_jittered = trigger_logic(signal_jittered)

#Print simulation summary
print(f"Total Samples: {NUM_STEPS}")
print(f"Triggered (Ideal): {ext_input_pulse_ideal.sum()} | (Jittered): {ext_input_pulse_jittered.sum()}")
print(f"Idle (Ideal): {NUM_STEPS - ext_input_pulse_ideal.sum()} | (Jittered): {NUM_STEPS - ext_input_pulse_jittered.sum()}\n")

#Plotting helper with flat-top triggers
def add_traces(fig, t, clk_40M, clk_400M, GATE, ext_input_pulse, ext_output_pulse, signal, col, showlegend=True, label_suffix=''):
    # Clock 40 MHz
    fig.add_trace(go.Scatter(
        x=t, y=clk_40M*2 + 8, mode='lines', name='clk_40M',
        line_color='black', showlegend=showlegend
    ), 1, col)

    # Clock 400 MHz
    fig.add_trace(go.Scatter(
        x=t, y=clk_400M*2 + 6, mode='lines', name='clk_400M',
        line_color='skyblue', showlegend=showlegend
    ), 1, col)

    # Gate
    fig.add_trace(go.Scatter(
        x=t, y=GATE*2 + 4, mode='lines', name='GATE',
        line_color='violet', showlegend=showlegend
    ), 1, col)

    # Trigger input (flat-top)
    fig.add_trace(go.Scatter(
        x=t, y=ext_input_pulse*2 + 2, mode='lines', name=f'ext_input_pulse{label_suffix}',
        line_color='orange', line_shape='hv',
        text=[f"t={ti:.2f} ns<br>Signal={si:.2f}" for ti, si in zip(t, signal)],
        hoverinfo='text', showlegend=showlegend
    ), 1, col)

    # Trigger output (flat-top)
    fig.add_trace(go.Scatter(
        x=t, y=ext_output_pulse*2, mode='lines', name=f'ext_output_pulse{label_suffix}',
        line_color='red', line_shape='hv',
        text=[f"t={ti:.2f} ns<br>OUT={to}" for ti, to in zip(t, ext_output_pulse)],
        hoverinfo='text', showlegend=showlegend
    ), 1, col)

#Main subplot layout
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Ideal Trigger Simulation", "Trigger Simulation with Gaussian Jitter"),
    shared_yaxes=True
)

# Add waveforms for ideal and jittered cases
add_traces(fig, t, clk_40M, clk_400M, GATE, ext_input_pulse_ideal, ext_output_pulse_ideal, signal_ideal, col=1)
add_traces(fig, t, clk_40M, clk_400M, GATE, ext_input_pulse_jittered, ext_output_pulse_jittered,
           signal_jittered, col=2, showlegend=False, label_suffix=' (jittered)')

# Highlight synchronized trigger windows
for cycle in np.arange(0, SIM_DURATION, 25):
    for c in (1, 2):
        fig.add_vrect(
            x0=cycle, x1=cycle + 2.5, fillcolor='red',
            opacity=0.15, layer='below', line_width=0, col=c
        )

fig.update_layout(
    title="Ideal vs Jittered Trigger Comparison (Flat-Top Waveforms, 2.5 ns GATE per 40 MHz Cycle)",
    xaxis_title="Time (ns)",
    yaxis=dict(showticklabels=False, range=[-1, 10]),
    height=500,
    hovermode='x unified',
    showlegend=True
)

fig.show()


#

# Gaussian _Jitter_ext_input_pulse

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

#Simulation parameters
SIM_DURATION = 100
NUM_STEPS = 500
THRESHOLD = 7.9

#User-defined jitter range
JITTER_MIN = 1
JITTER_MAX = 3
JITTER_STD = np.random.uniform(JITTER_MIN, JITTER_MAX)  # Random jitter for each run

#Time and signals
t = np.linspace(0, SIM_DURATION, NUM_STEPS)
signal_ideal = np.random.uniform(0, 8, NUM_STEPS)
signal_jittered = signal_ideal + np.random.normal(0, JITTER_STD, NUM_STEPS)

#Clock signals
clk_40M = 1 - ((t // 12.5) % 2).astype(int)   # 40 MHz, starts high
clk_400M = 1 - ((t // 1.25) % 2).astype(int)  # 400 MHz reference

#Gate signal: 2.5 ns pulse every 40 MHz cycle
gate_period = 25.0
gate_width = 2.5
GATE = ((t % gate_period) < gate_width).astype(int)

#Trigger logic
def trigger_logic(signal):
    ext_input_pulse = (signal > THRESHOLD).astype(int)
    ext_output_pulse = ext_input_pulse & GATE  # AND between ext_input_pulse and gate
    return ext_input_pulse, ext_output_pulse

ext_input_pulse_ideal, ext_output_pulse_ideal = trigger_logic(signal_ideal)
ext_input_pulse_jittered, ext_output_pulse_jittered = trigger_logic(signal_jittered)

#Jitter measurement
jitter_values = signal_jittered - signal_ideal
resultant_jitter_std = np.std(jitter_values)

print(f"Total Samples: {NUM_STEPS}")
print(f"Triggered (Ideal): {ext_input_pulse_ideal.sum()} | (Jittered): {ext_input_pulse_jittered.sum()}")
print(f"Idle (Ideal): {NUM_STEPS - ext_input_pulse_ideal.sum()} | (Jittered): {NUM_STEPS - ext_input_pulse_jittered.sum()}")
print(f"\nGenerated Jitter σ = {JITTER_STD:.4f} ns | Measured Jitter σ = {resultant_jitter_std:.4f} ns\n")

#Print all individual resultant jitters
#print("Individual Resultant Jitters (Jitter_i = signal_jittered_i - signal_ideal_i):")
#for i, j in enumerate(jitter_values):
    #print(f"  Sample {i+1:03d}:  {j:.4f} ns")


#Plot 1: Ideal vs Jittered Trigger
def add_traces(fig, t, clk_40M, clk_400M, GATE, ext_input_pulse, ext_output_pulse, signal, col, showlegend=True, label_suffix=''):
    fig.add_trace(go.Scatter(x=t, y=clk_40M*2 + 8, mode='lines', name='clk_40M',
                             line_color='black', showlegend=showlegend), 1, col)
    fig.add_trace(go.Scatter(x=t, y=clk_400M*2 + 6, mode='lines', name='clk_400M',
                             line_color='skyblue', showlegend=showlegend), 1, col)
    fig.add_trace(go.Scatter(x=t, y=GATE*2 + 4, mode='lines', name='GATE',
                             line_color='violet', showlegend=showlegend), 1, col)
    fig.add_trace(go.Scatter(
        x=t, y=ext_input_pulse*2 + 2, mode='lines+markers', name=f'ext_input_pulse{label_suffix}',
        line_color='orange', text=[f"t={ti:.2f} ns<br>Signal={si:.2f}" for ti, si in zip(t, signal)],
        hoverinfo='text', showlegend=showlegend
    ), 1, col)
    fig.add_trace(go.Scatter(
        x=t, y=ext_output_pulse*2, mode='lines+markers', name=f'ext_output_pulse{label_suffix}',
        line_color='red', text=[f"t={ti:.2f} ns<br>OUT={to}" for ti, to in zip(t, ext_output_pulse)],
        hoverinfo='text', showlegend=showlegend
    ), 1, col)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Ideal Trigger Simulation", "Trigger Simulation with Gaussian Jitter"),
    shared_yaxes=True
)

add_traces(fig, t, clk_40M, clk_400M, GATE, ext_input_pulse_ideal, ext_output_pulse_ideal, signal_ideal, col=1)
add_traces(fig, t, clk_40M, clk_400M, GATE, ext_input_pulse_jittered, ext_output_pulse_jittered,
           signal_jittered, col=2, showlegend=False, label_suffix=' (jittered)')

# Highlight synchronized trigger windows
for cycle in np.arange(0, SIM_DURATION, 25):
    for c in (1, 2):
        fig.add_vrect(x0=cycle, x1=cycle + 2.5, fillcolor='red',
                      opacity=0.15, layer='below', line_width=0, col=c)

fig.update_layout(
    title=f"Ideal vs Jittered Trigger Comparison<br>(Gate: 2.5 ns @ 40 MHz, Jitter σ={JITTER_STD:.2f} ns)",
    xaxis_title="Time (ns)",
    yaxis=dict(showticklabels=False, range=[-1, 10]),
    height=500,
    hovermode='x unified',
    showlegend=True
)

fig.show()

#Plot 2: Gaussian distribution of jitter
mu = np.mean(jitter_values)
sigma = np.std(jitter_values)
x = np.linspace(mu - 3*sigma, mu + 3*sigma, 100)
gaussian = (1/(sigma * np.sqrt(2*np.pi))) * np.exp(-0.5 * ((x - mu)/sigma)**2)

fig2 = go.Figure()
fig2.add_trace(go.Histogram(x=jitter_values, nbinsx=30, name='Jitter Samples',
                            histnorm='probability density', marker_color='lightblue'))
fig2.add_trace(go.Scatter(x=x, y=gaussian, mode='lines', name='Fitted Gaussian',
                          line_color='red'))

fig2.update_layout(
    title=f"Gaussian Distribution of Measured Jitter (σ = {resultant_jitter_std:.4f} ns)",
    xaxis_title="Jitter Deviation (ns)",
    yaxis_title="Probability Density",
    height=400,
    bargap=0.2
)

fig2.show()


# Gaussian Jitter_Clock_40MHz

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

#Simulation parameters
SIM_DURATION = 100
NUM_STEPS = 500
THRESHOLD = 7.9

#User-defined jitter range
JITTER_MIN = 1
JITTER_MAX = 3
JITTER_STD = np.random.uniform(JITTER_MIN, JITTER_MAX)  # Random jitter (σ) for each run

#Time base
t = np.linspace(0, SIM_DURATION, NUM_STEPS)

#Signal generation (ext_input_pulse remains ideal)
signal_ideal = np.random.uniform(0, 8, NUM_STEPS)
signal_jittered = signal_ideal  # no jitter added to trigger input anymore

#Clock signals
# Ideal 40 MHz clock period = 25 ns (T_high = 12.5 ns)
clk_400M = 1 - ((t // 1.25) % 2).astype(int)  # reference (400 MHz)

# Introduce Gaussian jitter to 40 MHz clock edges
clk_period = 25.0
jitter_noise = np.random.normal(0, JITTER_STD, len(t))
t_jittered = t + jitter_noise  # time-shifted edges due to jitter
clk_40M_jittered = 1 - ((t_jittered // 12.5) % 2).astype(int)
clk_40M_ideal = 1 - ((t // 12.5) % 2).astype(int)

#Gate signal (fixed 2.5 ns pulse every 40 MHz cycle)
gate_period = 25.0
gate_width = 2.5
GATE = ((t % gate_period) < gate_width).astype(int)

#Trigger logic (signal is ideal now)
def trigger_logic(signal):
    ext_input_pulse = (signal > THRESHOLD).astype(int)
    ext_output_pulse = ext_input_pulse & GATE  # AND between ext_input_pulse and gate
    return ext_input_pulse, ext_output_pulse

ext_input_pulse_ideal, ext_output_pulse_ideal = trigger_logic(signal_ideal)
ext_input_pulse_clockjitter, ext_output_pulse_clockjitter = trigger_logic(signal_jittered)

#Jitter measurement (clock edge deviation)
jitter_values = t_jittered - t
resultant_jitter_std = np.std(jitter_values)

#Results summary
print(f"Total Samples: {NUM_STEPS}")
print(f"Triggered (Ideal): {ext_input_pulse_ideal.sum()} | (Clock-jittered): {ext_input_pulse_clockjitter.sum()}")
print(f"Idle (Ideal): {NUM_STEPS - ext_input_pulse_ideal.sum()} | (Clock-jittered): {NUM_STEPS - ext_input_pulse_clockjitter.sum()}")
print(f"\nGenerated Clock Jitter σ = {JITTER_STD:.4f} ns | Measured Jitter σ = {resultant_jitter_std:.4f} ns\n")

#Plot 1: Ideal vs Clock-Jittered Trigger
def add_traces(fig, t, clk_40M, clk_400M, GATE, ext_input_pulse, ext_output_pulse, signal, col, showlegend=True, label_suffix=''):
    fig.add_trace(go.Scatter(x=t, y=clk_40M*2 + 8, mode='lines', name=f'clk_40M{label_suffix}',
                             line_color='black', showlegend=showlegend), 1, col)
    fig.add_trace(go.Scatter(x=t, y=clk_400M*2 + 6, mode='lines', name='clk_400M',
                             line_color='skyblue', showlegend=showlegend), 1, col)
    fig.add_trace(go.Scatter(x=t, y=GATE*2 + 4, mode='lines', name='GATE',
                             line_color='violet', showlegend=showlegend), 1, col)
    fig.add_trace(go.Scatter(
        x=t, y=ext_input_pulse*2 + 2, mode='lines', name=f'ext_input_pulse{label_suffix}',
        line=dict(color='orange', width=4), line_shape='hv', showlegend=showlegend
    ), 1, col)
    fig.add_trace(go.Scatter(
        x=t, y=ext_output_pulse*2, mode='lines', name=f'ext_output_pulse{label_suffix}',
        line=dict(color='red', width=4), line_shape='hv', showlegend=showlegend
    ), 1, col)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Ideal Clock Trigger Simulation", "Trigger Simulation with Clock Gaussian Jitter"),
    shared_yaxes=True
)

add_traces(fig, t, clk_40M_ideal, clk_400M, GATE, ext_input_pulse_ideal, ext_output_pulse_ideal, signal_ideal, col=1)
add_traces(fig, t, clk_40M_jittered, clk_400M, GATE, ext_input_pulse_clockjitter, ext_output_pulse_clockjitter,
           signal_jittered, col=2, showlegend=False, label_suffix=' (jittered)')

# Highlight gate windows
for cycle in np.arange(0, SIM_DURATION, 25):
    for c in (1, 2):
        fig.add_vrect(x0=cycle, x1=cycle + 2.5, fillcolor='red',
                      opacity=0.15, layer='below', line_width=0, col=c)

fig.update_layout(
    title=f"Ideal vs Clock-Jittered Trigger Comparison<br>(Gate: 2.5 ns @ 40 MHz, Clock Jitter σ={JITTER_STD:.2f} ns)",
    xaxis_title="Time (ns)",
    yaxis=dict(showticklabels=False, range=[-1, 10]),
    height=500,
    hovermode='x unified',
    showlegend=True
)
fig.show()

#Plot 2: Gaussian Distribution of Clock Jitter
mu = np.mean(jitter_values)
sigma = np.std(jitter_values)
x = np.linspace(mu - 3*sigma, mu + 3*sigma, 100)
gaussian = (1/(sigma * np.sqrt(2*np.pi))) * np.exp(-0.5 * ((x - mu)/sigma)**2)

fig2 = go.Figure()
fig2.add_trace(go.Histogram(x=jitter_values, nbinsx=30, name='Clock Jitter Samples',
                            histnorm='probability density', marker_color='lightblue'))
fig2.add_trace(go.Scatter(x=x, y=gaussian, mode='lines', name='Fitted Gaussian',
                          line_color='red'))

fig2.update_layout(
    title=f"Gaussian Distribution of Clock Jitter (σ = {resultant_jitter_std:.4f} ns)",
    xaxis_title="Clock Edge Deviation (ns)",
    yaxis_title="Probability Density",
    height=400,
    bargap=0.2
)
fig2.show()


# Gaussian Jitter in Clock 400MHz

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Simulation parameters
SIM_DURATION = 100
NUM_STEPS = 500
THRESHOLD = 7.9

# User-defined jitter range (random σ each run)
JITTER_MIN = 1
JITTER_MAX = 3
JITTER_STD = np.random.uniform(JITTER_MIN, JITTER_MAX)  # Gaussian jitter σ (ns)

# Time and signals
t = np.linspace(0, SIM_DURATION, NUM_STEPS)
signal_ideal = np.random.uniform(0, 8, NUM_STEPS)

# Clock signals
# 40 MHz ideal clock
clk_40M = 1 - ((t // 12.5) % 2).astype(int)

# 400 MHz clock with Gaussian jitter
# Generate time-domain jitter (phase displacement)
jitter_values = np.random.normal(0, JITTER_STD, NUM_STEPS)
t_jittered = t + jitter_values
clk_400M_ideal = 1 - ((t // 1.25) % 2).astype(int)
clk_400M_jittered = 1 - ((t_jittered // 1.25) % 2).astype(int)

# Gate signal (2.5 ns pulse every 25 ns)
gate_period = 25.0
gate_width = 2.5
GATE = ((t % gate_period) < gate_width).astype(int)

# Trigger logic
def trigger_logic(signal):
    ext_input_pulse = (signal > THRESHOLD).astype(int)
    ext_output_pulse = ext_input_pulse & GATE  # AND between input pulse and gate
    return ext_input_pulse, ext_output_pulse

# Compute triggers (same signal, different clocks)
ext_input_pulse_ideal, ext_output_pulse_ideal = trigger_logic(signal_ideal)
ext_input_pulse_jittered, ext_output_pulse_jittered = trigger_logic(signal_ideal)

# Jitter measurement
resultant_jitter_std = np.std(jitter_values)

print("\n--- GAUSSIAN CLOCK JITTER (Applied to 400 MHz Clock) ---")
print(f"Total Samples: {NUM_STEPS}")
print(f"Triggered (Ideal): {ext_input_pulse_ideal.sum()} | (Jittered Clock): {ext_input_pulse_jittered.sum()}")
print(f"Idle (Ideal): {NUM_STEPS - ext_input_pulse_ideal.sum()} | (Jittered Clock): {NUM_STEPS - ext_input_pulse_jittered.sum()}")
print(f"\nGenerated Clock Jitter σ = {JITTER_STD:.4f} ns | Measured Jitter σ = {resultant_jitter_std:.4f} ns\n")

# Plot 1: Ideal vs Jittered Trigger
def add_traces(fig, t, clk_40M, clk_400M, GATE, ext_input_pulse, ext_output_pulse,
               signal, col, showlegend=True, label_suffix=''):
    fig.add_trace(go.Scatter(x=t, y=clk_40M*2 + 8, mode='lines', name='clk_40M',
                             line_color='black', showlegend=showlegend), 1, col)
    fig.add_trace(go.Scatter(x=t, y=clk_400M*2 + 6, mode='lines', name='clk_400M',
                             line_color='skyblue', showlegend=showlegend), 1, col)
    fig.add_trace(go.Scatter(x=t, y=GATE*2 + 4, mode='lines', name='GATE',
                             line_color='violet', showlegend=showlegend), 1, col)
    fig.add_trace(go.Scatter(
        x=t, y=ext_input_pulse*2 + 2, mode='lines+markers',
        name=f'ext_input_pulse{label_suffix}', line_color='orange',
        text=[f"t={ti:.2f} ns<br>Signal={si:.2f}" for ti, si in zip(t, signal)],
        hoverinfo='text', showlegend=showlegend
    ), 1, col)
    fig.add_trace(go.Scatter(
        x=t, y=ext_output_pulse*2, mode='lines+markers',
        name=f'ext_output_pulse{label_suffix}', line_color='red',
        text=[f"t={ti:.2f} ns<br>OUT={to}" for ti, to in zip(t, ext_output_pulse)],
        hoverinfo='text', showlegend=showlegend
    ), 1, col)

# Create figure
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Ideal Clock (400 MHz)", "Gaussian Jittered 400 MHz Clock"),
    shared_yaxes=True
)

# Add traces
add_traces(fig, t, clk_40M, clk_400M_ideal, GATE, ext_input_pulse_ideal, ext_output_pulse_ideal,
           signal_ideal, col=1)
add_traces(fig, t, clk_40M, clk_400M_jittered, GATE, ext_input_pulse_jittered, ext_output_pulse_jittered,
           signal_ideal, col=2, showlegend=False, label_suffix=' (jittered clock)')

# Highlight synchronized gate windows
for cycle in np.arange(0, SIM_DURATION, 25):
    for c in (1, 2):
        fig.add_vrect(x0=cycle, x1=cycle + 2.5, fillcolor='red',
                      opacity=0.15, layer='below', line_width=0, col=c)

fig.update_layout(
    title=f"Ideal vs Gaussian Jittered 400 MHz Clock (σ = {JITTER_STD:.2f} ns)",
    xaxis_title="Time (ns)",
    yaxis=dict(showticklabels=False, range=[-1, 10]),
    height=500,
    hovermode='x unified',
    showlegend=True
)
fig.show()

# Plot 2: Gaussian Jitter Distribution
mu = np.mean(jitter_values)
sigma = np.std(jitter_values)
x = np.linspace(mu - 3*sigma, mu + 3*sigma, 100)
gaussian_pdf = (1/(sigma * np.sqrt(2*np.pi))) * np.exp(-0.5 * ((x - mu)/sigma)**2)

fig2 = go.Figure()
fig2.add_trace(go.Histogram(x=jitter_values, nbinsx=30, name='Clock Jitter Samples',
                            histnorm='probability density', marker_color='lightblue'))
fig2.add_trace(go.Scatter(x=x, y=gaussian_pdf, mode='lines', name='Fitted Gaussian',
                          line_color='red'))

fig2.update_layout(
    title=f"Gaussian Distribution of 400 MHz Clock Jitter (σ = {resultant_jitter_std:.4f} ns)",
    xaxis_title="Clock Edge Jitter (ns)",
    yaxis_title="Probability Density",
    height=400,
    bargap=0.2
)
fig2.show()


#

# Deterministic_Jitter_ext_input_pulse >>

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

#Simulation parameters
SIM_DURATION = 100      # ns
NUM_STEPS = 500
THRESHOLD = 7.9

#Deterministic jitter parameters
JITTER_AMPLITUDE = 2.0   # ns (± peak)
JITTER_FREQ = 0.05        # Hz equivalent periodicity

#Time vector
t = np.linspace(0, SIM_DURATION, NUM_STEPS)

#Core signal generation
signal_ideal = np.random.uniform(0, 8, NUM_STEPS)
jitter_values = JITTER_AMPLITUDE * np.sin(2 * np.pi * JITTER_FREQ * t)
signal_jittered = signal_ideal + jitter_values  # Deterministic jitter applied

#Clock & gate generation (vectorized)
clk_400M = 1 - ((t // 1.25) % 2).astype(int)   # 400 MHz
clk_40M = 1 - ((t // 12.5) % 2).astype(int)    # 40 MHz
GATE = ((t % 25.0) < 2.5).astype(int)          # 2.5 ns pulse per 25 ns

#Trigger logic (single function, reused)
def trigger_logic(signal):
    ext_input_pulse = (signal > THRESHOLD).astype(int)
    ext_output_pulse = ext_input_pulse & GATE
    return ext_input_pulse, ext_output_pulse

ext_input_pulse_ideal, ext_output_pulse_ideal = trigger_logic(signal_ideal)
ext_input_pulse_jittered, ext_output_pulse_jittered = trigger_logic(signal_jittered)

# Metrics
jitter_std = np.std(jitter_values)

#Results summary
print(f"Total Samples: {NUM_STEPS}")
print(f"Triggered (Ideal): {ext_input_pulse_ideal.sum()} | (Deterministic Jittered): {ext_input_pulse_jittered.sum()}")
print(f"Idle (Ideal): {NUM_STEPS - ext_input_pulse_ideal.sum()} | (Jittered): {NUM_STEPS - ext_input_pulse_jittered.sum()}")
print(f"\nDeterministic Jitter Amplitude = ±{JITTER_AMPLITUDE:.2f} ns | Measured σ = {jitter_std:.4f} ns\n")


#Plot helper (reusable for both simulations)
def add_traces(fig, t, clk_40M, clk_400M, GATE, ext_input_pulse, ext_output_pulse, col, label_suffix=''):
    """Add simulation waveforms to subplots efficiently."""
    offsets = [8, 6, 4, 2, 0]
    colors = ['black', 'skyblue', 'violet', 'orange', 'red']
    names = [f'clock_40M{label_suffix}', 'clock_400M', 'GATE',
             f'ext_input_pulse{label_suffix}', f'ext_output_pulse{label_suffix}']
    signals = [clk_40M, clk_400M, GATE, ext_input_pulse, ext_output_pulse]

def add_traces(fig, t, clk_40M, clk_400M, GATE, ext_input_pulse, ext_output_pulse, col, label_suffix=''):
    """Add simulation waveforms to subplots efficiently."""
    offsets = [8, 6, 4, 2, 0]
    colors = ['black', 'skyblue', 'violet', 'orange', 'red']
    names = [f'clock_40M{label_suffix}', 'clock_400M', 'GATE',
             f'ext_input_pulse{label_suffix}', f'ext_output_pulse{label_suffix}']
    signals = [clk_40M, clk_400M, GATE, ext_input_pulse, ext_output_pulse]

    for sig, off, colr, name in zip(signals, offsets, colors, names):
        # Use step lines for digital-style waveforms
        width = 4 if 'trig' in name else 2  # thicker for trigger signals
        fig.add_trace(go.Scatter(
            x=t,
            y=sig*2 + off,
            mode='lines',
            name=name,
            line=dict(color=colr, width=width),
            line_shape='hv',   # flat-top digital look
            showlegend=(col == 1)
        ), 1, col)


#Plot 1: Ideal vs Deterministic Jittered
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Ideal Trigger", "Deterministic Jittered Trigger"),
    shared_yaxes=True
)

add_traces(fig, t, clk_40M, clk_400M, GATE, ext_input_pulse_ideal, ext_output_pulse_ideal, 1)
add_traces(fig, t, clk_40M, clk_400M, GATE, ext_input_pulse_jittered, ext_output_pulse_jittered, 2, label_suffix=' (jittered)')

#Highlight gate windows (single loop)
for cycle in np.arange(0, SIM_DURATION, 25):
    fig.add_vrect(x0=cycle, x1=cycle + 2.5, fillcolor='red', opacity=0.15, layer='below', line_width=0)

fig.update_layout(
    title=f"Deterministic Jitter Simulation (±{JITTER_AMPLITUDE:.1f} ns @ {JITTER_FREQ} Hz)",
    xaxis_title="Time (ns)",
    yaxis=dict(showticklabels=False, range=[-1, 10]),
    height=480, hovermode='x unified'
)
fig.show()

#Plot 2: Jitter waveform + distribution
fig2 = make_subplots(rows=1, cols=2, subplot_titles=("Deterministic Jitter Waveform", "Jitter Distribution"))

#Waveform
fig2.add_trace(go.Scatter(x=t, y=jitter_values, mode='lines', name='Jitter waveform',
                          line_color='orange'), 1, 1)

#Histogram
fig2.add_trace(go.Histogram(x=jitter_values, nbinsx=30, histnorm='probability density',
                            marker_color='lightblue', name='Histogram'), 1, 2)

fig2.update_layout(
    title=f"Deterministic Jitter Visualization (σ = {jitter_std:.4f} ns)",
    xaxis_title="Time / Jitter (ns)",
    yaxis_title="Amplitude / Density",
    height=400, bargap=0.25
)
fig2.show()


#

# Periodic_Jitter_ext_input_pulse

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

#Simulation Parameters
SIM_DURATION = 100
NUM_STEPS = 500
THRESHOLD = 7.9

#Periodic Jitter
JITTER_AMPLITUDE = 2.0  # ns
JITTER_FREQ = 0.05      # Hz-equivalent periodic modulation

t = np.linspace(0, SIM_DURATION, NUM_STEPS)
signal_ideal = np.random.uniform(0, 8, NUM_STEPS)
jitter_values = JITTER_AMPLITUDE * np.sin(2 * np.pi * JITTER_FREQ * t)
signal_jittered = signal_ideal + jitter_values

#Clock & Gate
clk_400M = 1 - ((t // 1.25) % 2).astype(int)
clk_40M = 1 - ((t // 12.5) % 2).astype(int)
GATE = ((t % 25.0) < 2.5).astype(int)

#Trigger Logic
def trigger_logic(signal):
    ext_input_pulse = (signal > THRESHOLD).astype(int)
    ext_output_pulse = ext_input_pulse & GATE
    return ext_input_pulse, ext_output_pulse

ext_input_pulse_ideal, ext_output_pulse_ideal = trigger_logic(signal_ideal)
ext_input_pulse_jit, ext_output_pulse_jit = trigger_logic(signal_jittered)

# Metrics
jitter_std = np.std(jitter_values)

#Results Summary
print("\n--- PERIODIC JITTER ---")
print(f"Total Samples: {NUM_STEPS}")
print(f"Triggered (Ideal): {ext_input_pulse_ideal.sum()} | (Deterministic Jittered): {ext_input_pulse_jit.sum()}")
print(f"Idle (Ideal): {NUM_STEPS - ext_input_pulse_ideal.sum()} | (Jittered): {NUM_STEPS - ext_input_pulse_jit.sum()}")
print(f"\nDeterministic Jitter Amplitude = ±{JITTER_AMPLITUDE:.2f} ns | Measured σ = {jitter_std:.4f} ns\n")


#Plot 1: Ideal vs Jittered Trigger
def add_traces(fig, t, clk40, clk400, GATE, ext_input_pulse, ext_output_pulse, col, label=''):
    signals = [clk40, clk400, GATE, ext_input_pulse, ext_output_pulse]
    colors = ['black', 'skyblue', 'violet', 'orange', 'red']
    labels = ['clk_40M', 'clk_400M', 'GATE', 'ext_input_pulse', 'ext_output_pulse']
def add_traces(fig, t, clk_40M, clk_400M, GATE, ext_input_pulse, ext_output_pulse, col, label_suffix=''):
    """Add simulation waveforms to subplots efficiently."""
    offsets = [8, 6, 4, 2, 0]
    colors = ['black', 'skyblue', 'violet', 'orange', 'red']
    names = [f'clock_40M{label_suffix}', 'clock_400M', 'GATE',
             f'ext_input_pulse{label_suffix}', f'ext_output_pulse{label_suffix}']
    signals = [clk_40M, clk_400M, GATE, ext_input_pulse, ext_output_pulse]

    for sig, off, colr, name in zip(signals, offsets, colors, names):
        # Use step lines for digital-style waveforms
        width = 4 if 'trig' in name else 2  # thicker for trigger signals
        fig.add_trace(go.Scatter(
            x=t,
            y=sig*2 + off,
            mode='lines',
            name=name,
            line=dict(color=colr, width=width),
            line_shape='hv',   # flat-top digital look
            showlegend=(col == 1)
        ), 1, col)


fig = make_subplots(rows=1, cols=2, subplot_titles=("Ideal Trigger", "Periodic Jittered Trigger"), shared_yaxes=True)
add_traces(fig, t, clk_40M, clk_400M, GATE, ext_input_pulse_ideal, ext_output_pulse_ideal, 1)
add_traces(fig, t, clk_40M, clk_400M, GATE, ext_input_pulse_jit, ext_output_pulse_jit, 2, ' (PJ)')

for cycle in np.arange(0, SIM_DURATION, 25):
    fig.add_vrect(x0=cycle, x1=cycle+2.5, fillcolor='red', opacity=0.15, layer='below', line_width=0)

fig.update_layout(title=f"Periodic Jitter Simulation (±{JITTER_AMPLITUDE:.1f} ns @ {JITTER_FREQ} Hz)",
                  xaxis_title="Time (ns)", yaxis=dict(showticklabels=False, range=[-1, 10]),
                  height=480, hovermode='x unified')
fig.show()

#Plot 2: Jitter waveform & distribution
fig2 = make_subplots(rows=1, cols=2, subplot_titles=("Periodic Jitter Waveform", "Jitter Distribution"))
fig2.add_trace(go.Scatter(x=t, y=jitter_values, mode='lines', line_color='orange', name='Jitter waveform'), 1, 1)
fig2.add_trace(go.Histogram(x=jitter_values, nbinsx=30, histnorm='probability density',
                            marker_color='lightblue', name='Histogram'), 1, 2)
fig2.update_layout(title=f"Periodic Jitter Visualization (σ = {jitter_std:.4f} ns)",
                   xaxis_title="Time / Jitter (ns)", yaxis_title="Amplitude / Density",
                   height=400, bargap=0.25)
fig2.show()


# Periodic_jitter_in_Clock_40MHz

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# Simulation Parameters
SIM_DURATION = 100
NUM_STEPS = 500
THRESHOLD = 7.9

# Periodic Jitter (Phase modulation on 40 MHz clock)
JITTER_AMPLITUDE = 2.0  # ns
JITTER_FREQ = 0.05      # Hz-equivalent periodic modulation

# Time Vector
t = np.linspace(0, SIM_DURATION, NUM_STEPS)

# Ideal External Signal (no jitter)
signal_ideal = np.random.uniform(0, 8, NUM_STEPS)
signal_jittered = np.copy(signal_ideal)

# Jitter waveform
jitter_values = JITTER_AMPLITUDE * np.sin(2 * np.pi * JITTER_FREQ * t)


# Clock & Gate Generation
# Ideal Clocks
clk_400M = 1 - ((t // 1.25) % 2).astype(int)
clk_40M = 1 - ((t // 12.5) % 2).astype(int)

# Apply jitter to 40 MHz clock timing (phase jitter)
t_jittered = t + jitter_values  # modulate edge timing
clk_40M_jit = 1 - ((t_jittered // 12.5) % 2).astype(int)

# Gate Signal (2.5 ns active every 25 ns)
GATE = ((t % 25.0) < 2.5).astype(int)

# Trigger Logic
def trigger_logic(signal):
    ext_input_pulse = (signal > THRESHOLD).astype(int)
    ext_output_pulse = ext_input_pulse & GATE
    return ext_input_pulse, ext_output_pulse

# Ideal Trigger
ext_input_pulse_ideal, ext_output_pulse_ideal = trigger_logic(signal_ideal)

# Jittered Trigger (same signal, but jittered 40 MHz clock)
ext_input_pulse_jit, ext_output_pulse_jit = trigger_logic(signal_jittered)

# Metrics
jitter_std = np.std(jitter_values)


# Results Summary
print("\n--- PERIODIC JITTER (Applied to 40 MHz Clock) ---")
print(f"Total Samples: {NUM_STEPS}")
print(f"Triggered (Ideal): {ext_input_pulse_ideal.sum()} | (Jittered): {ext_input_pulse_jit.sum()}")
print(f"Idle (Ideal): {NUM_STEPS - ext_input_pulse_ideal.sum()} | (Jittered): {NUM_STEPS - ext_input_pulse_jit.sum()}")
print(f"\nDeterministic Clock Jitter Amplitude = ±{JITTER_AMPLITUDE:.2f} ns | Measured σ = {jitter_std:.4f} ns\n")


# Helper Function to Add Traces
def add_traces(fig, t, clk40, clk400, GATE, ext_input_pulse, ext_output_pulse, col, label_suffix=''):
    """Add simulation waveforms to subplots efficiently."""
    offsets = [8, 6, 4, 2, 0]
    colors = ['black', 'skyblue', 'violet', 'orange', 'red']
    names = [f'clock_40M{label_suffix}', 'clock_400M', 'GATE',
             f'ext_input_pulse{label_suffix}', f'ext_output_pulse{label_suffix}']
    signals = [clk40, clk400, GATE, ext_input_pulse, ext_output_pulse]

    for sig, off, colr, name in zip(signals, offsets, colors, names):
        width = 4 if 'trig' in name else 2  # thicker for trigger signals
        fig.add_trace(go.Scatter(
            x=t,
            y=sig * 2 + off,
            mode='lines',
            name=name,
            line=dict(color=colr, width=width),
            line_shape='hv',
            showlegend=(col == 1)
        ), 1, col)


# Plot 1: Ideal vs Jittered Trigger
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Ideal 40 MHz Clock", "40 MHz Clock with Periodic Phase Jitter"),
    shared_yaxes=True
)

add_traces(fig, t, clk_40M, clk_400M, GATE, ext_input_pulse_ideal, ext_output_pulse_ideal, 1)
add_traces(fig, t, clk_40M_jit, clk_400M, GATE, ext_input_pulse_jit, ext_output_pulse_jit, 2, ' (PJ)')

# Highlight gate regions
for cycle in np.arange(0, SIM_DURATION, 25):
    fig.add_vrect(x0=cycle, x1=cycle+2.5, fillcolor='red', opacity=0.15, layer='below', line_width=0)

fig.update_layout(
    title=f"Deterministic Periodic Phase Jitter on 40 MHz Clock (±{JITTER_AMPLITUDE:.1f} ns @ {JITTER_FREQ} Hz)",
    xaxis_title="Time (ns)",
    yaxis=dict(showticklabels=False, range=[-1, 10]),
    height=480,
    hovermode='x unified'
)
fig.show()

# Plot 2: Jitter waveform & distribution
fig2 = make_subplots(rows=1, cols=2,
                     subplot_titles=("Applied Phase Jitter Waveform", "Jitter Distribution"))
fig2.add_trace(go.Scatter(x=t, y=jitter_values, mode='lines', line_color='orange', name='Jitter waveform'), 1, 1)
fig2.add_trace(go.Histogram(x=jitter_values, nbinsx=30, histnorm='probability density',
                            marker_color='lightblue', name='Histogram'), 1, 2)
fig2.update_layout(title=f"Clock Phase Jitter Visualization (σ = {jitter_std:.4f} ns)",
                   xaxis_title="Time / Jitter (ns)", yaxis_title="Amplitude / Density",
                   height=400, bargap=0.25)
fig2.show()


# Periodic Jitter Clock 400MHz

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# Simulation Parameters
SIM_DURATION = 100
NUM_STEPS = 500
THRESHOLD = 7.9

# Periodic Jitter (Phase modulation on 400 MHz clock)
JITTER_AMPLITUDE = 2.0  # ns
JITTER_FREQ = 0.05      # Hz-equivalent periodic modulation

# Time Vector
t = np.linspace(0, SIM_DURATION, NUM_STEPS)

# Ideal External Signal (no jitter)
signal_ideal = np.random.uniform(0, 8, NUM_STEPS)
signal_jittered = np.copy(signal_ideal)

# Jitter waveform
jitter_values = JITTER_AMPLITUDE * np.sin(2 * np.pi * JITTER_FREQ * t)


# Clock & Gate Generation
# Ideal 40 MHz clock (no jitter)
clk_40M = 1 - ((t // 12.5) % 2).astype(int)

# Apply jitter to 400 MHz clock timing (phase jitter)
t_jittered = t + jitter_values  # modulate time with periodic jitter
clk_400M_ideal = 1 - ((t // 1.25) % 2).astype(int)
clk_400M_jit = 1 - ((t_jittered // 1.25) % 2).astype(int)

# Gate Signal (2.5 ns active every 25 ns)
GATE = ((t % 25.0) < 2.5).astype(int)

# Trigger Logic
def trigger_logic(signal):
    ext_input_pulse = (signal > THRESHOLD).astype(int)
    ext_output_pulse = ext_input_pulse & GATE
    return ext_input_pulse, ext_output_pulse

# Ideal Trigger
ext_input_pulse_ideal, ext_output_pulse_ideal = trigger_logic(signal_ideal)

# Jittered Trigger (same signal, but 400 MHz clock jittered)
ext_input_pulse_jit, ext_output_pulse_jit = trigger_logic(signal_jittered)

# Metrics
jitter_std = np.std(jitter_values)

# Results Summary
print("\n--- PERIODIC JITTER (Applied to 400 MHz Clock) ---")
print(f"Total Samples: {NUM_STEPS}")
print(f"Triggered (Ideal): {ext_input_pulse_ideal.sum()} | (Jittered): {ext_input_pulse_jit.sum()}")
print(f"Idle (Ideal): {NUM_STEPS - ext_input_pulse_ideal.sum()} | (Jittered): {NUM_STEPS - ext_input_pulse_jit.sum()}")
print(f"\nDeterministic Clock Jitter Amplitude = ±{JITTER_AMPLITUDE:.2f} ns | Measured σ = {jitter_std:.4f} ns\n")

# Helper Function to Add Traces
def add_traces(fig, t, clk40, clk400, GATE, ext_input_pulse, ext_output_pulse, col, label_suffix=''):
    """Add simulation waveforms to subplots efficiently."""
    offsets = [8, 6, 4, 2, 0]
    colors = ['black', 'skyblue', 'violet', 'orange', 'red']
    names = [f'clock_40M{label_suffix}', f'clock_400M{label_suffix}', 'GATE',
             f'ext_input_pulse{label_suffix}', f'ext_output_pulse{label_suffix}']
    signals = [clk40, clk400, GATE, ext_input_pulse, ext_output_pulse]

    for sig, off, colr, name in zip(signals, offsets, colors, names):
        width = 4 if 'trig' in name else 2  # thicker for trigger signals
        fig.add_trace(go.Scatter(
            x=t,
            y=sig * 2 + off,
            mode='lines',
            name=name,
            line=dict(color=colr, width=width),
            line_shape='hv',
            showlegend=(col == 1)
        ), 1, col)

# Plot 1: Ideal vs Jittered 400 MHz Clock
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Ideal 400 MHz Clock", "400 MHz Clock with Periodic Phase Jitter"),
    shared_yaxes=True
)

add_traces(fig, t, clk_40M, clk_400M_ideal, GATE, ext_input_pulse_ideal, ext_output_pulse_ideal, 1)
add_traces(fig, t, clk_40M, clk_400M_jit, GATE, ext_input_pulse_jit, ext_output_pulse_jit, 2, ' (PJ)')

# Highlight gate regions
for cycle in np.arange(0, SIM_DURATION, 25):
    fig.add_vrect(x0=cycle, x1=cycle + 2.5, fillcolor='red', opacity=0.15, layer='below', line_width=0)

fig.update_layout(
    title=f"Deterministic Periodic Phase Jitter on 400 MHz Clock (±{JITTER_AMPLITUDE:.1f} ns @ {JITTER_FREQ} Hz)",
    xaxis_title="Time (ns)",
    yaxis=dict(showticklabels=False, range=[-1, 10]),
    height=480,
    hovermode='x unified'
)
fig.show()

# Plot 2: Jitter waveform & distribution
fig2 = make_subplots(rows=1, cols=2,
                     subplot_titles=("Applied Phase Jitter Waveform", "Jitter Distribution"))

fig2.add_trace(go.Scatter(x=t, y=jitter_values, mode='lines',
                          line_color='orange', name='Jitter waveform'), 1, 1)
fig2.add_trace(go.Histogram(x=jitter_values, nbinsx=30,
                            histnorm='probability density',
                            marker_color='lightblue', name='Histogram'), 1, 2)

fig2.update_layout(title=f"400 MHz Clock Periodic Jitter Visualization (σ = {jitter_std:.4f} ns)",
                   xaxis_title="Time / Jitter (ns)",
                   yaxis_title="Amplitude / Density",
                   height=400, bargap=0.25)
fig2.show()


# Periodic Jitter in all Parameters

*periodic jitter applied to the 40 MHz clock, 400 MHz clock, gate timing, and external trigger timing via a jittered time base.*

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# Simulation Parameters
SIM_DURATION = 100.0     # ns
NUM_STEPS = 500
THRESHOLD = 7.9

# Periodic Jitter Parameters
JITTER_AMPLITUDE = 2.0   # ns (peak)
JITTER_FREQ = 0.05       # Hz-equivalent modulation


# Time Axis
t = np.linspace(0, SIM_DURATION, NUM_STEPS)

# Periodic deterministic jitter (time modulation)
jitter_values = JITTER_AMPLITUDE * np.sin(2 * np.pi * JITTER_FREQ * t)

# Jittered time base
t_jittered = t + jitter_values


# External Analog Signal
signal_ideal = np.random.uniform(0, 8, NUM_STEPS)

# Clock & Gate Generation
# Ideal clocks
clk_400M = 1 - ((t // 1.25) % 2).astype(int)
clk_40M  = 1 - ((t // 12.5) % 2).astype(int)

# Jittered clocks (edge displacement)
clk_400M_jit = 1 - ((t_jittered // 1.25) % 2).astype(int)
clk_40M_jit  = 1 - ((t_jittered // 12.5) % 2).astype(int)

# Ideal gate (25 ns period, 2.5 ns wide)
GATE = ((t % 25.0) < 2.5).astype(int)

# Jittered gate
GATE_jit = ((t_jittered % 25.0) < 2.5).astype(int)

# Trigger Logic
def trigger_logic(signal, gate):
    ext_input_pulse = (signal > THRESHOLD).astype(int)
    ext_output_pulse = ext_input_pulse & gate
    return ext_input_pulse, ext_output_pulse

# Ideal trigger
ext_input_pulse_ideal, ext_output_pulse_ideal = trigger_logic(signal_ideal, GATE)

# Jittered trigger timing
ext_input_pulse_jit, ext_output_pulse_jit = trigger_logic(signal_ideal, GATE_jit)

# Metrics
jitter_std = np.std(jitter_values)

print("\n--- PERIODIC JITTER APPLIED TO CLOCKS & TRIGGER ---")
print(f"Total Samples: {NUM_STEPS}")
print(f"Triggered (Ideal): {ext_input_pulse_ideal.sum()}")
print(f"Triggered (Jittered): {ext_input_pulse_jit.sum()}")
print(f"Deterministic Jitter Amplitude = ±{JITTER_AMPLITUDE:.2f} ns")
print(f"Measured σ = {jitter_std:.4f} ns\n")

# Plot Helpers
def add_traces(fig, t, clk40, clk400, gate, ext_in, ext_out, col, suffix=''):
    offsets = [8, 6, 4, 2, 0]
    colors = ['black', 'skyblue', 'violet', 'orange', 'red']
    names = [
        f'clock_40M{suffix}',
        f'clock_400M{suffix}',
        f'GATE{suffix}',
        f'ext_input_pulse{suffix}',
        f'ext_output_pulse{suffix}'
    ]
    signals = [clk40, clk400, gate, ext_in, ext_out]

    for sig, off, color, name in zip(signals, offsets, colors, names):
        fig.add_trace(
            go.Scatter(
                x=t,
                y=sig * 2 + off,
                mode='lines',
                line=dict(color=color, width=3),
                line_shape='hv',
                name=name,
                showlegend=(col == 1)
            ),
            row=1,
            col=col
        )

# Plot 1: Ideal vs Jittered Timing
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Ideal Timing", "Periodic Jitter Applied"),
    shared_yaxes=True
)

add_traces(
    fig, t,
    clk_40M, clk_400M, GATE,
    ext_input_pulse_ideal, ext_output_pulse_ideal,
    col=1
)

add_traces(
    fig, t,
    clk_40M_jit, clk_400M_jit, GATE_jit,
    ext_input_pulse_jit, ext_output_pulse_jit,
    col=2,
    suffix=" (PJ)"
)

# Highlight gate windows
for cycle in np.arange(0, SIM_DURATION, 25):
    fig.add_vrect(
        x0=cycle,
        x1=cycle + 2.5,
        fillcolor='red',
        opacity=0.12,
        layer='below',
        line_width=0
    )

fig.update_layout(
    title=f"Periodic Jitter Simulation (±{JITTER_AMPLITUDE:.1f} ns @ {JITTER_FREQ} Hz)",
    xaxis_title="Time (ns)",
    yaxis=dict(showticklabels=False, range=[-1, 10]),
    height=500,
    hovermode='x unified'
)

fig.show()

# Plot 2: Jitter Waveform & Distribution
fig2 = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Periodic Jitter Waveform", "Jitter Distribution")
)

fig2.add_trace(
    go.Scatter(
        x=t,
        y=jitter_values,
        mode='lines',
        line_color='orange',
        name='Jitter (ns)'
    ),
    row=1,
    col=1
)

fig2.add_trace(
    go.Histogram(
        x=jitter_values,
        nbinsx=30,
        histnorm='probability density',
        marker_color='lightblue',
        name='Histogram'
    ),
    row=1,
    col=2
)

fig2.update_layout(
    title=f"Periodic Jitter Statistics (σ = {jitter_std:.4f} ns)",
    xaxis_title="Time / Jitter (ns)",
    yaxis_title="Amplitude / Density",
    height=400,
    bargap=0.25
)

fig2.show()


#

# Data_Dependent_jitter_ext_input_pulse

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Simulation parameters
SIM_DURATION = 100
NUM_STEPS = 500
THRESHOLD = 7.9
DDJ_OFFSET = 1.5  # ns offset on each transition

# Time and signal generation
t = np.linspace(0, SIM_DURATION, NUM_STEPS)
signal_ideal = np.random.uniform(0, 8, NUM_STEPS)
signal_jittered = signal_ideal.copy()

# Apply Data Dependent Jitter (alternating offset after each threshold crossing)
crossings = np.where(np.diff((signal_ideal > THRESHOLD).astype(int)) != 0)[0]
for i, c in enumerate(crossings):
    signal_jittered[c:] += (-1)**i * DDJ_OFFSET

# Clocks and Gate signals
clk_400M = 1 - ((t // 1.25) % 2).astype(int)
clk_40M = 1 - ((t // 12.5) % 2).astype(int)
GATE = ((t % 25.0) < 2.5).astype(int)

# Trigger logic
def trigger_logic(signal):
    ext_input_pulse = (signal > THRESHOLD).astype(int)
    ext_output_pulse = ext_input_pulse & GATE
    return ext_input_pulse, ext_output_pulse

ext_input_pulse_ideal, ext_output_pulse_ideal = trigger_logic(signal_ideal)
ext_input_pulse_jit, ext_output_pulse_jit = trigger_logic(signal_jittered)

# Jitter analysis
jitter_values = signal_jittered - signal_ideal
jitter_std = np.std(jitter_values)

#Results Summary
print("\n--- DATA DEPENDENT JITTER ---")
print(f"Total Samples: {NUM_STEPS}")
print(f"Triggered (Ideal): {ext_input_pulse_ideal.sum()} | (Jittered): {ext_input_pulse_jit.sum()}")
print(f"Idle (Ideal): {NUM_STEPS - ext_input_pulse_ideal.sum()} | (Jittered): {NUM_STEPS - ext_input_pulse_jit.sum()}")

# Duty ratio can be useful too:
duty_ideal = 100 * ext_input_pulse_ideal.mean()
duty_jit = 100 * ext_input_pulse_jit.mean()
print(f"Duty Ratio (Ideal): {duty_ideal:.2f}% | (Jittered): {duty_jit:.2f}%")

print(f"\nTransition Offset = ±{DDJ_OFFSET:.2f} ns | Measured σ = {jitter_std:.4f} ns\n")


# PLOT 1: IDEAL vs DDJ TRIGGERS
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Ideal Trigger", "Data Dependent Jittered"),
    shared_yaxes=True
)

#Updated trace function with flat-top & bold triggers
def add_traces(fig, t, clk40, clk400, GATE, ext_input_pulse, ext_output_pulse, col, label=''):
    for i, (sig, c, name) in enumerate(zip(
        [clk40, clk400, GATE, ext_input_pulse, ext_output_pulse],
        ['black', 'skyblue', 'violet', 'orange', 'red'],
        ['clk_40M', 'clk_400M', 'GATE', 'ext_input_pulse', 'ext_output_pulse'])):


        width = 4 if 'trig' in name else 2

        fig.add_trace(go.Scatter(
            x=t,
            y=sig*2 + 8 - 2*i,
            mode='lines',
            line=dict(color=c, width=width),
            line_shape='hv',  #digital-style flat-top
            name=f"{name}{label}",
            showlegend=(col == 1)
        ), 1, col)

# Add both plots
add_traces(fig, t, clk_40M, clk_400M, GATE, ext_input_pulse_ideal, ext_output_pulse_ideal, 1)
add_traces(fig, t, clk_40M, clk_400M, GATE, ext_input_pulse_jit, ext_output_pulse_jit, 2, ' (DDJ)')

# Highlight gate windows
for cycle in np.arange(0, SIM_DURATION, 25):
    fig.add_vrect(x0=cycle, x1=cycle+2.5, fillcolor='red', opacity=0.15, layer='below', line_width=0)

fig.update_layout(
    title="Data Dependent Jitter Simulation",
    xaxis_title="Time (ns)",
    yaxis=dict(showticklabels=False, range=[-1, 10]),
    height=480,
    hovermode='x unified'
)
fig.show()


#PLOT 2: JITTER WAVEFORM + HISTOGRAM
fig2 = make_subplots(rows=1, cols=2, subplot_titles=("DDJ Waveform", "Jitter Distribution"))

# Waveform
fig2.add_trace(go.Scatter(x=t, y=jitter_values, mode='lines', line_color='orange', name='Jitter waveform'), 1, 1)

# Histogram
fig2.add_trace(go.Histogram(x=jitter_values, nbinsx=30, histnorm='probability density',
                            marker_color='lightblue', name='Histogram'), 1, 2)

fig2.update_layout(
    title=f"DDJ Visualization (σ = {jitter_std:.4f} ns)",
    height=400,
    bargap=0.25
)
fig2.show()


# Data Dependent Jitter in Clock 40MHz

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Simulation parameters
SIM_DURATION = 100
NUM_STEPS = 500
THRESHOLD = 7.9
DDJ_OFFSET = 1.5  # ns offset per data-dependent transition

# Time and signal generation
t = np.linspace(0, SIM_DURATION, NUM_STEPS)
signal_ideal = np.random.uniform(0, 8, NUM_STEPS)

# Generate Ideal Clocks & Gate
clk_400M = 1 - ((t // 1.25) % 2).astype(int)
clk_40M = 1 - ((t // 12.5) % 2).astype(int)
GATE = ((t % 25.0) < 2.5).astype(int)

# Apply Data-Dependent Jitter to 40 MHz Clock
crossings = np.where(np.diff((signal_ideal > THRESHOLD).astype(int)) != 0)[0]

# Create jitter timeline — phase modulation per transition
t_jittered = t.copy()
for i, c in enumerate(crossings):
    t_jittered[c:] += (-1)**i * DDJ_OFFSET  # alternate offset direction

# Recalculate 40 MHz clock with data-dependent jitter
clk_40M_jit = 1 - ((t_jittered // 12.5) % 2).astype(int)

# Trigger Logic (same as before)
def trigger_logic(signal):
    ext_input_pulse = (signal > THRESHOLD).astype(int)
    ext_output_pulse = ext_input_pulse & GATE
    return ext_input_pulse, ext_output_pulse

# Ideal trigger outputs
ext_input_pulse_ideal, ext_output_pulse_ideal = trigger_logic(signal_ideal)

# Jittered case uses same input signal, but clock is jittered
ext_input_pulse_jit, ext_output_pulse_jit = trigger_logic(signal_ideal)

# Jitter Analysis
jitter_values = t_jittered - t
jitter_std = np.std(jitter_values)

# Results Summary
print("\n--- DATA DEPENDENT JITTER (Applied to 40 MHz Clock) ---")
print(f"Total Samples: {NUM_STEPS}")
print(f"Triggered (Ideal): {ext_input_pulse_ideal.sum()} | (Jittered): {ext_input_pulse_jit.sum()}")
print(f"Idle (Ideal): {NUM_STEPS - ext_input_pulse_ideal.sum()} | (Jittered): {NUM_STEPS - ext_input_pulse_jit.sum()}")

# Duty ratios
duty_ideal = 100 * ext_input_pulse_ideal.mean()
duty_jit = 100 * ext_input_pulse_jit.mean()
print(f"Duty Ratio (Ideal): {duty_ideal:.2f}% | (Jittered): {duty_jit:.2f}%")

print(f"\nTransition Offset = ±{DDJ_OFFSET:.2f} ns | Measured σ = {jitter_std:.4f} ns\n")

# Helper function to plot traces
def add_traces(fig, t, clk40, clk400, GATE, ext_input_pulse, ext_output_pulse, col, label=''):
    for i, (sig, c, name) in enumerate(zip(
        [clk40, clk400, GATE, ext_input_pulse, ext_output_pulse],
        ['black', 'skyblue', 'violet', 'orange', 'red'],
        ['clk_40M', 'clk_400M', 'GATE', 'ext_input_pulse', 'ext_output_pulse'])):

        width = 4 if 'trig' in name else 2
        fig.add_trace(go.Scatter(
            x=t,
            y=sig*2 + 8 - 2*i,
            mode='lines',
            line=dict(color=c, width=width),
            line_shape='hv',
            name=f"{name}{label}",
            showlegend=(col == 1)
        ), 1, col)

# PLOT 1: Ideal vs DDJ-Affected Clock
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Ideal 40 MHz Clock", "Data-Dependent Jittered Clock"),
    shared_yaxes=True
)

add_traces(fig, t, clk_40M, clk_400M, GATE, ext_input_pulse_ideal, ext_output_pulse_ideal, 1)
add_traces(fig, t, clk_40M_jit, clk_400M, GATE, ext_input_pulse_jit, ext_output_pulse_jit, 2, ' (DDJ)')

# Highlight gate windows
for cycle in np.arange(0, SIM_DURATION, 25):
    fig.add_vrect(x0=cycle, x1=cycle+2.5, fillcolor='red', opacity=0.15, layer='below', line_width=0)

fig.update_layout(
    title="Data-Dependent Jitter Simulation (Applied to 40 MHz Clock)",
    xaxis_title="Time (ns)",
    yaxis=dict(showticklabels=False, range=[-1, 10]),
    height=480,
    hovermode='x unified'
)
fig.show()

# PLOT 2: Jitter Waveform + Histogram
fig2 = make_subplots(rows=1, cols=2, subplot_titles=("DDJ Phase Offset Waveform", "Jitter Distribution"))

fig2.add_trace(go.Scatter(x=t, y=jitter_values, mode='lines', line_color='orange', name='Jitter waveform'), 1, 1)
fig2.add_trace(go.Histogram(x=jitter_values, nbinsx=30, histnorm='probability density',
                            marker_color='lightblue', name='Histogram'), 1, 2)

fig2.update_layout(
    title=f"DDJ Visualization (σ = {jitter_std:.4f} ns)",
    height=400,
    bargap=0.25
)
fig2.show()


# Data Dependent Jitter Clock 400MHz

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Simulation parameters
SIM_DURATION = 100
NUM_STEPS = 500
THRESHOLD = 7.9
DDJ_OFFSET = 1.5  # ns offset per data-dependent transition

# Time and signal generation
t = np.linspace(0, SIM_DURATION, NUM_STEPS)
signal_ideal = np.random.uniform(0, 8, NUM_STEPS)


# Generate Ideal Clocks & Gate
clk_40M = 1 - ((t // 12.5) % 2).astype(int)
clk_400M = 1 - ((t // 1.25) % 2).astype(int)
GATE = ((t % 25.0) < 2.5).astype(int)


# Apply Data-Dependent Jitter to 400 MHz Clock
crossings = np.where(np.diff((signal_ideal > THRESHOLD).astype(int)) != 0)[0]

# Create jitter timeline — alternate offset on each transition
t_jittered = t.copy()
for i, c in enumerate(crossings):
    t_jittered[c:] += (-1)**i * DDJ_OFFSET  # alternate +/– offset

# Recalculate 400 MHz clock with DDJ
clk_400M_jit = 1 - ((t_jittered // 1.25) % 2).astype(int)

# Trigger Logic
def trigger_logic(signal):
    ext_input_pulse = (signal > THRESHOLD).astype(int)
    ext_output_pulse = ext_input_pulse & GATE
    return ext_input_pulse, ext_output_pulse

# Ideal trigger
ext_input_pulse_ideal, ext_output_pulse_ideal = trigger_logic(signal_ideal)

# Jittered case (same input, jittered 400 MHz clock)
ext_input_pulse_jit, ext_output_pulse_jit = trigger_logic(signal_ideal)

# Jitter Analysis
jitter_values = t_jittered - t
jitter_std = np.std(jitter_values)

print("\n--- DATA DEPENDENT JITTER (Applied to 400 MHz Clock) ---")
print(f"Total Samples: {NUM_STEPS}")
print(f"Triggered (Ideal): {ext_input_pulse_ideal.sum()} | (Jittered): {ext_input_pulse_jit.sum()}")
print(f"Idle (Ideal): {NUM_STEPS - ext_input_pulse_ideal.sum()} | (Jittered): {NUM_STEPS - ext_input_pulse_jit.sum()}")

duty_ideal = 100 * ext_input_pulse_ideal.mean()
duty_jit = 100 * ext_input_pulse_jit.mean()
print(f"Duty Ratio (Ideal): {duty_ideal:.2f}% | (Jittered): {duty_jit:.2f}%")
print(f"\nTransition Offset = ±{DDJ_OFFSET:.2f} ns | Measured σ = {jitter_std:.4f} ns\n")

def add_traces(fig, t, clk40, clk400, GATE, ext_input_pulse, ext_output_pulse, col, label=''):
    for i, (sig, c, name) in enumerate(zip(
        [clk40, clk400, GATE, ext_input_pulse, ext_output_pulse],
        ['black', 'skyblue', 'violet', 'orange', 'red'],
        ['clk_40M', 'clk_400M', 'GATE', 'ext_input_pulse', 'ext_output_pulse'])):

        width = 4 if 'trig' in name else 2
        fig.add_trace(go.Scatter(
            x=t,
            y=sig * 2 + 8 - 2 * i,
            mode='lines',
            line=dict(color=c, width=width),
            line_shape='hv',
            name=f"{name}{label}",
            showlegend=(col == 1)
        ), 1, col)


fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Ideal 400 MHz Clock", "Data-Dependent Jittered 400 MHz Clock"),
    shared_yaxes=True
)

add_traces(fig, t, clk_40M, clk_400M, GATE, ext_input_pulse_ideal, ext_output_pulse_ideal, 1)
add_traces(fig, t, clk_40M, clk_400M_jit, GATE, ext_input_pulse_jit, ext_output_pulse_jit, 2, ' (DDJ)')

# Highlight gate regions
for cycle in np.arange(0, SIM_DURATION, 25):
    fig.add_vrect(x0=cycle, x1=cycle + 2.5, fillcolor='red', opacity=0.15, layer='below', line_width=0)

fig.update_layout(
    title="Data-Dependent Jitter Simulation (Applied to 400 MHz Clock)",
    xaxis_title="Time (ns)",
    yaxis=dict(showticklabels=False, range=[-1, 10]),
    height=480,
    hovermode='x unified'
)
fig.show()


fig2 = make_subplots(rows=1, cols=2, subplot_titles=("DDJ Phase Offset Waveform", "Jitter Distribution"))
fig2.add_trace(go.Scatter(x=t, y=jitter_values, mode='lines',
                          line_color='orange', name='Jitter waveform'), 1, 1)
fig2.add_trace(go.Histogram(x=jitter_values, nbinsx=30, histnorm='probability density',
                            marker_color='lightblue', name='Histogram'), 1, 2)
fig2.update_layout(
    title=f"400 MHz Clock DDJ Visualization (σ = {jitter_std:.4f} ns)",
    height=400, bargap=0.25
)
fig2.show()


#

# Duty_cycle_Distortion_jitter_ext_input_pulse

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Simulation parameters
SIM_DURATION = 100
NUM_STEPS = 500
THRESHOLD = 7.9
DCD_RATIO = 0.6       # 60% high, 40% low
BIAS_AMPLITUDE = 1.5  # ns bias amplitude

# Time vector and signals
t = np.linspace(0, SIM_DURATION, NUM_STEPS)
signal_ideal = np.random.uniform(0, 8, NUM_STEPS)

# Duty-cycle distortion waveform (bias alternates based on duty ratio)
duty_wave = ((t % 10) < (10 * DCD_RATIO)).astype(float) * 2 - 1
signal_jittered = signal_ideal + duty_wave * BIAS_AMPLITUDE

# Clock and gate generation
clk_400M = 1 - ((t // 1.25) % 2).astype(int)
clk_40M = 1 - ((t // 12.5) % 2).astype(int)
GATE = ((t % 25.0) < 2.5).astype(int)

# Trigger logic
def trigger_logic(signal):
    ext_input_pulse = (signal > THRESHOLD).astype(int)
    ext_output_pulse = ext_input_pulse & GATE
    return ext_input_pulse, ext_output_pulse

# Compute triggers
ext_input_pulse_ideal, ext_output_pulse_ideal = trigger_logic(signal_ideal)
ext_input_pulse_jit, ext_output_pulse_jit = trigger_logic(signal_jittered)

# Jitter metrics
jitter_values = signal_jittered - signal_ideal
jitter_std = np.std(jitter_values)
jitter_min, jitter_max = np.min(jitter_values), np.max(jitter_values)

#Results Summary
print("\n--- DUTY CYCLE DISTORTION ---")
print(f"Total Samples: {NUM_STEPS}")
print(f"Triggered (Ideal): {ext_input_pulse_ideal.sum()} | (DCD Jittered): {ext_input_pulse_jit.sum()}")
print(f"Idle (Ideal): {NUM_STEPS - ext_input_pulse_ideal.sum()} | (DCD Jittered): {NUM_STEPS - ext_input_pulse_jit.sum()}")

duty_ideal = 100 * ext_input_pulse_ideal.mean()
duty_jit = 100 * ext_input_pulse_jit.mean()
print(f"Duty Ratio (Ideal): {duty_ideal:.2f}% | (DCD Jittered): {duty_jit:.2f}%")

print(f"\nDuty Cycle = {DCD_RATIO*100:.0f}% high")
print(f"Measured σ = {jitter_std:.4f} ns")
print(f"Measured Jitter Range: {jitter_min:.2f} ns → {jitter_max:.2f} ns\n")


# PLOT 1: IDEAL vs DCD TRIGGERS
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Ideal Trigger", "Duty Cycle Distorted Trigger"),
    shared_yaxes=True
)

# Updated add_traces for flat-top digital style and thicker trigger lines
def add_traces(fig, t, clk40, clk400, GATE, ext_input_pulse, ext_output_pulse, col, label=''):
    for i, (sig, color, name) in enumerate(zip(
        [clk40, clk400, GATE, ext_input_pulse, ext_output_pulse],
        ['black', 'skyblue', 'violet', 'orange', 'red'],
        ['clk_40M', 'clk_400M', 'GATE', 'ext_input_pulse', 'ext_output_pulse']
    )):
        width = 4 if 'trig' in name else 2  # thicker lines for trigger signals
        fig.add_trace(go.Scatter(
            x=t,
            y=sig * 2 + 8 - 2 * i,
            mode='lines',
            line=dict(color=color, width=width),
            line_shape='hv',  # digital-style step waveforms
            name=f"{name}{label}",
            showlegend=(col == 1)
        ), 1, col)

# Add both ideal and DCD traces
add_traces(fig, t, clk_40M, clk_400M, GATE, ext_input_pulse_ideal, ext_output_pulse_ideal, 1)
add_traces(fig, t, clk_40M, clk_400M, GATE, ext_input_pulse_jit, ext_output_pulse_jit, 2, ' (DCD)')

# Highlight gate windows
for cycle in np.arange(0, SIM_DURATION, 25):
    fig.add_vrect(x0=cycle, x1=cycle + 2.5, fillcolor='red', opacity=0.15, layer='below', line_width=0)

fig.update_layout(
    title="Duty Cycle Distortion Simulation",
    xaxis_title="Time (ns)",
    yaxis=dict(showticklabels=False, range=[-1, 10]),
    height=480,
    hovermode='x unified'
)
fig.show()


# PLOT 2: DCD WAVEFORM + HISTOGRAM
fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=("DCD Waveform", "Jitter Distribution")
)

# Jitter waveform
fig2.add_trace(go.Scatter(
    x=t,
    y=jitter_values,
    mode='lines',
    line_color='orange',
    name='Jitter waveform'
), 1, 1)

# Histogram of jitter
fig2.add_trace(go.Histogram(
    x=jitter_values,
    nbinsx=30,
    histnorm='probability density',
    marker_color='lightblue',
    name='Histogram'
), 1, 2)

fig2.update_layout(
    title=f"DCD Visualization (σ = {jitter_std:.4f} ns)",
    height=400,
    bargap=0.25
)
fig2.show()


# Duty Cycle Distortion Clock 40MHz

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots


SIM_DURATION = 100
NUM_STEPS = 500
THRESHOLD = 7.9
DCD_RATIO = 0.6       # 60% high, 40% low duty cycle distortion
BIAS_AMPLITUDE = 1.5  # ns bias amplitude (visual phase skew scale)


t = np.linspace(0, SIM_DURATION, NUM_STEPS)
signal_ideal = np.random.uniform(0, 8, NUM_STEPS)


clk_400M = 1 - ((t // 1.25) % 2).astype(int)
clk_40M = 1 - ((t // 12.5) % 2).astype(int)
GATE = ((t % 25.0) < 2.5).astype(int)


# Compute period boundaries
period = 25.0  # ns
t_mod = t % period

# High for 60% of period (15 ns), low for 40% (10 ns)
clk_40M_dcd = (t_mod < (period * DCD_RATIO)).astype(int)


duty_wave = ((t % 10) < (10 * DCD_RATIO)).astype(float) * 2 - 1
bias_wave = duty_wave * BIAS_AMPLITUDE


def trigger_logic(signal):
    ext_input_pulse = (signal > THRESHOLD).astype(int)
    ext_output_pulse = ext_input_pulse & GATE
    return ext_input_pulse, ext_output_pulse

# Ideal trigger paths
ext_input_pulse_ideal, ext_output_pulse_ideal = trigger_logic(signal_ideal)

# DCD test (same input pulse, distorted 40 MHz clock)
ext_input_pulse_dcd, ext_output_pulse_dcd = trigger_logic(signal_ideal)


# Duty cycle stats for clock
ideal_duty_cycle = 50.0
actual_duty_cycle = DCD_RATIO * 100
jitter_std = np.std(bias_wave)
jitter_min, jitter_max = np.min(bias_wave), np.max(bias_wave)


print("\n--- DUTY CYCLE DISTORTION (Applied to 40 MHz Clock) ---")
print(f"Total Samples: {NUM_STEPS}")
print(f"Triggered (Ideal): {ext_input_pulse_ideal.sum()} | (DCD): {ext_input_pulse_dcd.sum()}")
print(f"Idle (Ideal): {NUM_STEPS - ext_input_pulse_ideal.sum()} | (DCD): {NUM_STEPS - ext_input_pulse_dcd.sum()}")

duty_ideal = 100 * ext_input_pulse_ideal.mean()
duty_dcd = 100 * ext_input_pulse_dcd.mean()
print(f"Duty Ratio (Ideal): {duty_ideal:.2f}% | (DCD): {duty_dcd:.2f}%")

print(f"\nClock Duty Cycle = {DCD_RATIO*100:.0f}% high / {100 - DCD_RATIO*100:.0f}% low")
print(f"Measured σ = {jitter_std:.4f} ns")
print(f"Measured Jitter Range: {jitter_min:.2f} ns → {jitter_max:.2f} ns\n")


def add_traces(fig, t, clk40, clk400, GATE, ext_input_pulse, ext_output_pulse, col, label=''):
    for i, (sig, color, name) in enumerate(zip(
        [clk40, clk400, GATE, ext_input_pulse, ext_output_pulse],
        ['black', 'skyblue', 'violet', 'orange', 'red'],
        ['clk_40M', 'clk_400M', 'GATE', 'ext_input_pulse', 'ext_output_pulse']
    )):
        width = 4 if 'trig' in name else 2
        fig.add_trace(go.Scatter(
            x=t,
            y=sig * 2 + 8 - 2 * i,
            mode='lines',
            line=dict(color=color, width=width),
            line_shape='hv',
            name=f"{name}{label}",
            showlegend=(col == 1)
        ), 1, col)


fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Ideal 40 MHz Clock", "Duty Cycle Distorted 40 MHz Clock"),
    shared_yaxes=True
)

add_traces(fig, t, clk_40M, clk_400M, GATE, ext_input_pulse_ideal, ext_output_pulse_ideal, 1)
add_traces(fig, t, clk_40M_dcd, clk_400M, GATE, ext_input_pulse_dcd, ext_output_pulse_dcd, 2, ' (DCD)')

# Highlight gate windows
for cycle in np.arange(0, SIM_DURATION, 25):
    fig.add_vrect(x0=cycle, x1=cycle + 2.5, fillcolor='red', opacity=0.15, layer='below', line_width=0)

fig.update_layout(
    title="Duty Cycle Distortion Simulation (Applied to 40 MHz Clock)",
    xaxis_title="Time (ns)",
    yaxis=dict(showticklabels=False, range=[-1, 10]),
    height=480,
    hovermode='x unified'
)
fig.show()


fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Duty Bias Waveform", "Bias Distribution")
)

fig2.add_trace(go.Scatter(
    x=t,
    y=bias_wave,
    mode='lines',
    line_color='orange',
    name='DCD bias waveform'
), 1, 1)

fig2.add_trace(go.Histogram(
    x=bias_wave,
    nbinsx=30,
    histnorm='probability density',
    marker_color='lightblue',
    name='Histogram'
), 1, 2)

fig2.update_layout(
    title=f"Clock Duty Cycle Bias Visualization (σ = {jitter_std:.4f} ns)",
    height=400,
    bargap=0.25
)
fig2.show()


# Duty Cycle Distortion Clock 400MHz

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots


SIM_DURATION = 100      # ns
NUM_STEPS = 500
THRESHOLD = 7.9
DCD_RATIO = 0.6         # 60% high, 40% low duty cycle distortion
BIAS_AMPLITUDE = 1.5    # ns bias amplitude (for visualization only)


t = np.linspace(0, SIM_DURATION, NUM_STEPS)
signal_ideal = np.random.uniform(0, 8, NUM_STEPS)



clk_40M = 1 - ((t // 12.5) % 2).astype(int)
GATE = ((t % 25.0) < 2.5).astype(int)


period_400 = 2.5  # ns
t_mod_400 = t % period_400

# High for 60% of the period, low for 40% → DCD applied
clk_400M_dcd = (t_mod_400 < (period_400 * DCD_RATIO)).astype(int)

# Ideal 400 MHz (50% duty)
clk_400M_ideal = 1 - ((t // 1.25) % 2).astype(int)

# Bias waveform for visualization (not part of clock logic)
duty_wave = ((t % 10) < (10 * DCD_RATIO)).astype(float) * 2 - 1
bias_wave = duty_wave * BIAS_AMPLITUDE


def trigger_logic(signal):
    ext_input_pulse = (signal > THRESHOLD).astype(int)
    ext_output_pulse = ext_input_pulse & GATE
    return ext_input_pulse, ext_output_pulse

# Ideal trigger
ext_input_pulse_ideal, ext_output_pulse_ideal = trigger_logic(signal_ideal)

# DCD jittered (same signal, but 400 MHz clock is distorted)
ext_input_pulse_dcd, ext_output_pulse_dcd = trigger_logic(signal_ideal)


ideal_duty_cycle = 50.0
actual_duty_cycle = DCD_RATIO * 100
jitter_std = np.std(bias_wave)
jitter_min, jitter_max = np.min(bias_wave), np.max(bias_wave)


print("\n--- DUTY CYCLE DISTORTION (Applied to 400 MHz Clock) ---")
print(f"Total Samples: {NUM_STEPS}")
print(f"Triggered (Ideal): {ext_input_pulse_ideal.sum()} | (DCD): {ext_input_pulse_dcd.sum()}")
print(f"Idle (Ideal): {NUM_STEPS - ext_input_pulse_ideal.sum()} | (DCD): {NUM_STEPS - ext_input_pulse_dcd.sum()}")
duty_ideal = 100 * ext_input_pulse_ideal.mean()
duty_dcd = 100 * ext_input_pulse_dcd.mean()
print(f"Duty Ratio (Ideal): {duty_ideal:.2f}% | (DCD): {duty_dcd:.2f}%")
print(f"\n400 MHz Clock Duty Cycle = {DCD_RATIO*100:.0f}% high / {100 - DCD_RATIO*100:.0f}% low")
print(f"Measured σ = {jitter_std:.4f} ns")
print(f"Measured Jitter Range: {jitter_min:.2f} ns → {jitter_max:.2f} ns\n")


def add_traces(fig, t, clk40, clk400, GATE, ext_input_pulse, ext_output_pulse, col, label=''):
    """Adds digital-style traces for visualization."""
    for i, (sig, color, name) in enumerate(zip(
        [clk40, clk400, GATE, ext_input_pulse, ext_output_pulse],
        ['black', 'skyblue', 'violet', 'orange', 'red'],
        ['clk_40M', 'clk_400M', 'GATE', 'ext_input_pulse', 'ext_output_pulse']
    )):
        width = 4 if 'trig' in name else 2
        fig.add_trace(go.Scatter(
            x=t, y=sig * 2 + 8 - 2 * i,
            mode='lines',
            line=dict(color=color, width=width),
            line_shape='hv',
            name=f"{name}{label}",
            showlegend=(col == 1)
        ), 1, col)


fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Ideal 400 MHz Clock", "400 MHz Clock with Duty Cycle Distortion"),
    shared_yaxes=True
)

add_traces(fig, t, clk_40M, clk_400M_ideal, GATE, ext_input_pulse_ideal, ext_output_pulse_ideal, 1)
add_traces(fig, t, clk_40M, clk_400M_dcd, GATE, ext_input_pulse_dcd, ext_output_pulse_dcd, 2, ' (DCD)')

# Highlight gate regions
for cycle in np.arange(0, SIM_DURATION, 25):
    fig.add_vrect(x0=cycle, x1=cycle + 2.5, fillcolor='red', opacity=0.15, layer='below', line_width=0)

fig.update_layout(
    title="Duty Cycle Distortion Applied to 400 MHz Clock",
    xaxis_title="Time (ns)",
    yaxis=dict(showticklabels=False, range=[-1, 10]),
    height=480,
    hovermode='x unified'
)
fig.show()


fig2 = make_subplots(rows=1, cols=2, subplot_titles=("Duty Bias Waveform", "Bias Distribution"))
fig2.add_trace(go.Scatter(x=t, y=bias_wave, mode='lines', line_color='orange', name='DCD Bias Waveform'), 1, 1)
fig2.add_trace(go.Histogram(
    x=bias_wave, nbinsx=30, histnorm='probability density',
    marker_color='lightblue', name='Histogram'
), 1, 2)

fig2.update_layout(
    title=f"400 MHz Clock Duty Cycle Bias Visualization (σ = {jitter_std:.4f} ns)",
    height=400,
    bargap=0.25
)
fig2.show()


#

# Bounded_uncorelated_Jitter_ext_input_pulse

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

#Simulation Parameters
SIM_DURATION = 100      # ns
NUM_STEPS = 500
THRESHOLD = 7.9
BUJ_RANGE = 2.0         # ±2 ns bounded uncorrelated jitter range

#Time and Signals
t = np.linspace(0, SIM_DURATION, NUM_STEPS)
signal_ideal = np.random.uniform(0, 8, NUM_STEPS)

# Random bounded uncorrelated jitter
jitter_values = np.random.uniform(-BUJ_RANGE, BUJ_RANGE, NUM_STEPS)
signal_jittered = signal_ideal + jitter_values

#Clock & Gate Generation
clk_400M = 1 - ((t // 1.25) % 2).astype(int)
clk_40M = 1 - ((t // 12.5) % 2).astype(int)
GATE = ((t % 25.0) < 2.5).astype(int)

#Trigger Logic
def trigger_logic(signal):
    ext_input_pulse = (signal > THRESHOLD).astype(int)
    ext_output_pulse = ext_input_pulse & GATE
    return ext_input_pulse, ext_output_pulse

# Compute triggers
ext_input_pulse_ideal, ext_output_pulse_ideal = trigger_logic(signal_ideal)
ext_input_pulse_jit, ext_output_pulse_jit = trigger_logic(signal_jittered)

#Metrics
jitter_std = np.std(jitter_values)
jitter_min, jitter_max = np.min(jitter_values), np.max(jitter_values)

#Results Summary
print("\n--- BOUNDED UNCORRELATED JITTER ---")
print(f"Total Samples: {NUM_STEPS}")
print(f"Triggered (Ideal): {ext_input_pulse_ideal.sum()} | (BUJ Jittered): {ext_input_pulse_jit.sum()}")
print(f"Idle (Ideal): {NUM_STEPS - ext_input_pulse_ideal.sum()} | (BUJ Jittered): {NUM_STEPS - ext_input_pulse_jit.sum()}")

duty_ideal = 100 * ext_input_pulse_ideal.mean()
duty_jit = 100 * ext_input_pulse_jit.mean()
print(f"Duty Ratio (Ideal): {duty_ideal:.2f}% | (BUJ Jittered): {duty_jit:.2f}%")

print(f"\nBound Range = ±{BUJ_RANGE:.2f} ns")
print(f"Measured σ = {jitter_std:.4f} ns")
print(f"Measured Jitter Range: {jitter_min:.2f} ns → {jitter_max:.2f} ns\n")


# PLOT 1: Ideal vs Bounded Uncorrelated Jittered Simulation
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Ideal Trigger", "Bounded Uncorrelated Jittered Trigger"),
    shared_yaxes=True
)

# --- Improved digital-style add_traces (flat-top + thick trigger lines) ---
def add_traces(fig, t, clk40, clk400, GATE, ext_input_pulse, ext_output_pulse, col, label=''):
    for i, (sig, color, name) in enumerate(zip(
        [clk40, clk400, GATE, ext_input_pulse, ext_output_pulse],
        ['black', 'skyblue', 'violet', 'orange', 'red'],
        ['clk_40M', 'clk_400M', 'GATE', 'ext_input_pulse', 'ext_output_pulse']
    )):
        width = 4 if 'trig' in name else 2  # 🔹 Make trigger lines thicker
        fig.add_trace(go.Scatter(
            x=t,
            y=sig * 2 + 8 - 2 * i,
            mode='lines',
            line=dict(color=color, width=width),
            line_shape='hv',  # 🔹 Flat-top digital waveform
            name=f"{name}{label}",
            showlegend=(col == 1)
        ), 1, col)

# Add traces
add_traces(fig, t, clk_40M, clk_400M, GATE, ext_input_pulse_ideal, ext_output_pulse_ideal, 1)
add_traces(fig, t, clk_40M, clk_400M, GATE, ext_input_pulse_jit, ext_output_pulse_jit, 2, ' (BUJ)')

# Highlight gate windows
for cycle in np.arange(0, SIM_DURATION, 25):
    fig.add_vrect(x0=cycle, x1=cycle + 2.5, fillcolor='red', opacity=0.15, layer='below', line_width=0)

fig.update_layout(
    title="Bounded Uncorrelated Jitter Simulation",
    xaxis_title="Time (ns)",
    yaxis=dict(showticklabels=False, range=[-1, 10]),
    height=480,
    hovermode='x unified'
)
fig.show()


# PLOT 2: BUJ Waveform and Jitter Distribution
fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=("BUJ Waveform", "Jitter Distribution")
)

# Jitter waveform
fig2.add_trace(go.Scatter(
    x=t,
    y=jitter_values,
    mode='lines',
    line_color='orange',
    name='Jitter waveform'
), 1, 1)

# Histogram (PDF normalized)
fig2.add_trace(go.Histogram(
    x=jitter_values,
    nbinsx=30,
    histnorm='probability density',
    marker_color='lightblue',
    name='Histogram'
), 1, 2)

fig2.update_layout(
    title=f"BUJ Visualization (σ = {jitter_std:.4f} ns)",
    height=400,
    bargap=0.25
)
fig2.show()


# Bounded Uncorrelated Jitter Clock 40MHz

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Simulation Parameters
SIM_DURATION = 100      # ns
NUM_STEPS = 500
THRESHOLD = 7.9
BUJ_RANGE = 2.0         # ±2 ns bounded uncorrelated jitter range

# Time and Signals
t = np.linspace(0, SIM_DURATION, NUM_STEPS)
signal_ideal = np.random.uniform(0, 8, NUM_STEPS)

# Clock & Gate Generation
clk_400M = 1 - ((t // 1.25) % 2).astype(int)
clk_40M_ideal = 1 - ((t // 12.5) % 2).astype(int)
GATE = ((t % 25.0) < 2.5).astype(int)


jitter_values = np.random.uniform(-BUJ_RANGE, BUJ_RANGE, NUM_STEPS)

# Apply jitter to time vector → jittered clock edges
t_jittered = t + jitter_values

# Recompute 40 MHz clock using jittered time base
clk_40M_jittered = 1 - ((t_jittered // 12.5) % 2).astype(int)

# Trigger Logic
def trigger_logic(signal):
    ext_input_pulse = (signal > THRESHOLD).astype(int)
    ext_output_pulse = ext_input_pulse & GATE
    return ext_input_pulse, ext_output_pulse

# Ideal trigger
ext_input_pulse_ideal, ext_output_pulse_ideal = trigger_logic(signal_ideal)

# Jittered clock (same signal)
ext_input_pulse_jit, ext_output_pulse_jit = trigger_logic(signal_ideal)

# Metrics
jitter_std = np.std(jitter_values)
jitter_min, jitter_max = np.min(jitter_values), np.max(jitter_values)


print("\n--- BOUNDED UNCORRELATED JITTER (Applied to 40 MHz Clock) ---")
print(f"Total Samples: {NUM_STEPS}")
print(f"Triggered (Ideal): {ext_input_pulse_ideal.sum()} | (BUJ Clock): {ext_input_pulse_jit.sum()}")
print(f"Idle (Ideal): {NUM_STEPS - ext_input_pulse_ideal.sum()} | (BUJ Clock): {NUM_STEPS - ext_input_pulse_jit.sum()}")

duty_ideal = 100 * ext_input_pulse_ideal.mean()
duty_jit = 100 * ext_input_pulse_jit.mean()
print(f"Duty Ratio (Ideal): {duty_ideal:.2f}% | (BUJ Clock): {duty_jit:.2f}%")

print(f"\nBounded Jitter Range = ±{BUJ_RANGE:.2f} ns")
print(f"Measured σ = {jitter_std:.4f} ns")
print(f"Measured Jitter Range: {jitter_min:.2f} ns → {jitter_max:.2f} ns\n")


def add_traces(fig, t, clk40, clk400, GATE, ext_input_pulse, ext_output_pulse, col, label=''):
    for i, (sig, color, name) in enumerate(zip(
        [clk40, clk400, GATE, ext_input_pulse, ext_output_pulse],
        ['black', 'skyblue', 'violet', 'orange', 'red'],
        ['clk_40M', 'clk_400M', 'GATE', 'ext_input_pulse', 'ext_output_pulse']
    )):
        width = 4 if 'trig' in name else 2  # thicker for trigger signals
        fig.add_trace(go.Scatter(
            x=t,
            y=sig * 2 + 8 - 2 * i,
            mode='lines',
            line=dict(color=color, width=width),
            line_shape='hv',
            name=f"{name}{label}",
            showlegend=(col == 1)
        ), 1, col)


fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Ideal 40 MHz Clock", "Bounded Uncorrelated Jittered 40 MHz Clock"),
    shared_yaxes=True
)

add_traces(fig, t, clk_40M_ideal, clk_400M, GATE, ext_input_pulse_ideal, ext_output_pulse_ideal, 1)
add_traces(fig, t, clk_40M_jittered, clk_400M, GATE, ext_input_pulse_jit, ext_output_pulse_jit, 2, ' (BUJ)')

# Highlight gate windows
for cycle in np.arange(0, SIM_DURATION, 25):
    fig.add_vrect(x0=cycle, x1=cycle + 2.5, fillcolor='red', opacity=0.15, layer='below', line_width=0)

fig.update_layout(
    title="Bounded Uncorrelated Jitter Simulation (Applied to 40 MHz Clock)",
    xaxis_title="Time (ns)",
    yaxis=dict(showticklabels=False, range=[-1, 10]),
    height=480,
    hovermode='x unified'
)
fig.show()

fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Random BUJ Waveform", "Jitter Distribution")
)

# Time-domain jitter waveform
fig2.add_trace(go.Scatter(
    x=t,
    y=jitter_values,
    mode='lines',
    line_color='orange',
    name='Jitter waveform'
), 1, 1)

# Jitter histogram (PDF normalized)
fig2.add_trace(go.Histogram(
    x=jitter_values,
    nbinsx=30,
    histnorm='probability density',
    marker_color='lightblue',
    name='Histogram'
), 1, 2)

fig2.update_layout(
    title=f"Random Clock Edge Variation (σ = {jitter_std:.4f} ns)",
    height=400,
    bargap=0.25
)
fig2.show()


# Bounded Uncorrelated Jitter Clock 400MHz

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

SIM_DURATION = 100      # ns
NUM_STEPS = 500
THRESHOLD = 7.9
BUJ_RANGE = 2.0         # ±2 ns bounded uncorrelated jitter range

t = np.linspace(0, SIM_DURATION, NUM_STEPS)
signal_ideal = np.random.uniform(0, 8, NUM_STEPS)


clk_40M = 1 - ((t // 12.5) % 2).astype(int)     # Ideal 40 MHz clock
GATE = ((t % 25.0) < 2.5).astype(int)           # 2.5 ns gate every 25 ns


# Generate random bounded jitter
jitter_values = np.random.uniform(-BUJ_RANGE, BUJ_RANGE, NUM_STEPS)

# Apply jitter to time vector → jittered clock edges
t_jittered = t + jitter_values

# Recompute 400 MHz clock using jittered time base
clk_400M_ideal = 1 - ((t // 1.25) % 2).astype(int)
clk_400M_jittered = 1 - ((t_jittered // 1.25) % 2).astype(int)


def trigger_logic(signal):
    ext_input_pulse = (signal > THRESHOLD).astype(int)
    ext_output_pulse = ext_input_pulse & GATE
    return ext_input_pulse, ext_output_pulse

# Ideal trigger
ext_input_pulse_ideal, ext_output_pulse_ideal = trigger_logic(signal_ideal)

# Jittered clock (same signal)
ext_input_pulse_jit, ext_output_pulse_jit = trigger_logic(signal_ideal)

jitter_std = np.std(jitter_values)
jitter_min, jitter_max = np.min(jitter_values), np.max(jitter_values)

print("\n--- BOUNDED UNCORRELATED JITTER (Applied to 400 MHz Clock) ---")
print(f"Total Samples: {NUM_STEPS}")
print(f"Triggered (Ideal): {ext_input_pulse_ideal.sum()} | (BUJ Clock): {ext_input_pulse_jit.sum()}")
print(f"Idle (Ideal): {NUM_STEPS - ext_input_pulse_ideal.sum()} | (BUJ Clock): {NUM_STEPS - ext_input_pulse_jit.sum()}")

duty_ideal = 100 * ext_input_pulse_ideal.mean()
duty_jit = 100 * ext_input_pulse_jit.mean()
print(f"Duty Ratio (Ideal): {duty_ideal:.2f}% | (BUJ Clock): {duty_jit:.2f}%")

print(f"\nBounded Jitter Range = ±{BUJ_RANGE:.2f} ns")
print(f"Measured σ = {jitter_std:.4f} ns")
print(f"Measured Jitter Range: {jitter_min:.2f} ns → {jitter_max:.2f} ns\n")

def add_traces(fig, t, clk40, clk400, GATE, ext_input_pulse, ext_output_pulse, col, label=''):
    for i, (sig, color, name) in enumerate(zip(
        [clk40, clk400, GATE, ext_input_pulse, ext_output_pulse],
        ['black', 'skyblue', 'violet', 'orange', 'red'],
        ['clk_40M', 'clk_400M', 'GATE', 'ext_input_pulse', 'ext_output_pulse']
    )):
        width = 4 if 'trig' in name else 2
        fig.add_trace(go.Scatter(
            x=t,
            y=sig * 2 + 8 - 2 * i,
            mode='lines',
            line=dict(color=color, width=width),
            line_shape='hv',
            name=f"{name}{label}",
            showlegend=(col == 1)
        ), 1, col)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Ideal 400 MHz Clock", "Bounded Uncorrelated Jittered 400 MHz Clock"),
    shared_yaxes=True
)

add_traces(fig, t, clk_40M, clk_400M_ideal, GATE, ext_input_pulse_ideal, ext_output_pulse_ideal, 1)
add_traces(fig, t, clk_40M, clk_400M_jittered, GATE, ext_input_pulse_jit, ext_output_pulse_jit, 2, ' (BUJ)')

# Highlight gate windows
for cycle in np.arange(0, SIM_DURATION, 25):
    fig.add_vrect(x0=cycle, x1=cycle + 2.5, fillcolor='red', opacity=0.15, layer='below', line_width=0)

fig.update_layout(
    title="Bounded Uncorrelated Jitter Simulation (Applied to 400 MHz Clock)",
    xaxis_title="Time (ns)",
    yaxis=dict(showticklabels=False, range=[-1, 10]),
    height=480,
    hovermode='x unified'
)
fig.show()

# PLOT 2: Jitter Waveform + Histogram
fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Random BUJ Waveform", "Jitter Distribution")
)

# Jitter waveform
fig2.add_trace(go.Scatter(
    x=t, y=jitter_values,
    mode='lines',
    line_color='orange',
    name='Jitter waveform'
), 1, 1)

# Histogram (PDF)
fig2.add_trace(go.Histogram(
    x=jitter_values,
    nbinsx=30,
    histnorm='probability density',
    marker_color='lightblue',
    name='Histogram'
), 1, 2)

fig2.update_layout(
    title=f"400 MHz Clock Random Edge Variation (σ = {jitter_std:.4f} ns)",
    height=400,
    bargap=0.25
)
fig2.show()


#

# **Drive Mounting**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
from google.colab import drive
drive.flush_and_unmount()


# Data Set Generation



1.   Case 1: Clock(Ideal)+External Pulse(Jittered)
2.   Case 2: Clock(Jittered)+External Pulse(Ideal)
3.   Case 3: Clock(Jittered)+External Pulse(Jittered)



#

# **Gaussian Jitter**

## Case 1


### 1.1

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# USER INPUTS
FREQ_MHZ = float(input("Enter clock frequency (MHz): ") or 40.0)   # e.g. 40 MHz
NUM_TRIALS = int(input("Enter number of simulation runs: ") or 10)  # e.g. 10 runs

# Option to test a range of σ values
range_choice = input("Do you want to run across a σ range? (y/n): ").strip().lower()

if range_choice == 'y':
    SIGMA_START = int(input("Enter starting σ value (integer): ") or 1)
    SIGMA_END = int(input("Enter ending σ value (integer): ") or 5)
else:
    single_sigma = float(input("Enter single Gaussian jitter σ (ns): ") or 1.0)
    SIGMA_START = SIGMA_END = int(single_sigma)

# SIMULATION PARAMETERS
SIM_DURATION = 100  # ns
NUM_STEPS = 2000
t = np.linspace(0, SIM_DURATION, NUM_STEPS)
period_ns = 1000 / FREQ_MHZ  # MHz → ns

# FUNCTION TO RUN ONE SIMULATION
def run_one_simulation(t, period_ns, sigma_ns):
    # Ideal pulse (follows ideal clock)
    clk_ideal = ((t % period_ns) < (period_ns / 2)).astype(int)
    ext_pulse_ideal = clk_ideal.copy()

    # Apply Gaussian jitter to the time base
    jitter_values = np.random.normal(0, sigma_ns, len(t))
    t_jittered = t + jitter_values
    ext_pulse_jittered = (((t_jittered % period_ns) < (period_ns / 2))).astype(int)

    # Error detection: 1 when mismatch occurs
    error_signal = np.where(ext_pulse_ideal != ext_pulse_jittered, 1, 0)
    total_errors = np.sum(error_signal)

    return total_errors, ext_pulse_ideal, ext_pulse_jittered, error_signal, jitter_values

# RUN SIMULATIONS FOR EACH SIGMA VALUE
sigma_values = list(range(SIGMA_START, SIGMA_END + 1))
mean_errors_list = []

for sigma in sigma_values:
    error_counts = []
    for i in range(NUM_TRIALS):
        total_errors, ext_pulse_ideal, ext_pulse_jittered, error_signal, jitter_values = run_one_simulation(t, period_ns, sigma)
        error_counts.append(total_errors)
    mean_error = np.mean(error_counts)
    mean_errors_list.append(mean_error)
    print(f"σ = {sigma} ns → Mean Total Errors = {mean_error:.2f}")

# SUMMARY
print("\n--- GAUSSIAN JITTER ERROR SUMMARY ---")
print(f"Clock Frequency     : {FREQ_MHZ:.2f} MHz")
print(f"Number of Runs      : {NUM_TRIALS}")
print(f"σ Range Tested      : {SIGMA_START} to {SIGMA_END}")

# PLOT 1 (last run visualization)
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=("Ideal External Pulse",
                                    "Gaussian Jittered Pulse",
                                    "Error (Difference)"),
                    vertical_spacing=0.05)

fig.add_trace(go.Scatter(x=t, y=ext_pulse_ideal*2,
                         mode='lines', line=dict(color='blue', width=3),
                         line_shape='hv', name='Ideal'), 1, 1)

fig.add_trace(go.Scatter(x=t, y=ext_pulse_jittered*2,
                         mode='lines', line=dict(color='orange', width=3),
                         line_shape='hv', name='Jittered'), 2, 1)

fig.add_trace(go.Scatter(x=t, y=error_signal*2,
                         mode='lines', line=dict(color='red', width=2),
                         line_shape='hv', name='Error'), 3, 1)

fig.update_layout(
    title=f"Gaussian Jitter Simulation — σ={sigma_values[-1]:.2f} ns @ {FREQ_MHZ:.2f} MHz (Last Run)",
    height=650,
    hovermode='x unified',
    showlegend=False,
    xaxis3_title="Time (ns)",
    yaxis=dict(showticklabels=False)
)
fig.show()

# PLOT 2 (σ vs Mean Total Errors)
fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=sigma_values, y=mean_errors_list,
    mode='lines+markers',
    line=dict(color='purple', width=3),
    marker=dict(size=8),
    name='Mean Total Errors'
))

fig2.update_layout(
    title=f"Mean Total Errors vs Gaussian Jitter σ @ {FREQ_MHZ:.2f} MHz",
    xaxis_title="Jitter Standard Deviation σ (ns)",
    yaxis_title="Mean Total Errors",
    height=500,
    template="plotly_white"
)
fig2.show()


In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# USER INPUTS
FREQ_MHZ = float(input("Enter clock frequency (MHz): ") or 40.0)   # e.g. 40 MHz
NUM_TRIALS = int(input("Enter number of simulation runs: ") or 10)  # e.g. 10 runs

# Option to test a range of σ values
range_choice = input("Do you want to run across a σ range? (y/n): ").strip().lower()

if range_choice == 'y':
    SIGMA_START = int(input("Enter starting σ value (integer): ") or 1)
    SIGMA_END = int(input("Enter ending σ value (integer): ") or 5)
else:
    single_sigma = float(input("Enter single Gaussian jitter σ (ns): ") or 1.0)
    SIGMA_START = SIGMA_END = int(single_sigma)

# SIMULATION PARAMETERS
SIM_DURATION = 100  # ns
NUM_STEPS = 2000
t = np.linspace(0, SIM_DURATION, NUM_STEPS)
period_ns = 1000 / FREQ_MHZ  # MHz → ns

# FUNCTION TO RUN ONE SIMULATION
def run_one_simulation(t, period_ns, sigma_ns):
    # Ideal pulse (follows ideal clock)
    clk_ideal = ((t % period_ns) < (period_ns / 2)).astype(int)
    ext_pulse_ideal = clk_ideal.copy()

    # Apply Gaussian jitter to the time base
    jitter_values = np.random.normal(0, sigma_ns, len(t))
    t_jittered = t + jitter_values
    ext_pulse_jittered = (((t_jittered % period_ns) < (period_ns / 2))).astype(int)

    # Error detection: 1 when mismatch occurs
    error_signal = np.where(ext_pulse_ideal != ext_pulse_jittered, 1, 0)
    total_errors = np.sum(error_signal)

    return total_errors, ext_pulse_ideal, ext_pulse_jittered, error_signal, jitter_values

# RUN SIMULATIONS FOR EACH SIGMA VALUE
sigma_values = list(range(SIGMA_START, SIGMA_END + 1))
mean_errors_list = []

for sigma in sigma_values:
    error_counts = []
    for i in range(NUM_TRIALS):
        total_errors, ext_pulse_ideal, ext_pulse_jittered, error_signal, jitter_values = run_one_simulation(t, period_ns, sigma)
        error_counts.append(total_errors)
    mean_error = np.mean(error_counts)
    mean_errors_list.append(mean_error)
    print(f"σ = {sigma} ns → Mean Total Errors = {mean_error:.2f}")

# SUMMARY
print("\n--- GAUSSIAN JITTER ERROR SUMMARY ---")
print(f"Clock Frequency     : {FREQ_MHZ:.2f} MHz")
print(f"Number of Runs      : {NUM_TRIALS}")
print(f"σ Range Tested      : {SIGMA_START} to {SIGMA_END}")

# PLOT 1 (last run visualization)
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=("Ideal External Pulse",
                                    "Gaussian Jittered Pulse",
                                    "Error (Difference)"),
                    vertical_spacing=0.05)

fig.add_trace(go.Scatter(x=t, y=ext_pulse_ideal*2,
                         mode='lines', line=dict(color='blue', width=3),
                         line_shape='hv', name='Ideal'), 1, 1)

fig.add_trace(go.Scatter(x=t, y=ext_pulse_jittered*2,
                         mode='lines', line=dict(color='orange', width=3),
                         line_shape='hv', name='Jittered'), 2, 1)

fig.add_trace(go.Scatter(x=t, y=error_signal*2,
                         mode='lines', line=dict(color='red', width=2),
                         line_shape='hv', name='Error'), 3, 1)

fig.update_layout(
    title=f"Gaussian Jitter Simulation — σ={sigma_values[-1]:.2f} ns @ {FREQ_MHZ:.2f} MHz (Last Run)",
    height=650,
    hovermode='x unified',
    showlegend=False,
    xaxis3_title="Time (ns)",
    yaxis=dict(showticklabels=False)
)
fig.show()

# PLOT 2 (σ vs Mean Total Errors)
fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=sigma_values, y=mean_errors_list,
    mode='lines+markers',
    line=dict(color='purple', width=3),
    marker=dict(size=8),
    name='Mean Total Errors'
))

fig2.update_layout(
    title=f"Mean Total Errors vs Gaussian Jitter σ @ {FREQ_MHZ:.2f} MHz",
    xaxis_title="Jitter Standard Deviation σ (ns)",
    yaxis_title="Mean Total Errors",
    height=500,
    template="plotly_white"
)
fig2.show()


### 1.2

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# USER INPUTS
NUM_FREQS = int(input("Enter number of frequencies to test: ") or 2)
FREQ_LIST = []

for i in range(NUM_FREQS):
    freq = float(input(f"Enter frequency {i+1} (MHz): ") or 40.0)
    FREQ_LIST.append(freq)

NUM_TRIALS = int(input("Enter number of simulation runs per frequency: ") or 10)

# Option to test a range of σ values
range_choice = input("Do you want to run across a σ range? (y/n): ").strip().lower()
if range_choice == 'y':
    SIGMA_START = int(input("Enter starting σ value (integer): ") or 1)
    SIGMA_END = int(input("Enter ending σ value (integer): ") or 5)
else:
    single_sigma = float(input("Enter single Gaussian jitter σ (ns): ") or 1.0)
    SIGMA_START = SIGMA_END = int(single_sigma)

# SIMULATION PARAMETERS
SIM_DURATION = 100  # ns
NUM_STEPS = 2000
t = np.linspace(0, SIM_DURATION, NUM_STEPS)

# FUNCTION TO RUN ONE SIMULATION
def run_one_simulation(t, period_ns, sigma_ns):
    # Ideal pulse
    clk_ideal = ((t % period_ns) < (period_ns / 2)).astype(int)
    ext_pulse_ideal = clk_ideal.copy()

    # Apply Gaussian jitter
    jitter_values = np.random.normal(0, sigma_ns, len(t))
    t_jittered = t + jitter_values
    ext_pulse_jittered = (((t_jittered % period_ns) < (period_ns / 2))).astype(int)

    # Error detection
    error_signal = np.where(ext_pulse_ideal != ext_pulse_jittered, 1, 0)
    total_errors = np.sum(error_signal)

    return total_errors, ext_pulse_ideal, ext_pulse_jittered, error_signal, jitter_values

# MAIN SIMULATION LOOP
sigma_values = list(range(SIGMA_START, SIGMA_END + 1))
all_mean_errors = {}

for freq in FREQ_LIST:
    print(f"\n=== Running simulations for {freq:.2f} MHz ===")
    period_ns = 1000 / freq
    mean_errors_list = []

    for sigma in sigma_values:
        error_counts = []
        for _ in range(NUM_TRIALS):
            total_errors, ext_pulse_ideal, ext_pulse_jittered, error_signal, jitter_values = run_one_simulation(t, period_ns, sigma)
            error_counts.append(total_errors)

        mean_error = np.mean(error_counts)
        mean_errors_list.append(mean_error)
        #print(f"σ = {sigma} ns → Mean Total Errors = {mean_error:.2f}")

    all_mean_errors[freq] = mean_errors_list

# SUMMARY
print("\n--- GAUSSIAN JITTER ERROR SUMMARY ---")
print(f"Frequencies Tested  : {', '.join([f'{f:.2f} MHz' for f in FREQ_LIST])}")
print(f"Number of Runs      : {NUM_TRIALS}")
print(f"σ Range Tested      : {SIGMA_START} to {SIGMA_END}")

# PLOT 1: Last Frequency’s Waveforms (for reference)
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=("Ideal External Pulse",
                                    "Gaussian Jittered Pulse",
                                    "Error (Difference)"),
                    vertical_spacing=0.05)

fig.add_trace(go.Scatter(x=t, y=ext_pulse_ideal*2,
                         mode='lines', line=dict(color='blue', width=3),
                         line_shape='hv', name='Ideal'), 1, 1)

fig.add_trace(go.Scatter(x=t, y=ext_pulse_jittered*2,
                         mode='lines', line=dict(color='orange', width=3),
                         line_shape='hv', name='Jittered'), 2, 1)

fig.add_trace(go.Scatter(x=t, y=error_signal*2,
                         mode='lines', line=dict(color='red', width=2),
                         line_shape='hv', name='Error'), 3, 1)

fig.update_layout(
    title=f"Gaussian Jitter Simulation — σ={sigma_values[-1]:.2f} ns @ {FREQ_LIST[-1]:.2f} MHz (Last Run)",
    height=650,
    hovermode='x unified',
    showlegend=False,
    xaxis3_title="Time (ns)",
    yaxis=dict(showticklabels=False)
)
fig.show()

# PLOT 2: Mean Total Errors vs σ for Each Frequency
fig2 = go.Figure()

colors = ['purple', 'teal', 'orange', 'green', 'red', 'blue']

for idx, freq in enumerate(FREQ_LIST):
    fig2.add_trace(go.Scatter(
        x=sigma_values, y=all_mean_errors[freq],
        mode='lines+markers',
        line=dict(color=colors[idx % len(colors)], width=3),
        marker=dict(size=8),
        name=f'{freq:.2f} MHz'
    ))

fig2.update_layout(
    title="Mean Total Errors vs Gaussian Jitter σ for Multiple Frequencies",
    xaxis_title="Jitter Standard Deviation σ (ns)",
    yaxis_title="Mean Total Errors",
    height=550,
    template="plotly_white",
    legend_title="Frequencies"
)
fig2.show()


### 1.3

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from datetime import datetime

# Set your Google Drive save folder
base_folder = '/content/drive/MyDrive/Jitter_Simulations'
os.makedirs(base_folder, exist_ok=True)

# USER INPUTS
NUM_FREQS = int(input("Enter number of frequencies to test: ") or 2)
FREQ_LIST = []

for i in range(NUM_FREQS):
    freq = float(input(f"Enter frequency {i+1} (MHz): ") or 40.0)
    FREQ_LIST.append(freq)

NUM_TRIALS = int(input("Enter number of simulation runs per frequency: ") or 10)

# Option to test a range of σ values
range_choice = input("Do you want to run across a σ range? (y/n): ").strip().lower()
if range_choice == 'y':
    SIGMA_START = int(input("Enter starting σ value (integer): ") or 1)
    SIGMA_END = int(input("Enter ending σ value (integer): ") or 5)
else:
    single_sigma = float(input("Enter single Gaussian jitter σ (ns): ") or 1.0)
    SIGMA_START = SIGMA_END = int(single_sigma)

# SIMULATION PARAMETERS
SIM_DURATION = 100  # ns
NUM_STEPS = 100000
t = np.linspace(0, SIM_DURATION, NUM_STEPS)

# FUNCTION TO RUN ONE SIMULATION
def run_one_simulation(t, period_ns, sigma_ns):
    # Ideal pulse
    clk_ideal = ((t % period_ns) < (period_ns / 2)).astype(int)
    ext_pulse_ideal = clk_ideal.copy()

    # Apply Gaussian jitter
    jitter_values = np.random.normal(0, sigma_ns, len(t))
    t_jittered = t + jitter_values
    ext_pulse_jittered = (((t_jittered % period_ns) < (period_ns / 2))).astype(int)

    # Error detection
    error_signal = np.where(ext_pulse_ideal != ext_pulse_jittered, 1, 0)
    total_errors = np.sum(error_signal)

    return total_errors, ext_pulse_ideal, ext_pulse_jittered, error_signal, jitter_values

# MAIN SIMULATION LOOP
sigma_values = list(range(SIGMA_START, SIGMA_END + 1))
all_mean_errors = {}
results = []

for freq in FREQ_LIST:
    print(f"\n=== Running simulations for {freq:.2f} MHz ===")
    period_ns = 1000 / freq
    mean_errors_list = []

    for sigma in sigma_values:
        error_counts = []
        for _ in range(NUM_TRIALS):
            total_errors, ext_pulse_ideal, ext_pulse_jittered, error_signal, jitter_values = run_one_simulation(t, period_ns, sigma)
            error_counts.append(total_errors)

        mean_error = np.mean(error_counts)
        mean_errors_list.append(mean_error)
        results.append({
            "Frequency (MHz)": freq,
            "Jitter Std Dev (ns)": sigma,
            "Mean Total Errors": mean_error
        })
        #print(f"σ = {sigma} ns → Mean Total Errors = {mean_error:.2f}")

    all_mean_errors[freq] = mean_errors_list

# SUMMARY
print("\n--- GAUSSIAN JITTER ERROR SUMMARY ---")
print(f"Frequencies Tested  : {', '.join([f'{f:.2f} MHz' for f in FREQ_LIST])}")
print(f"Number of Runs      : {NUM_TRIALS}")
print(f"σ Range Tested      : {SIGMA_START} to {SIGMA_END}")

# EXPORT RESULTS TO DRIVE WITH DATE & TIME STAMP
df = pd.DataFrame(results)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"GaussianJitterSimCase1_{NUM_STEPS}samples_{NUM_TRIALS}runs_{timestamp}.csv"
save_path = os.path.join(base_folder, filename)
df.to_csv(save_path, index=False)

print(f"\n Data exported successfully to:\n{save_path}")

# PLOT 1 — Last Frequency’s Waveforms
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=("Ideal External Pulse",
                                    "Gaussian Jittered Pulse",
                                    "Error (Difference)"),
                    vertical_spacing=0.05)

fig.add_trace(go.Scatter(x=t, y=ext_pulse_ideal*2,
                         mode='lines', line=dict(color='blue', width=3),
                         line_shape='hv', name='Ideal'), 1, 1)

fig.add_trace(go.Scatter(x=t, y=ext_pulse_jittered*2,
                         mode='lines', line=dict(color='orange', width=3),
                         line_shape='hv', name='Jittered'), 2, 1)

fig.add_trace(go.Scatter(x=t, y=error_signal*2,
                         mode='lines', line=dict(color='red', width=2),
                         line_shape='hv', name='Error'), 3, 1)

fig.update_layout(
    title=f"Gaussian Jitter Simulation — σ={sigma_values[-1]:.2f} ns @ {FREQ_LIST[-1]:.2f} MHz (Last Run)",
    height=650,
    hovermode='x unified',
    showlegend=False,
    xaxis3_title="Time (ns)",
    yaxis=dict(showticklabels=False)
)
fig.show()

# PLOT 2 — Mean Total Errors vs σ for Each Frequency
fig2 = go.Figure()
colors = ['purple', 'teal', 'orange', 'green', 'red', 'blue']

for idx, freq in enumerate(FREQ_LIST):
    fig2.add_trace(go.Scatter(
        x=sigma_values, y=all_mean_errors[freq],
        mode='lines+markers',
        line=dict(color=colors[idx % len(colors)], width=3),
        marker=dict(size=8),
        name=f'{freq:.2f} MHz'
    ))

fig2.update_layout(
    title="Mean Total Errors vs Gaussian Jitter σ for Multiple Frequencies",
    xaxis_title="Jitter Standard Deviation σ (ns)",
    yaxis_title="Mean Total Errors",
    height=550,
    template="plotly_white",
    legend_title="Frequencies"
)
fig2.show()


## Case 2

### 2.1

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# USER INPUTS
NUM_FREQS = int(input("Enter number of frequencies to test: ") or 2)
FREQ_LIST = []

for i in range(NUM_FREQS):
    freq = float(input(f"Enter frequency {i+1} (MHz): ") or 40.0)
    FREQ_LIST.append(freq)

NUM_TRIALS = int(input("Enter number of simulation runs per frequency: ") or 10)

# Option to test a range of σ values
range_choice = input("Do you want to run across a σ range? (y/n): ").strip().lower()
if range_choice == 'y':
    SIGMA_START = int(input("Enter starting σ value (integer): ") or 1)
    SIGMA_END = int(input("Enter ending σ value (integer): ") or 5)
else:
    single_sigma = float(input("Enter single Gaussian jitter σ (ns): ") or 1.0)
    SIGMA_START = SIGMA_END = int(single_sigma)

# SIMULATION PARAMETERS
SIM_DURATION = 100  # ns
NUM_STEPS = 2000
t = np.linspace(0, SIM_DURATION, NUM_STEPS)

# FUNCTION TO RUN ONE SIMULATION
def run_one_simulation(t, period_ns, sigma_ns):
    # External pulse (ideal, clean reference)
    ext_pulse_ideal = ((t % period_ns) < (period_ns / 2)).astype(int)

    # Clock — jittered version
    jitter_values = np.random.normal(0, sigma_ns, len(t))
    t_jittered = t + jitter_values
    clk_jittered = (((t_jittered % period_ns) < (period_ns / 2))).astype(int)

    # Error detection: mismatch between ideal external pulse & jittered clock
    error_signal = np.where(ext_pulse_ideal != clk_jittered, 1, 0)
    total_errors = np.sum(error_signal)

    return total_errors, ext_pulse_ideal, clk_jittered, error_signal, jitter_values

# MAIN SIMULATION LOOP
sigma_values = list(range(SIGMA_START, SIGMA_END + 1))
all_mean_errors = {}

for freq in FREQ_LIST:
    print(f"\n=== Running simulations for {freq:.2f} MHz ===")
    period_ns = 1000 / freq
    mean_errors_list = []

    for sigma in sigma_values:
        error_counts = []
        for _ in range(NUM_TRIALS):
            total_errors, ext_pulse_ideal, clk_jittered, error_signal, jitter_values = run_one_simulation(t, period_ns, sigma)
            error_counts.append(total_errors)

        mean_error = np.mean(error_counts)
        mean_errors_list.append(mean_error)
        #print(f"σ = {sigma} ns → Mean Total Errors = {mean_error:.2f}")

    all_mean_errors[freq] = mean_errors_list

# SUMMARY
print("\n--- CASE 2: CLOCK JITTERED / EXTERNAL PULSE IDEAL ---")
print(f"Frequencies Tested  : {', '.join([f'{f:.2f} MHz' for f in FREQ_LIST])}")
print(f"Number of Runs      : {NUM_TRIALS}")
print(f"σ Range Tested      : {SIGMA_START} to {SIGMA_END}")

# PLOT 1 — Last Frequency’s Waveforms (for reference)
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=("Ideal External Pulse",
                                    "Jittered Clock Output",
                                    "Error (Difference)"),
                    vertical_spacing=0.05)

fig.add_trace(go.Scatter(x=t, y=ext_pulse_ideal*2,
                         mode='lines', line=dict(color='blue', width=3),
                         line_shape='hv', name='External Pulse (Ideal)'), 1, 1)

fig.add_trace(go.Scatter(x=t, y=clk_jittered*2,
                         mode='lines', line=dict(color='orange', width=3),
                         line_shape='hv', name='Clock (Jittered)'), 2, 1)

fig.add_trace(go.Scatter(x=t, y=error_signal*2,
                         mode='lines', line=dict(color='red', width=2),
                         line_shape='hv', name='Error'), 3, 1)

fig.update_layout(
    title=f"Case 2 — Clock Jittered vs Ideal External Pulse (σ = {sigma_values[-1]:.2f} ns @ {FREQ_LIST[-1]:.2f} MHz)",
    height=650,
    hovermode='x unified',
    showlegend=False,
    xaxis3_title="Time (ns)",
    yaxis=dict(showticklabels=False)
)
fig.show()

# PLOT 2 — Mean Total Errors vs σ for Each Frequency
fig2 = go.Figure()
colors = ['purple', 'teal', 'orange', 'green', 'red', 'blue']

for idx, freq in enumerate(FREQ_LIST):
    fig2.add_trace(go.Scatter(
        x=sigma_values, y=all_mean_errors[freq],
        mode='lines+markers',
        line=dict(color=colors[idx % len(colors)], width=3),
        marker=dict(size=8),
        name=f'{freq:.2f} MHz'
    ))

fig2.update_layout(
    title="Case 2 — Mean Total Errors vs Gaussian Jitter σ (for Clock Jittered Case)",
    xaxis_title="Jitter Standard Deviation σ (ns)",
    yaxis_title="Mean Total Errors",
    height=550,
    template="plotly_white",
    legend_title="Frequencies"
)
fig2.show()


### 2.2

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# USER INPUTS
NUM_FREQS = int(input("Enter number of frequencies to test: ") or 2)
FREQ_LIST = []

for i in range(NUM_FREQS):
    freq = float(input(f"Enter frequency {i+1} (MHz): ") or 40.0)
    FREQ_LIST.append(freq)

NUM_TRIALS = int(input("Enter number of simulation runs per frequency: ") or 10)

# Option to test a range of σ values
range_choice = input("Do you want to run across a σ range? (y/n): ").strip().lower()
if range_choice == 'y':
    SIGMA_START = int(input("Enter starting σ value (integer): ") or 1)
    SIGMA_END = int(input("Enter ending σ value (integer): ") or 5)
else:
    single_sigma = float(input("Enter single Gaussian jitter σ (ns): ") or 1.0)
    SIGMA_START = SIGMA_END = int(single_sigma)

# SIMULATION PARAMETERS
SIM_DURATION = 100  # ns
NUM_STEPS = 2000
t = np.linspace(0, SIM_DURATION, NUM_STEPS)

# FUNCTION TO RUN ONE SIMULATION
def run_one_simulation(t, period_ns, sigma_ns):
    # External pulse (ideal, clean reference)
    ext_pulse_ideal = ((t % period_ns) < (period_ns / 2)).astype(int)

    # Clock — jittered version
    jitter_values = np.random.normal(0, sigma_ns, len(t))
    t_jittered = t + jitter_values
    clk_jittered = (((t_jittered % period_ns) < (period_ns / 2))).astype(int)

    # Error detection: mismatch between ideal external pulse & jittered clock
    error_signal = np.where(ext_pulse_ideal != clk_jittered, 1, 0)
    total_errors = np.sum(error_signal)

    return total_errors, ext_pulse_ideal, clk_jittered, error_signal, jitter_values

# MAIN SIMULATION LOOP
sigma_values = list(range(SIGMA_START, SIGMA_END + 1))
all_mean_errors = {}

for freq in FREQ_LIST:
    print(f"\n=== Running simulations for {freq:.2f} MHz ===")
    period_ns = 1000 / freq
    mean_errors_list = []

    for sigma in sigma_values:
        error_counts = []
        for _ in range(NUM_TRIALS):
            total_errors, ext_pulse_ideal, clk_jittered, error_signal, jitter_values = run_one_simulation(t, period_ns, sigma)
            error_counts.append(total_errors)

        mean_error = np.mean(error_counts)
        mean_errors_list.append(mean_error)
        #1
        #print(f"σ = {sigma} ns → Mean Total Errors = {mean_error:.2f}")

    all_mean_errors[freq] = mean_errors_list

# SUMMARY
print("\n--- CASE 2: CLOCK JITTERED / EXTERNAL PULSE IDEAL ---")
print(f"Frequencies Tested  : {', '.join([f'{f:.2f} MHz' for f in FREQ_LIST])}")
print(f"Number of Runs      : {NUM_TRIALS}")
print(f"σ Range Tested      : {SIGMA_START} to {SIGMA_END}")

# PLOT 1 — Last Frequency’s Waveforms (for reference)
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=("Ideal External Pulse",
                                    "Jittered Clock Output",
                                    "Error (Difference)"),
                    vertical_spacing=0.05)

fig.add_trace(go.Scatter(x=t, y=ext_pulse_ideal*2,
                         mode='lines', line=dict(color='blue', width=3),
                         line_shape='hv', name='External Pulse (Ideal)'), 1, 1)

fig.add_trace(go.Scatter(x=t, y=clk_jittered*2,
                         mode='lines', line=dict(color='orange', width=3),
                         line_shape='hv', name='Clock (Jittered)'), 2, 1)

fig.add_trace(go.Scatter(x=t, y=error_signal*2,
                         mode='lines', line=dict(color='red', width=2),
                         line_shape='hv', name='Error'), 3, 1)

fig.update_layout(
    title=f"Case 2 — Clock Jittered vs Ideal External Pulse (σ = {sigma_values[-1]:.2f} ns @ {FREQ_LIST[-1]:.2f} MHz)",
    height=650,
    hovermode='x unified',
    showlegend=False,
    xaxis3_title="Time (ns)",
    yaxis=dict(showticklabels=False)
)
fig.show()

# PLOT 2 — Mean Total Errors vs σ for Each Frequency
fig2 = go.Figure()
colors = ['purple', 'teal', 'orange', 'green', 'red', 'blue']

for idx, freq in enumerate(FREQ_LIST):
    fig2.add_trace(go.Scatter(
        x=sigma_values, y=all_mean_errors[freq],
        mode='lines+markers',
        line=dict(color=colors[idx % len(colors)], width=3),
        marker=dict(size=8),
        name=f'{freq:.2f} MHz'
    ))

fig2.update_layout(
    title="Case 2 — Mean Total Errors vs Gaussian Jitter σ (for Clock Jittered Case)",
    xaxis_title="Jitter Standard Deviation σ (ns)",
    yaxis_title="Mean Total Errors",
    height=550,
    template="plotly_white",
    legend_title="Frequencies"
)
fig2.show()


### 2.3

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from datetime import datetime

# Set your Drive save folder here
base_folder = '/content/drive/MyDrive/Jitter_Simulations'
os.makedirs(base_folder, exist_ok=True)

# USER INPUTS
NUM_FREQS = int(input("Enter number of frequencies to test: ") or 2)
FREQ_LIST = []

for i in range(NUM_FREQS):
    freq = float(input(f"Enter frequency {i+1} (MHz): ") or 40.0)
    FREQ_LIST.append(freq)

NUM_TRIALS = int(input("Enter number of simulation runs per frequency: ") or 10)

# Option to test a range of σ values
range_choice = input("Do you want to run across a σ range? (y/n): ").strip().lower()
if range_choice == 'y':
    SIGMA_START = int(input("Enter starting σ value (integer): ") or 1)
    SIGMA_END = int(input("Enter ending σ value (integer): ") or 5)
else:
    single_sigma = float(input("Enter single Gaussian jitter σ (ns): ") or 1.0)
    SIGMA_START = SIGMA_END = int(single_sigma)

# SIMULATION PARAMETERS
SIM_DURATION = 100  # ns
NUM_STEPS = 100000
t = np.linspace(0, SIM_DURATION, NUM_STEPS)

# FUNCTION TO RUN ONE SIMULATION
def run_one_simulation(t, period_ns, sigma_ns):
    ext_pulse_ideal = ((t % period_ns) < (period_ns / 2)).astype(int)
    jitter_values = np.random.normal(0, sigma_ns, len(t))
    t_jittered = t + jitter_values
    clk_jittered = (((t_jittered % period_ns) < (period_ns / 2))).astype(int)
    error_signal = np.where(ext_pulse_ideal != clk_jittered, 1, 0)
    total_errors = np.sum(error_signal)
    return total_errors, ext_pulse_ideal, clk_jittered, error_signal, jitter_values

# MAIN SIMULATION LOOP
sigma_values = list(range(SIGMA_START, SIGMA_END + 1))
all_mean_errors = {}
results = []

for freq in FREQ_LIST:
    print(f"\n=== Running simulations for {freq:.2f} MHz ===")
    period_ns = 1000 / freq
    mean_errors_list = []

    for sigma in sigma_values:
        error_counts = []
        for _ in range(NUM_TRIALS):
            total_errors, ext_pulse_ideal, clk_jittered, error_signal, jitter_values = run_one_simulation(t, period_ns, sigma)
            error_counts.append(total_errors)

        mean_error = np.mean(error_counts)
        mean_errors_list.append(mean_error)
        results.append({
            "Frequency (MHz)": freq,
            "Jitter Std Dev (ns)": sigma,
            "Mean Total Errors": mean_error
        })
        #print(f"σ = {sigma} ns → Mean Total Errors = {mean_error:.2f}")

    all_mean_errors[freq] = mean_errors_list

# SUMMARY
print("\n--- CASE 2: CLOCK JITTERED / EXTERNAL PULSE IDEAL ---")
print(f"Frequencies Tested  : {', '.join([f'{f:.2f} MHz' for f in FREQ_LIST])}")
print(f"Number of Runs      : {NUM_TRIALS}")
print(f"σ Range Tested      : {SIGMA_START} to {SIGMA_END}")

# EXPORT TO DRIVE AS CSV WITH TIMESTAMP
df = pd.DataFrame(results)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"Case2_JitterSim_{NUM_STEPS}samples_{NUM_TRIALS}runs_{timestamp}.csv"
save_path = os.path.join(base_folder, filename)
df.to_csv(save_path, index=False)

print(f"\n Data exported successfully to:\n{save_path}")

# PLOT 1 — Last Frequency’s Waveforms
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=("Ideal External Pulse",
                                    "Jittered Clock Output",
                                    "Error (Difference)"),
                    vertical_spacing=0.05)

fig.add_trace(go.Scatter(x=t, y=ext_pulse_ideal*2,
                         mode='lines', line=dict(color='blue', width=3),
                         line_shape='hv', name='External Pulse (Ideal)'), 1, 1)
fig.add_trace(go.Scatter(x=t, y=clk_jittered*2,
                         mode='lines', line=dict(color='orange', width=3),
                         line_shape='hv', name='Clock (Jittered)'), 2, 1)
fig.add_trace(go.Scatter(x=t, y=error_signal*2,
                         mode='lines', line=dict(color='red', width=2),
                         line_shape='hv', name='Error'), 3, 1)

fig.update_layout(
    title=f"Case 2 — Clock Jittered vs Ideal External Pulse (σ = {sigma_values[-1]:.2f} ns @ {FREQ_LIST[-1]:.2f} MHz)",
    height=650,
    hovermode='x unified',
    showlegend=False,
    xaxis3_title="Time (ns)",
    yaxis=dict(showticklabels=False)
)
fig.show()

# PLOT 2 — Mean Total Errors vs σ for Each Frequency
fig2 = go.Figure()
colors = ['purple', 'teal', 'orange', 'green', 'red', 'blue']

for idx, freq in enumerate(FREQ_LIST):
    fig2.add_trace(go.Scatter(
        x=sigma_values, y=all_mean_errors[freq],
        mode='lines+markers',
        line=dict(color=colors[idx % len(colors)], width=3),
        marker=dict(size=8),
        name=f'{freq:.2f} MHz'
    ))

fig2.update_layout(
    title="Case 2 — Mean Total Errors vs Gaussian Jitter σ (for Clock Jittered Case)",
    xaxis_title="Jitter Standard Deviation σ (ns)",
    yaxis_title="Mean Total Errors",
    height=550,
    template="plotly_white",
    legend_title="Frequencies"
)
fig2.show()


## Case 3

###  3.1

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# USER INPUTS
NUM_FREQS = int(input("Enter number of frequencies to test: ") or 2)
FREQ_LIST = []

for i in range(NUM_FREQS):
    freq = float(input(f"Enter frequency {i+1} (MHz): ") or 40.0)
    FREQ_LIST.append(freq)

NUM_TRIALS = int(input("Enter number of simulation runs per frequency: ") or 10)

# Option to test a range of σ values
range_choice = input("Do you want to run across a σ range? (y/n): ").strip().lower()
if range_choice == 'y':
    SIGMA_START = int(input("Enter starting σ value (integer): ") or 1)
    SIGMA_END = int(input("Enter ending σ value (integer): ") or 5)
else:
    single_sigma = float(input("Enter single Gaussian jitter σ (ns): ") or 1.0)
    SIGMA_START = SIGMA_END = int(single_sigma)

# SIMULATION PARAMETERS
SIM_DURATION = 100  # ns
NUM_STEPS = 2000
t = np.linspace(0, SIM_DURATION, NUM_STEPS)

# FUNCTION TO RUN ONE SIMULATION — BOTH CLOCK AND EXTERNAL PULSE JITTERED
def run_one_simulation(t, period_ns, sigma_ns):
    # Apply independent Gaussian jitter to both signals
    jitter_clk = np.random.normal(0, sigma_ns, len(t))
    jitter_ext = np.random.normal(0, sigma_ns, len(t))

    # Jittered time bases
    t_clk_jittered = t + jitter_clk
    t_ext_jittered = t + jitter_ext

    # Generate signals
    clk_jittered = (((t_clk_jittered % period_ns) < (period_ns / 2))).astype(int)
    ext_pulse_jittered = (((t_ext_jittered % period_ns) < (period_ns / 2))).astype(int)

    # Error detection: mismatch between the two jittered signals
    error_signal = np.where(ext_pulse_jittered != clk_jittered, 1, 0)
    total_errors = np.sum(error_signal)

    return total_errors, clk_jittered, ext_pulse_jittered, error_signal, jitter_clk, jitter_ext

# MAIN SIMULATION LOOP
sigma_values = list(range(SIGMA_START, SIGMA_END + 1))
all_mean_errors = {}

for freq in FREQ_LIST:
    print(f"\n=== Running simulations for {freq:.2f} MHz ===")
    period_ns = 1000 / freq
    mean_errors_list = []

    for sigma in sigma_values:
        error_counts = []
        for _ in range(NUM_TRIALS):
            total_errors, clk_jittered, ext_pulse_jittered, error_signal, jitter_clk, jitter_ext = run_one_simulation(t, period_ns, sigma)
            error_counts.append(total_errors)

        mean_error = np.mean(error_counts)
        mean_errors_list.append(mean_error)
        #print(f"σ = {sigma} ns → Mean Total Errors = {mean_error:.2f}")

    all_mean_errors[freq] = mean_errors_list

# SUMMARY
print("\n--- CASE 3: CLOCK + EXTERNAL PULSE BOTH JITTERED ---")
print(f"Frequencies Tested  : {', '.join([f'{f:.2f} MHz' for f in FREQ_LIST])}")
print(f"Number of Runs      : {NUM_TRIALS}")
print(f"σ Range Tested      : {SIGMA_START} to {SIGMA_END}")

# PLOT 1 — Last Frequency’s Waveforms (for reference)
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=("Jittered External Pulse",
                                    "Jittered Clock Output",
                                    "Error (Difference)"),
                    vertical_spacing=0.05)

fig.add_trace(go.Scatter(x=t, y=ext_pulse_jittered*2,
                         mode='lines', line=dict(color='blue', width=3),
                         line_shape='hv', name='External Pulse (Jittered)'), 1, 1)

fig.add_trace(go.Scatter(x=t, y=clk_jittered*2,
                         mode='lines', line=dict(color='orange', width=3),
                         line_shape='hv', name='Clock (Jittered)'), 2, 1)

fig.add_trace(go.Scatter(x=t, y=error_signal*2,
                         mode='lines', line=dict(color='red', width=2),
                         line_shape='hv', name='Error'), 3, 1)

fig.update_layout(
    title=f"Case 3 — Clock + External Pulse Both Jittered (σ = {sigma_values[-1]:.2f} ns @ {FREQ_LIST[-1]:.2f} MHz)",
    height=650,
    hovermode='x unified',
    showlegend=False,
    xaxis3_title="Time (ns)",
    yaxis=dict(showticklabels=False)
)
fig.show()

# PLOT 2 — Mean Total Errors vs σ for Each Frequency
fig2 = go.Figure()
colors = ['purple', 'teal', 'orange', 'green', 'red', 'blue']

for idx, freq in enumerate(FREQ_LIST):
    fig2.add_trace(go.Scatter(
        x=sigma_values, y=all_mean_errors[freq],
        mode='lines+markers',
        line=dict(color=colors[idx % len(colors)], width=3),
        marker=dict(size=8),
        name=f'{freq:.2f} MHz'
    ))

fig2.update_layout(
    title="Case 3 — Mean Total Errors vs Gaussian Jitter σ (Clock + External Pulse Jittered)",
    xaxis_title="Jitter Standard Deviation σ (ns)",
    yaxis_title="Mean Total Errors",
    height=550,
    template="plotly_white",
    legend_title="Frequencies"
)
fig2.show()


### 3.2

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# USER INPUTS
NUM_FREQS = int(input("Enter number of frequencies to test: ") or 2)
FREQ_LIST = []

for i in range(NUM_FREQS):
    freq = float(input(f"Enter frequency {i+1} (MHz): ") or 40.0)
    FREQ_LIST.append(freq)

NUM_TRIALS = int(input("Enter number of simulation runs per frequency: ") or 10)

# Option to test a range of σ values
range_choice = input("Do you want to run across a σ range? (y/n): ").strip().lower()
if range_choice == 'y':
    SIGMA_START = int(input("Enter starting σ value (integer): ") or 1)
    SIGMA_END = int(input("Enter ending σ value (integer): ") or 5)
else:
    single_sigma = float(input("Enter single Gaussian jitter σ (ns): ") or 1.0)
    SIGMA_START = SIGMA_END = int(single_sigma)

# SIMULATION PARAMETERS
SIM_DURATION = 100  # ns
NUM_STEPS = 2000
t = np.linspace(0, SIM_DURATION, NUM_STEPS)

# FUNCTION TO RUN ONE SIMULATION — BOTH CLOCK AND EXTERNAL PULSE JITTERED
def run_one_simulation(t, period_ns, sigma_ns):
    # Apply independent Gaussian jitter to both signals
    jitter_clk = np.random.normal(0, sigma_ns, len(t))
    jitter_ext = np.random.normal(0, sigma_ns, len(t))

    # Jittered time bases
    t_clk_jittered = t + jitter_clk
    t_ext_jittered = t + jitter_ext

    # Generate signals
    clk_jittered = (((t_clk_jittered % period_ns) < (period_ns / 2))).astype(int)
    ext_pulse_jittered = (((t_ext_jittered % period_ns) < (period_ns / 2))).astype(int)

    # Error detection: mismatch between the two jittered signals
    error_signal = np.where(ext_pulse_jittered != clk_jittered, 1, 0)
    total_errors = np.sum(error_signal)

    return total_errors, clk_jittered, ext_pulse_jittered, error_signal, jitter_clk, jitter_ext

# MAIN SIMULATION LOOP
sigma_values = list(range(SIGMA_START, SIGMA_END + 1))
all_mean_errors = {}

for freq in FREQ_LIST:
    print(f"\n=== Running simulations for {freq:.2f} MHz ===")
    period_ns = 1000 / freq
    mean_errors_list = []

    for sigma in sigma_values:
        error_counts = []
        for _ in range(NUM_TRIALS):
            total_errors, clk_jittered, ext_pulse_jittered, error_signal, jitter_clk, jitter_ext = run_one_simulation(t, period_ns, sigma)
            error_counts.append(total_errors)

        mean_error = np.mean(error_counts)
        mean_errors_list.append(mean_error)
        #print(f"σ = {sigma} ns → Mean Total Errors = {mean_error:.2f}")

    all_mean_errors[freq] = mean_errors_list

# SUMMARY
print("\n--- CASE 3: CLOCK + EXTERNAL PULSE BOTH JITTERED ---")
print(f"Frequencies Tested  : {', '.join([f'{f:.2f} MHz' for f in FREQ_LIST])}")
print(f"Number of Runs      : {NUM_TRIALS}")
print(f"σ Range Tested      : {SIGMA_START} to {SIGMA_END}")

# PLOT 1 — Last Frequency’s Waveforms (for reference)
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=("Jittered External Pulse",
                                    "Jittered Clock Output",
                                    "Error (Difference)"),
                    vertical_spacing=0.05)

fig.add_trace(go.Scatter(x=t, y=ext_pulse_jittered*2,
                         mode='lines', line=dict(color='blue', width=3),
                         line_shape='hv', name='External Pulse (Jittered)'), 1, 1)

fig.add_trace(go.Scatter(x=t, y=clk_jittered*2,
                         mode='lines', line=dict(color='orange', width=3),
                         line_shape='hv', name='Clock (Jittered)'), 2, 1)

fig.add_trace(go.Scatter(x=t, y=error_signal*2,
                         mode='lines', line=dict(color='red', width=2),
                         line_shape='hv', name='Error'), 3, 1)

fig.update_layout(
    title=f"Case 3 — Clock + External Pulse Both Jittered (σ = {sigma_values[-1]:.2f} ns @ {FREQ_LIST[-1]:.2f} MHz)",
    height=650,
    hovermode='x unified',
    showlegend=False,
    xaxis3_title="Time (ns)",
    yaxis=dict(showticklabels=False)
)
fig.show()

# PLOT 2 — Mean Total Errors vs σ for Each Frequency
fig2 = go.Figure()
colors = ['purple', 'teal', 'orange', 'green', 'red', 'blue']

for idx, freq in enumerate(FREQ_LIST):
    fig2.add_trace(go.Scatter(
        x=sigma_values, y=all_mean_errors[freq],
        mode='lines+markers',
        line=dict(color=colors[idx % len(colors)], width=3),
        marker=dict(size=8),
        name=f'{freq:.2f} MHz'
    ))

fig2.update_layout(
    title="Case 3 — Mean Total Errors vs Gaussian Jitter σ (Clock + External Pulse Jittered)",
    xaxis_title="Jitter Standard Deviation σ (ns)",
    yaxis_title="Mean Total Errors",
    height=550,
    template="plotly_white",
    legend_title="Frequencies"
)
fig2.show()


### 3.3

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from datetime import datetime

# Set your Drive save folder here
base_folder = '/content/drive/MyDrive/Jitter_Simulations'
os.makedirs(base_folder, exist_ok=True)

# USER INPUTS
NUM_FREQS = int(input("Enter number of frequencies to test: ") or 2)
FREQ_LIST = [float(input(f"Enter frequency {i+1} (MHz): ") or 40.0) for i in range(NUM_FREQS)]
NUM_TRIALS = int(input("Enter number of simulation runs per frequency: ") or 10)

# Option to test a range of σ values
range_choice = input("Do you want to run across a σ range? (y/n): ").strip().lower()
if range_choice == 'y':
    SIGMA_START = int(input("Enter starting σ value (integer): ") or 1)
    SIGMA_END = int(input("Enter ending σ value (integer): ") or 5)
else:
    single_sigma = float(input("Enter single Gaussian jitter σ (ns): ") or 1.0)
    SIGMA_START = SIGMA_END = int(single_sigma)

# SIMULATION PARAMETERS
SIM_DURATION = 100  # ns
NUM_STEPS = 100000
t = np.linspace(0, SIM_DURATION, NUM_STEPS)

# FUNCTION TO RUN ONE SIMULATION — BOTH CLOCK AND EXTERNAL PULSE JITTERED
def run_one_simulation(t, period_ns, sigma_ns):
    jitter_clk = np.random.normal(0, sigma_ns, len(t))
    jitter_ext = np.random.normal(0, sigma_ns, len(t))
    t_clk_jittered = t + jitter_clk
    t_ext_jittered = t + jitter_ext
    clk_jittered = (((t_clk_jittered % period_ns) < (period_ns / 2))).astype(int)
    ext_pulse_jittered = (((t_ext_jittered % period_ns) < (period_ns / 2))).astype(int)
    error_signal = np.where(ext_pulse_jittered != clk_jittered, 1, 0)
    total_errors = np.sum(error_signal)
    return total_errors, clk_jittered, ext_pulse_jittered, error_signal, jitter_clk, jitter_ext

# MAIN SIMULATION LOOP
sigma_values = list(range(SIGMA_START, SIGMA_END + 1))
all_mean_errors = {}
results = []

# We'll store the last run’s waveforms for plotting
last_clk_jittered = None
last_ext_pulse_jittered = None
last_error_signal = None

for freq in FREQ_LIST:
    print(f"\n=== Running simulations for {freq:.2f} MHz ===")
    period_ns = 1000 / freq
    mean_errors_list = []

    for sigma in sigma_values:
        error_counts = []
        for _ in range(NUM_TRIALS):
            total_errors, clk_jittered, ext_pulse_jittered, error_signal, jitter_clk, jitter_ext = run_one_simulation(t, period_ns, sigma)
            error_counts.append(total_errors)

            # Save the last run data for plotting
            last_clk_jittered = clk_jittered
            last_ext_pulse_jittered = ext_pulse_jittered
            last_error_signal = error_signal

        mean_error = np.mean(error_counts)
        mean_errors_list.append(mean_error)
        results.append({
            "Frequency (MHz)": freq,
            "Jitter Std Dev (ns)": sigma,
            "Mean Total Errors": mean_error
        })
        #print(f"σ = {sigma} ns → Mean Total Errors = {mean_error:.2f}")

    all_mean_errors[freq] = mean_errors_list

# SUMMARY
print("\n--- CASE 3: CLOCK + EXTERNAL PULSE BOTH JITTERED ---")
print(f"Frequencies Tested  : {', '.join([f'{f:.2f} MHz' for f in FREQ_LIST])}")
print(f"Number of Runs      : {NUM_TRIALS}")
print(f"σ Range Tested      : {SIGMA_START} to {SIGMA_END}")

# EXPORT TO DRIVE AS CSV WITH TIMESTAMP
df = pd.DataFrame(results)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"Case3_JitterSim_{NUM_STEPS}samples_{NUM_TRIALS}runs_{timestamp}.csv"
save_path = os.path.join(base_folder, filename)
df.to_csv(save_path, index=False)
print(f"\nData exported successfully to:\n{save_path}")

# PLOT 1 — Last Frequency’s Waveforms
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=("Jittered External Pulse",
                                    "Jittered Clock Output",
                                    "Error (Difference)"),
                    vertical_spacing=0.05)

fig.add_trace(go.Scatter(x=t, y=last_ext_pulse_jittered*2,
                         mode='lines', line=dict(color='blue', width=3),
                         line_shape='hv', name='External Pulse (Jittered)'), 1, 1)

fig.add_trace(go.Scatter(x=t, y=last_clk_jittered*2,
                         mode='lines', line=dict(color='orange', width=3),
                         line_shape='hv', name='Clock (Jittered)'), 2, 1)

fig.add_trace(go.Scatter(x=t, y=last_error_signal*2,
                         mode='lines', line=dict(color='red', width=2),
                         line_shape='hv', name='Error'), 3, 1)

fig.update_layout(title=f"Case 3 — Clock + External Pulse Both Jittered "
                        f"(σ = {sigma_values[-1]:.2f} ns @ {FREQ_LIST[-1]:.2f} MHz)",
                  height=650, hovermode='x unified', showlegend=False,
                  xaxis3_title="Time (ns)", yaxis=dict(showticklabels=False))
fig.show()

# PLOT 2 — Mean Total Errors vs σ for Each Frequency
fig2 = go.Figure()
colors = ['purple', 'teal', 'orange', 'green', 'red', 'blue']
for idx, freq in enumerate(FREQ_LIST):
    fig2.add_trace(go.Scatter(
        x=sigma_values, y=all_mean_errors[freq],
        mode='lines+markers',
        line=dict(color=colors[idx % len(colors)], width=3),
        marker=dict(size=8),
        name=f'{freq:.2f} MHz'
    ))
fig2.update_layout(title="Case 3 — Mean Total Errors vs Gaussian Jitter σ "
                         "(Clock + External Pulse Jittered)",
                   xaxis_title="Jitter Standard Deviation σ (ns)",
                   yaxis_title="Mean Total Errors",
                   height=550, template="plotly_white",
                   legend_title="Frequencies")
fig2.show()


## Metrics_ExternalPulse

### 4.1.1_RMS

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import os
from datetime import datetime

base_folder = '/content/drive/MyDrive/Jitter_Simulations'
os.makedirs(base_folder, exist_ok=True)

NUM_FREQS = int(input("Enter number of frequencies to test: ") or 2)
FREQ_LIST = [float(input(f"Enter frequency {i+1} (MHz): ") or 40.0)
             for i in range(NUM_FREQS)]
NUM_TRIALS = int(input("Enter number of simulation runs per frequency: ") or 10)

range_choice = input("Do you want to run across a σ range? (y/n): ").strip().lower()
if range_choice == 'y':
    SIGMA_START = int(input("Enter starting σ value (ns): ") or 1)
    SIGMA_END = int(input("Enter ending σ value (ns): ") or 5)
else:
    single_sigma = float(input("Enter single Gaussian jitter σ (ns): ") or 1.0)
    SIGMA_START = SIGMA_END = int(single_sigma)

sigma_values = list(range(SIGMA_START, SIGMA_END + 1))

SIM_DURATION = 100
NUM_STEPS = 100000
t = np.linspace(0, SIM_DURATION, NUM_STEPS)

# RMS JITTER FUNCTION
def compute_rms_jitter(t, period_ns, sigma_ns):
    # Gaussian jitter applied ONLY to external pulse
    jitter_ext = np.random.normal(0, sigma_ns, len(t))

    # Ideal clock
    t_clk = t
    t_ext = t + jitter_ext

    clk = (((t_clk % period_ns) < (period_ns / 2))).astype(int)
    ext = (((t_ext % period_ns) < (period_ns / 2))).astype(int)

    clk_edges = np.where(np.diff(clk) == 1)[0]
    ext_edges = np.where(np.diff(ext) == 1)[0]

    num_edges = min(len(clk_edges), len(ext_edges))
    if num_edges < 5:
        return np.nan

    clk_edge_times = t_clk[clk_edges[:num_edges]]
    ext_edge_times = t_ext[ext_edges[:num_edges]]

    ideal_edges = np.arange(num_edges) * period_ns

    tie_clk = clk_edge_times - ideal_edges
    tie_ext = ext_edge_times - ideal_edges

    relative_jitter = tie_ext - tie_clk

    rms_relative = np.sqrt(np.mean(relative_jitter ** 2))
    return rms_relative

# SIMULATION
results = []

for freq in FREQ_LIST:
    print(f"\nRunning RMS jitter simulation for {freq:.2f} MHz")
    period_ns = 1000 / freq

    for sigma in sigma_values:
        rms_vals = []
        for _ in range(NUM_TRIALS):
            rms_vals.append(compute_rms_jitter(t, period_ns, sigma))

        results.append({
            "Frequency (MHz)": freq,
            "Injected σ (ns)": sigma,
            "Measured RMS Jitter (ns)": np.nanmean(rms_vals)
        })

# ------------------- SAVE -------------------
df = pd.DataFrame(results)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"RMS_Jitter_ExtOnly_{NUM_STEPS}samples_{NUM_TRIALS}runs_{timestamp}.csv"
save_path = os.path.join(base_folder, filename)
df.to_csv(save_path, index=False)

print(f"\nRMS jitter data exported to:\n{save_path}")

# PLOT
fig = go.Figure()
colors = ['blue', 'red', 'green', 'purple', 'orange', 'teal']

for i, freq in enumerate(FREQ_LIST):
    sub = df[df["Frequency (MHz)"] == freq]
    fig.add_trace(go.Scatter(
        x=sub["Injected σ (ns)"],
        y=sub["Measured RMS Jitter (ns)"],
        mode='lines+markers',
        line=dict(width=3, color=colors[i % len(colors)]),
        marker=dict(size=8),
        name=f"{freq:.2f} MHz"
    ))

fig.update_layout(
    title="RMS External Pulse Jitter vs Injected σ<br>(Ideal Clock Reference)",
    xaxis_title="Injected External Jitter σ (ns)",
    yaxis_title="Measured RMS Jitter (ns)",
    height=550,
    template="plotly_white"
)

fig.show()

# LINEAR FIT
print("\nExtracted RMS Jitter Relationship:")

for freq in FREQ_LIST:
    sub = df[df["Frequency (MHz)"] == freq]
    coeffs = np.polyfit(
        sub["Injected σ (ns)"],
        sub["Measured RMS Jitter (ns)"],
        1
    )
    print(f"{freq:.2f} MHz → RMS ≈ {coeffs[0]:.3f}·σ + {coeffs[1]:.3f}")


### RMS with higher sim duration for 20MHz and 40MHz

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import os
from datetime import datetime

base_folder = '/content/drive/MyDrive/Jitter_Simulations'
os.makedirs(base_folder, exist_ok=True)

NUM_FREQS = int(input("Enter number of frequencies to test: ") or 2)
FREQ_LIST = [float(input(f"Enter frequency {i+1} (MHz): ") or 40.0)
             for i in range(NUM_FREQS)]
NUM_TRIALS = int(input("Enter number of simulation runs per frequency: ") or 10)

range_choice = input("Do you want to run across a σ range? (y/n): ").strip().lower()
if range_choice == 'y':
    SIGMA_START = int(input("Enter starting σ value (ns): ") or 1)
    SIGMA_END = int(input("Enter ending σ value (ns): ") or 5)
else:
    single_sigma = float(input("Enter single Gaussian jitter σ (ns): ") or 1.0)
    SIGMA_START = SIGMA_END = int(single_sigma)

sigma_values = list(range(SIGMA_START, SIGMA_END + 1))

SIM_DURATION = 500
NUM_STEPS = 100000
t = np.linspace(0, SIM_DURATION, NUM_STEPS)

# RMS JITTER FUNCTION
def compute_rms_jitter(t, period_ns, sigma_ns):
    # Gaussian jitter applied ONLY to external pulse
    jitter_ext = np.random.normal(0, sigma_ns, len(t))

    # Ideal clock
    t_clk = t
    t_ext = t + jitter_ext

    clk = (((t_clk % period_ns) < (period_ns / 2))).astype(int)
    ext = (((t_ext % period_ns) < (period_ns / 2))).astype(int)

    clk_edges = np.where(np.diff(clk) == 1)[0]
    ext_edges = np.where(np.diff(ext) == 1)[0]

    num_edges = min(len(clk_edges), len(ext_edges))
    if num_edges < 5:
        return np.nan

    clk_edge_times = t_clk[clk_edges[:num_edges]]
    ext_edge_times = t_ext[ext_edges[:num_edges]]

    ideal_edges = np.arange(num_edges) * period_ns

    tie_clk = clk_edge_times - ideal_edges
    tie_ext = ext_edge_times - ideal_edges

    relative_jitter = tie_ext - tie_clk

    rms_relative = np.sqrt(np.mean(relative_jitter ** 2))
    return rms_relative

# SIMULATION
results = []

for freq in FREQ_LIST:
    print(f"\nRunning RMS jitter simulation for {freq:.2f} MHz")
    period_ns = 1000 / freq

    for sigma in sigma_values:
        rms_vals = []
        for _ in range(NUM_TRIALS):
            rms_vals.append(compute_rms_jitter(t, period_ns, sigma))

        results.append({
            "Frequency (MHz)": freq,
            "Injected σ (ns)": sigma,
            "Measured RMS Jitter (ns)": np.nanmean(rms_vals)
        })

# ------------------- SAVE -------------------
df = pd.DataFrame(results)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"RMS_Jitter_ExtOnly_{NUM_STEPS}samples_{NUM_TRIALS}runs_{timestamp}.csv"
save_path = os.path.join(base_folder, filename)
df.to_csv(save_path, index=False)

print(f"\nRMS jitter data exported to:\n{save_path}")

# PLOT
fig = go.Figure()
colors = ['blue', 'red', 'green', 'purple', 'orange', 'teal']

for i, freq in enumerate(FREQ_LIST):
    sub = df[df["Frequency (MHz)"] == freq]
    fig.add_trace(go.Scatter(
        x=sub["Injected σ (ns)"],
        y=sub["Measured RMS Jitter (ns)"],
        mode='lines+markers',
        line=dict(width=3, color=colors[i % len(colors)]),
        marker=dict(size=8),
        name=f"{freq:.2f} MHz"
    ))

fig.update_layout(
    title="RMS External Pulse Jitter vs Injected σ<br>(Ideal Clock Reference)",
    xaxis_title="Injected External Jitter σ (ns)",
    yaxis_title="Measured RMS Jitter (ns)",
    height=550,
    template="plotly_white"
)

fig.show()

# LINEAR FIT
print("\nExtracted RMS Jitter Relationship:")

for freq in FREQ_LIST:
    sub = df[df["Frequency (MHz)"] == freq]
    coeffs = np.polyfit(
        sub["Injected σ (ns)"],
        sub["Measured RMS Jitter (ns)"],
        1
    )
    print(f"{freq:.2f} MHz → RMS ≈ {coeffs[0]:.3f}·σ + {coeffs[1]:.3f}")


### 4.1.2 peak to peak

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import os
from datetime import datetime

# DRIVE SETUP
base_folder = '/content/drive/MyDrive/Jitter_Simulations'
os.makedirs(base_folder, exist_ok=True)

# USER INPUTS
NUM_FREQS = int(input("Enter number of frequencies to test: ") or 2)
FREQ_LIST = [float(input(f"Enter frequency {i+1} (MHz): ") or 40.0)
             for i in range(NUM_FREQS)]
NUM_TRIALS = int(input("Enter number of simulation runs per frequency: ") or 10)

range_choice = input("Do you want to run across a σ range? (y/n): ").strip().lower()
if range_choice == 'y':
    SIGMA_START = int(input("Enter starting σ value (ns): ") or 1)
    SIGMA_END = int(input("Enter ending σ value (ns): ") or 5)
else:
    single_sigma = float(input("Enter single Gaussian jitter σ (ns): ") or 1.0)
    SIGMA_START = SIGMA_END = int(single_sigma)

sigma_values = list(range(SIGMA_START, SIGMA_END + 1))

# SIMULATION GRID
SIM_DURATION = 100        # ns
NUM_STEPS = 100000
t = np.linspace(0, SIM_DURATION, NUM_STEPS)

# PEAK-TO-PEAK JITTER FUNCTION
def compute_pp_jitter(t, period_ns, sigma_ns):
    # Gaussian jitter applied ONLY to external pulse
    jitter_ext = np.random.normal(0, sigma_ns, len(t))

    # Ideal clock
    t_clk = t
    t_ext = t + jitter_ext

    clk = (((t_clk % period_ns) < (period_ns / 2))).astype(int)
    ext = (((t_ext % period_ns) < (period_ns / 2))).astype(int)

    # Rising edges
    clk_edges = np.where(np.diff(clk) == 1)[0]
    ext_edges = np.where(np.diff(ext) == 1)[0]

    num_edges = min(len(clk_edges), len(ext_edges))
    if num_edges < 5:
        return np.nan

    clk_edge_times = t_clk[clk_edges[:num_edges]]
    ext_edge_times = t_ext[ext_edges[:num_edges]]

    # Ideal edge times
    ideal_edges = np.arange(num_edges) * period_ns

    # Time Interval Error (TIE)
    tie_clk = clk_edge_times - ideal_edges     # ~0
    tie_ext = ext_edge_times - ideal_edges

    # Relative jitter (external vs ideal clock)
    relative_jitter = tie_ext - tie_clk

    # Peak-to-Peak jitter
    pp_relative = np.max(relative_jitter) - np.min(relative_jitter)
    return pp_relative

# MAIN SIMULATION
results = []

for freq in FREQ_LIST:
    print(f"\nRunning Peak-to-Peak jitter simulation for {freq:.2f} MHz")
    period_ns = 1000 / freq

    for sigma in sigma_values:
        pp_vals = []
        for _ in range(NUM_TRIALS):
            pp_vals.append(compute_pp_jitter(t, period_ns, sigma))

        results.append({
            "Frequency (MHz)": freq,
            "Injected σ (ns)": sigma,
            "Measured Peak-to-Peak Jitter (ns)": np.nanmean(pp_vals)
        })

# SAVE RESULTS
df = pd.DataFrame(results)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"PP_Jitter_ExtOnly_{NUM_STEPS}samples_{NUM_TRIALS}runs_{timestamp}.csv"
save_path = os.path.join(base_folder, filename)
df.to_csv(save_path, index=False)

print(f"\nPeak-to-Peak jitter data exported to:\n{save_path}")

# PLOT
fig = go.Figure()
colors = ['blue', 'red', 'green', 'purple', 'orange', 'teal']

for i, freq in enumerate(FREQ_LIST):
    sub = df[df["Frequency (MHz)"] == freq]
    fig.add_trace(go.Scatter(
        x=sub["Injected σ (ns)"],
        y=sub["Measured Peak-to-Peak Jitter (ns)"],
        mode='lines+markers',
        line=dict(width=3, color=colors[i % len(colors)]),
        marker=dict(size=8),
        name=f"{freq:.2f} MHz"
    ))

fig.update_layout(
    title="Peak-to-Peak External Pulse Jitter vs Injected σ<br>"
          "(Ideal Clock Reference)",
    xaxis_title="Injected External Jitter σ (ns)",
    yaxis_title="Measured Peak-to-Peak Jitter (ns)",
    height=550,
    template="plotly_white"
)

fig.show()

# LINEAR FIT
print("\nExtracted Peak-to-Peak Jitter Relationship:")

for freq in FREQ_LIST:
    sub = df[df["Frequency (MHz)"] == freq]
    coeffs = np.polyfit(
        sub["Injected σ (ns)"],
        sub["Measured Peak-to-Peak Jitter (ns)"],
        1
    )
    print(f"{freq:.2f} MHz → P-P ≈ {coeffs[0]:.3f}·σ + {coeffs[1]:.3f}")


### 4.1.3 RMS&PkPk

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import os
from datetime import datetime

# DRIVE SETUP
base_folder = '/content/drive/MyDrive/Jitter_Simulations'
os.makedirs(base_folder, exist_ok=True)

# USER INPUTS
NUM_FREQS = int(input("Enter number of frequencies to test: ") or 2)
FREQ_LIST = [float(input(f"Enter frequency {i+1} (MHz): ") or 40.0)
             for i in range(NUM_FREQS)]
NUM_TRIALS = int(input("Enter number of simulation runs per frequency: ") or 10)

range_choice = input("Do you want to run across a σ range? (y/n): ").strip().lower()
if range_choice == 'y':
    SIGMA_START = int(input("Enter starting σ value (ns): ") or 1)
    SIGMA_END = int(input("Enter ending σ value (ns): ") or 5)
else:
    single_sigma = float(input("Enter single Gaussian jitter σ (ns): ") or 1.0)
    SIGMA_START = SIGMA_END = int(single_sigma)

sigma_values = list(range(SIGMA_START, SIGMA_END + 1))

# SIMULATION GRID
SIM_DURATION = 100
NUM_STEPS = 100000
t = np.linspace(0, SIM_DURATION, NUM_STEPS)

# ------------------- UPDATED FUNCTION (RMS + P2P) -------------------
def compute_jitter_metrics(t, period_ns, sigma_ns):
    jitter_ext = np.random.normal(0, sigma_ns, len(t))

    t_clk = t
    t_ext = t + jitter_ext

    clk = (((t_clk % period_ns) < (period_ns / 2))).astype(int)
    ext = (((t_ext % period_ns) < (period_ns / 2))).astype(int)

    clk_edges = np.where(np.diff(clk) == 1)[0]
    ext_edges = np.where(np.diff(ext) == 1)[0]

    num_edges = min(len(clk_edges), len(ext_edges))
    if num_edges < 5:
        return np.nan, np.nan

    clk_edge_times = t_clk[clk_edges[:num_edges]]
    ext_edge_times = t_ext[ext_edges[:num_edges]]

    ideal_edges = np.arange(num_edges) * period_ns

    tie_clk = clk_edge_times - ideal_edges
    tie_ext = ext_edge_times - ideal_edges

    relative_jitter = tie_ext - tie_clk

    #RMS
    rms = np.sqrt(np.mean(relative_jitter ** 2))

    # Peak-to-Peak
    p2p = np.max(relative_jitter) - np.min(relative_jitter)

    return rms, p2p

# MAIN SIMULATION
results = []

for freq in FREQ_LIST:
    print(f"\nRunning jitter simulation for {freq:.2f} MHz")
    period_ns = 1000 / freq

    for sigma in sigma_values:
        rms_vals = []
        pp_vals = []

        for _ in range(NUM_TRIALS):
            rms, p2p = compute_jitter_metrics(t, period_ns, sigma)
            rms_vals.append(rms)
            pp_vals.append(p2p)

        results.append({
            "Frequency (MHz)": freq,
            "Injected σ (ns)": sigma,
            "Measured RMS Jitter (ns)": np.nanmean(rms_vals),
            "Measured Peak-to-Peak Jitter (ns)": np.nanmean(pp_vals)
        })

# SAVE RESULTS
df = pd.DataFrame(results)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"DualMetric_Jitter_ExtOnly_{NUM_STEPS}samples_{NUM_TRIALS}runs_{timestamp}.csv"
save_path = os.path.join(base_folder, filename)
df.to_csv(save_path, index=False)

print(f"\nJitter data exported to:\n{save_path}")

# PLOT
fig = go.Figure()
colors = ['blue', 'red', 'green', 'purple', 'orange', 'teal']

for i, freq in enumerate(FREQ_LIST):
    sub = df[df["Frequency (MHz)"] == freq]

    # RMS
    fig.add_trace(go.Scatter(
        x=sub["Injected σ (ns)"],
        y=sub["Measured RMS Jitter (ns)"],
        mode='lines+markers',
        line=dict(width=3, color=colors[i % len(colors)]),
        name=f"{freq:.2f} MHz (RMS)"
    ))

    # P2P
    fig.add_trace(go.Scatter(
        x=sub["Injected σ (ns)"],
        y=sub["Measured Peak-to-Peak Jitter (ns)"],
        mode='lines+markers',
        line=dict(width=3, dash='dash', color=colors[i % len(colors)]),
        name=f"{freq:.2f} MHz (P2P)"
    ))

fig.update_layout(
    title="RMS & Peak-to-Peak External Pulse Jitter vs Injected σ<br>(Ideal Clock Reference)",
    xaxis_title="Injected External Jitter σ (ns)",
    yaxis_title="Jitter (ns)",
    height=550,
    template="plotly_white"
)

fig.show()

# LINEAR FIT
print("\nExtracted Jitter Relationships:")

for freq in FREQ_LIST:
    sub = df[df["Frequency (MHz)"] == freq]

    coeffs_rms = np.polyfit(
        sub["Injected σ (ns)"],
        sub["Measured RMS Jitter (ns)"],
        1
    )

    coeffs_p2p = np.polyfit(
        sub["Injected σ (ns)"],
        sub["Measured Peak-to-Peak Jitter (ns)"],
        1
    )

    print(f"\n{freq:.2f} MHz:")
    print(f"  RMS ≈ {coeffs_rms[0]:.3f}·σ + {coeffs_rms[1]:.3f}")
    print(f"  P-P ≈ {coeffs_p2p[0]:.3f}·σ + {coeffs_p2p[1]:.3f}")

## Metrics_Clock

### 4.2.1_RMS

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import os
from datetime import datetime

# DRIVE SETUP
base_folder = '/content/drive/MyDrive/Jitter_Simulations'
os.makedirs(base_folder, exist_ok=True)

# USER INPUTS
NUM_FREQS = int(input("Enter number of frequencies to test: ") or 2)
FREQ_LIST = [float(input(f"Enter frequency {i+1} (MHz): ") or 40.0)
             for i in range(NUM_FREQS)]
NUM_TRIALS = int(input("Enter number of simulation runs per frequency: ") or 10)

range_choice = input("Do you want to run across a σ range? (y/n): ").strip().lower()
if range_choice == 'y':
    SIGMA_START = int(input("Enter starting σ value (ns): ") or 1)
    SIGMA_END = int(input("Enter ending σ value (ns): ") or 5)
else:
    single_sigma = float(input("Enter single Gaussian jitter σ (ns): ") or 1.0)
    SIGMA_START = SIGMA_END = int(single_sigma)

sigma_values = list(range(SIGMA_START, SIGMA_END + 1))

# SIMULATION GRID
SIM_DURATION = 100        # ns
NUM_STEPS = 100000
t = np.linspace(0, SIM_DURATION, NUM_STEPS)

# RMS JITTER FUNCTION (CLOCK ONLY)
def compute_rms_jitter(t, period_ns, sigma_ns):
    # Gaussian jitter applied ONLY to clock
    jitter_clk = np.random.normal(0, sigma_ns, len(t))

    # Ideal external trigger (no jitter)
    t_ext = t
    t_clk = t + jitter_clk

    ext = (((t_ext % period_ns) < (period_ns / 2))).astype(int)
    clk = (((t_clk % period_ns) < (period_ns / 2))).astype(int)

    # Rising edges
    ext_edges = np.where(np.diff(ext) == 1)[0]
    clk_edges = np.where(np.diff(clk) == 1)[0]

    num_edges = min(len(ext_edges), len(clk_edges))
    if num_edges < 5:
        return np.nan

    ext_edge_times = t_ext[ext_edges[:num_edges]]
    clk_edge_times = t_clk[clk_edges[:num_edges]]

    # Ideal edge times
    ideal_edges = np.arange(num_edges) * period_ns

    # Time Interval Error (TIE)
    tie_ext = ext_edge_times - ideal_edges      # ~0
    tie_clk = clk_edge_times - ideal_edges

    # Relative jitter (clock vs ideal trigger)
    relative_jitter = tie_clk - tie_ext

    rms_relative = np.sqrt(np.mean(relative_jitter ** 2))
    return rms_relative

# MAIN SIMULATION
results = []

for freq in FREQ_LIST:
    print(f"\nRunning RMS jitter simulation for {freq:.2f} MHz")
    period_ns = 1000 / freq

    for sigma in sigma_values:
        rms_vals = []
        for _ in range(NUM_TRIALS):
            rms_vals.append(compute_rms_jitter(t, period_ns, sigma))

        results.append({
            "Frequency (MHz)": freq,
            "Injected σ (ns)": sigma,
            "Measured RMS Jitter (ns)": np.nanmean(rms_vals)
        })

# SAVE RESULTS
df = pd.DataFrame(results)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"RMS_Jitter_ClockOnly_{NUM_STEPS}samples_{NUM_TRIALS}runs_{timestamp}.csv"
save_path = os.path.join(base_folder, filename)
df.to_csv(save_path, index=False)

print(f"\nRMS jitter data exported to:\n{save_path}")

# PLOT
fig = go.Figure()
colors = ['blue', 'red', 'green', 'purple', 'orange', 'teal']

for i, freq in enumerate(FREQ_LIST):
    sub = df[df["Frequency (MHz)"] == freq]
    fig.add_trace(go.Scatter(
        x=sub["Injected σ (ns)"],
        y=sub["Measured RMS Jitter (ns)"],
        mode='lines+markers',
        line=dict(width=3, color=colors[i % len(colors)]),
        marker=dict(size=8),
        name=f"{freq:.2f} MHz"
    ))

fig.update_layout(
    title="RMS Clock Jitter vs Injected σ<br>"
          "(Ideal External Trigger Reference)",
    xaxis_title="Injected Clock Jitter σ (ns)",
    yaxis_title="Measured RMS Jitter (ns)",
    height=550,
    template="plotly_white"
)

fig.show()

#LINEAR FIT
print("\nExtracted RMS Jitter Relationship:")

for freq in FREQ_LIST:
    sub = df[df["Frequency (MHz)"] == freq]
    coeffs = np.polyfit(
        sub["Injected σ (ns)"],
        sub["Measured RMS Jitter (ns)"],
        1
    )
    print(f"{freq:.2f} MHz → RMS ≈ {coeffs[0]:.3f}·σ + {coeffs[1]:.3f}")


### 4.2.2 peak to peak

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import os
from datetime import datetime

#DRIVE SETUP
base_folder = '/content/drive/MyDrive/Jitter_Simulations'
os.makedirs(base_folder, exist_ok=True)

#USER INPUTS
NUM_FREQS = int(input("Enter number of frequencies to test: ") or 2)
FREQ_LIST = [float(input(f"Enter frequency {i+1} (MHz): ") or 40.0)
             for i in range(NUM_FREQS)]
NUM_TRIALS = int(input("Enter number of simulation runs per frequency: ") or 10)

range_choice = input("Do you want to run across a σ range? (y/n): ").strip().lower()
if range_choice == 'y':
    SIGMA_START = int(input("Enter starting σ value (ns): ") or 1)
    SIGMA_END = int(input("Enter ending σ value (ns): ") or 5)
else:
    single_sigma = float(input("Enter single Gaussian jitter σ (ns): ") or 1.0)
    SIGMA_START = SIGMA_END = int(single_sigma)

sigma_values = list(range(SIGMA_START, SIGMA_END + 1))

# SIMULATION GRID
SIM_DURATION = 100        # ns
NUM_STEPS = 100_000
t = np.linspace(0, SIM_DURATION, NUM_STEPS)

#PEAK-TO-PEAK JITTER FUNCTION
def compute_pp_jitter(t, period_ns, sigma_ns):
    # Gaussian jitter applied ONLY to clock
    jitter_clk = np.random.normal(0, sigma_ns, len(t))

    # Ideal external trigger
    t_ext = t
    t_clk = t + jitter_clk

    ext = (((t_ext % period_ns) < (period_ns / 2))).astype(int)
    clk = (((t_clk % period_ns) < (period_ns / 2))).astype(int)

    # Rising edges
    ext_edges = np.where(np.diff(ext) == 1)[0]
    clk_edges = np.where(np.diff(clk) == 1)[0]

    num_edges = min(len(ext_edges), len(clk_edges))
    if num_edges < 5:
        return np.nan

    ext_edge_times = t_ext[ext_edges[:num_edges]]
    clk_edge_times = t_clk[clk_edges[:num_edges]]

    # Ideal edge times
    ideal_edges = np.arange(num_edges) * period_ns

    # Time Interval Error (TIE)
    tie_ext = ext_edge_times - ideal_edges
    tie_clk = clk_edge_times - ideal_edges

    # Relative jitter (clock vs ideal reference)
    relative_jitter = tie_clk - tie_ext

    # Peak-to-Peak jitter
    pp_relative = np.max(relative_jitter) - np.min(relative_jitter)
    return pp_relative

# MAIN SIMULATION
results = []

for freq in FREQ_LIST:
    print(f"\nRunning Peak-to-Peak jitter simulation for {freq:.2f} MHz")
    period_ns = 1000 / freq

    for sigma in sigma_values:
        pp_vals = []
        for _ in range(NUM_TRIALS):
            pp_vals.append(compute_pp_jitter(t, period_ns, sigma))

        results.append({
            "Frequency (MHz)": freq,
            "Injected σ (ns)": sigma,
            "Measured Peak-to-Peak Jitter (ns)": np.nanmean(pp_vals)
        })

# SAVE RESULTS
df = pd.DataFrame(results)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"PP_Jitter_ClockOnly_{NUM_STEPS}samples_{NUM_TRIALS}runs_{timestamp}.csv"
save_path = os.path.join(base_folder, filename)
df.to_csv(save_path, index=False)

print(f"\nPeak-to-Peak jitter data exported to:\n{save_path}")

# PLOT
fig = go.Figure()
colors = ['blue', 'red', 'green', 'purple', 'orange', 'teal']

for i, freq in enumerate(FREQ_LIST):
    sub = df[df["Frequency (MHz)"] == freq]
    fig.add_trace(go.Scatter(
        x=sub["Injected σ (ns)"],
        y=sub["Measured Peak-to-Peak Jitter (ns)"],
        mode='lines+markers',
        line=dict(width=3, color=colors[i % len(colors)]),
        marker=dict(size=8),
        name=f"{freq:.2f} MHz"
    ))

fig.update_layout(
    title="Peak-to-Peak Clock Jitter vs Injected σ<br>"
          "(Ideal External Trigger Reference)",
    xaxis_title="Injected Clock Jitter σ (ns)",
    yaxis_title="Measured Peak-to-Peak Jitter (ns)",
    height=550,
    template="plotly_white"
)

fig.show()

# LINEAR FIT
print("\nExtracted Peak-to-Peak Jitter Relationship:")

for freq in FREQ_LIST:
    sub = df[df["Frequency (MHz)"] == freq]
    coeffs = np.polyfit(
        sub["Injected σ (ns)"],
        sub["Measured Peak-to-Peak Jitter (ns)"],
        1
    )
    print(f"{freq:.2f} MHz → P-P ≈ {coeffs[0]:.3f}·σ + {coeffs[1]:.3f}")


## Metrics_relativejitter(clock+extpulse)

### 4.3.1 RMS

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import os
from datetime import datetime

base_folder = '/content/drive/MyDrive/Jitter_Simulations'
os.makedirs(base_folder, exist_ok=True)


NUM_FREQS = int(input("Enter number of frequencies to test: ") or 2)
FREQ_LIST = [float(input(f"Enter frequency {i+1} (MHz): ") or 40.0) for i in range(NUM_FREQS)]
NUM_TRIALS = int(input("Enter number of simulation runs per frequency: ") or 10)

range_choice = input("Do you want to run across a σ range? (y/n): ").strip().lower()
if range_choice == 'y':
    SIGMA_START = int(input("Enter starting σ value (ns): ") or 1)
    SIGMA_END = int(input("Enter ending σ value (ns): ") or 5)
else:
    single_sigma = float(input("Enter single Gaussian jitter σ (ns): ") or 1.0)
    SIGMA_START = SIGMA_END = int(single_sigma)

sigma_values = list(range(SIGMA_START, SIGMA_END + 1))

SIM_DURATION = 100        # ns
NUM_STEPS = 100000      # higher resolution to reduce quantization
t = np.linspace(0, SIM_DURATION, NUM_STEPS)


def compute_rms_jitter(t, period_ns, sigma_ns):
    # Independent Gaussian jitter
    jitter_clk = np.random.normal(0, sigma_ns, len(t))
    jitter_ext = np.random.normal(0, sigma_ns, len(t))

    t_clk = t + jitter_clk
    t_ext = t + jitter_ext

    clk = (((t_clk % period_ns) < (period_ns / 2))).astype(int)
    ext = (((t_ext % period_ns) < (period_ns / 2))).astype(int)

    # Rising edges
    clk_edges = np.where(np.diff(clk) == 1)[0]
    ext_edges = np.where(np.diff(ext) == 1)[0]

    num_edges = min(len(clk_edges), len(ext_edges))
    if num_edges < 5:
        return np.nan

    clk_edge_times = t_clk[clk_edges[:num_edges]]
    ext_edge_times = t_ext[ext_edges[:num_edges]]

    # Ideal edge times (same index)
    ideal_edges = np.arange(num_edges) * period_ns

    # Time Interval Error (TIE)
    tie_clk = clk_edge_times - ideal_edges
    tie_ext = ext_edge_times - ideal_edges

    # Relative jitter (combined)
    relative_jitter = tie_ext - tie_clk

    rms_relative = np.sqrt(np.mean(relative_jitter ** 2))
    return rms_relative


results = []

for freq in FREQ_LIST:
    print(f"\nRunning RMS jitter simulation for {freq:.2f} MHz")
    period_ns = 1000 / freq

    for sigma in sigma_values:
        rms_vals = []
        for _ in range(NUM_TRIALS):
            rms = compute_rms_jitter(t, period_ns, sigma)
            rms_vals.append(rms)

        results.append({
            "Frequency (MHz)": freq,
            "Injected σ (ns)": sigma,
            "Measured RMS Jitter (ns)": np.nanmean(rms_vals)
        })


df = pd.DataFrame(results)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"RMS_Jitter_TIE_{NUM_STEPS}samples_{NUM_TRIALS}runs_{timestamp}.csv"
save_path = os.path.join(base_folder, filename)
df.to_csv(save_path, index=False)

print(f"\nRMS jitter data exported to:\n{save_path}")


fig = go.Figure()
colors = ['blue', 'red', 'green', 'purple', 'orange', 'teal']

for i, freq in enumerate(FREQ_LIST):
    sub = df[df["Frequency (MHz)"] == freq]
    fig.add_trace(go.Scatter(
        x=sub["Injected σ (ns)"],
        y=sub["Measured RMS Jitter (ns)"],
        mode='lines+markers',
        line=dict(width=3, color=colors[i % len(colors)]),
        marker=dict(size=8),
        name=f"{freq:.2f} MHz"
    ))

fig.update_layout(
    title="RMS Relative Jitter vs Injected σ<br>"
          "(TIE-Based Measurement — Physically Correct)",
    xaxis_title="Injected Jitter σ (ns)",
    yaxis_title="Measured RMS Relative Jitter (ns)",
    height=550,
    template="plotly_white"
)

fig.show()

print("\nExtracted RMS Jitter Relationship:")

for freq in FREQ_LIST:
    sub = df[df["Frequency (MHz)"] == freq]
    coeffs = np.polyfit(
        sub["Injected σ (ns)"],
        sub["Measured RMS Jitter (ns)"],
        1
    )
    print(f"{freq:.2f} MHz → RMS ≈ {coeffs[0]:.3f}·σ + {coeffs[1]:.3f}")


### 4.3.2 peak to peak

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import os
from datetime import datetime

# DRIVE SETUP
base_folder = '/content/drive/MyDrive/Jitter_Simulations'
os.makedirs(base_folder, exist_ok=True)

# USER INPUTS
NUM_FREQS = int(input("Enter number of frequencies to test: ") or 2)
FREQ_LIST = [float(input(f"Enter frequency {i+1} (MHz): ") or 40.0)
             for i in range(NUM_FREQS)]
NUM_TRIALS = int(input("Enter number of simulation runs per frequency: ") or 10)

range_choice = input("Do you want to run across a σ range? (y/n): ").strip().lower()
if range_choice == 'y':
    SIGMA_START = int(input("Enter starting σ value (ns): ") or 1)
    SIGMA_END = int(input("Enter ending σ value (ns): ") or 5)
else:
    single_sigma = float(input("Enter single Gaussian jitter σ (ns): ") or 1.0)
    SIGMA_START = SIGMA_END = int(single_sigma)

sigma_values = list(range(SIGMA_START, SIGMA_END + 1))

# SIMULATION GRID
SIM_DURATION = 100        # ns
NUM_STEPS = 100000       # high resolution
t = np.linspace(0, SIM_DURATION, NUM_STEPS)

# PEAK-TO-PEAK JITTER FUNCTION
def compute_ptp_jitter(t, period_ns, sigma_ns):
    jitter_clk = np.random.normal(0, sigma_ns, len(t))
    jitter_ext = np.random.normal(0, sigma_ns, len(t))

    t_clk = t + jitter_clk
    t_ext = t + jitter_ext

    clk = (((t_clk % period_ns) < (period_ns / 2))).astype(int)
    ext = (((t_ext % period_ns) < (period_ns / 2))).astype(int)

    clk_edges = np.where(np.diff(clk) == 1)[0]
    ext_edges = np.where(np.diff(ext) == 1)[0]

    num_edges = min(len(clk_edges), len(ext_edges))
    if num_edges < 5:
        return np.nan

    clk_edge_times = t_clk[clk_edges[:num_edges]]
    ext_edge_times = t_ext[ext_edges[:num_edges]]

    ideal_edges = np.arange(num_edges) * period_ns

    tie_clk = clk_edge_times - ideal_edges
    tie_ext = ext_edge_times - ideal_edges

    relative_jitter = tie_ext - tie_clk

    ptp_relative = np.max(relative_jitter) - np.min(relative_jitter)
    return ptp_relative

# MAIN SIMULATION
results = []

for freq in FREQ_LIST:
    print(f"\nRunning Peak-to-Peak jitter simulation for {freq:.2f} MHz")
    period_ns = 1000 / freq

    for sigma in sigma_values:
        ptp_vals = []
        for _ in range(NUM_TRIALS):
            ptp = compute_ptp_jitter(t, period_ns, sigma)
            ptp_vals.append(ptp)

        results.append({
            "Frequency (MHz)": freq,
            "Injected σ (ns)": sigma,
            "Measured Pk-Pk Jitter (ns)": np.nanmean(ptp_vals)
        })

# SAVE RESULTS
df = pd.DataFrame(results)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"PkPk_Jitter_TIE_{NUM_STEPS}samples_{NUM_TRIALS}runs_{timestamp}.csv"
save_path = os.path.join(base_folder, filename)
df.to_csv(save_path, index=False)

print(f"\nPeak-to-Peak jitter data exported to:\n{save_path}")

# PLOT
fig = go.Figure()
colors = ['blue', 'red', 'green', 'purple', 'orange', 'teal']

for i, freq in enumerate(FREQ_LIST):
    sub = df[df["Frequency (MHz)"] == freq]
    fig.add_trace(go.Scatter(
        x=sub["Injected σ (ns)"],
        y=sub["Measured Pk-Pk Jitter (ns)"],
        mode='lines+markers',
        line=dict(width=3, color=colors[i % len(colors)]),
        marker=dict(size=8),
        name=f"{freq:.2f} MHz"
    ))

fig.update_layout(
    title="Peak-to-Peak Relative Jitter vs Injected σ<br>"
          "(TIE-Based Measurement)",
    xaxis_title="Injected Jitter σ (ns)",
    yaxis_title="Measured Peak-to-Peak Relative Jitter (ns)",
    height=550,
    template="plotly_white"
)

fig.show()

# LINEAR FIT
print("\nExtracted Peak-to-Peak Jitter Relationship:")

for freq in FREQ_LIST:
    sub = df[df["Frequency (MHz)"] == freq]
    coeffs = np.polyfit(
        sub["Injected σ (ns)"],
        sub["Measured Pk-Pk Jitter (ns)"],
        1
    )
    print(f"{freq:.2f} MHz → Pk–Pk ≈ {coeffs[0]:.3f}·σ + {coeffs[1]:.3f}")


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import os
from datetime import datetime

# ------------------- DRIVE SETUP -------------------
base_folder = '/content/drive/MyDrive/Jitter_Simulations'
os.makedirs(base_folder, exist_ok=True)

# ------------------- USER INPUTS -------------------
NUM_FREQS = int(input("Enter number of frequencies to test: ") or 2)
FREQ_LIST = [float(input(f"Enter frequency {i+1} (MHz): ") or 40.0)
             for i in range(NUM_FREQS)]
NUM_TRIALS = int(input("Enter number of simulation runs per frequency: ") or 10)

range_choice = input("Do you want to run across a σ range? (y/n): ").strip().lower()
if range_choice == 'y':
    SIGMA_START = int(input("Enter starting σ value (ns): ") or 1)
    SIGMA_END = int(input("Enter ending σ value (ns): ") or 5)
else:
    single_sigma = float(input("Enter single Gaussian jitter σ (ns): ") or 1.0)
    SIGMA_START = SIGMA_END = int(single_sigma)

sigma_values = list(range(SIGMA_START, SIGMA_END + 1))

# ------------------- SIMULATION GRID -------------------
SIM_DURATION = 100        # ns
NUM_STEPS = 100000       # high resolution
t = np.linspace(0, SIM_DURATION, NUM_STEPS)

# ------------------- PEAK-TO-PEAK JITTER FUNCTION -------------------
def compute_ptp_jitter(t, period_ns, sigma_ns):
    jitter_clk = np.random.normal(0, sigma_ns, len(t))
    jitter_ext = np.random.normal(0, sigma_ns, len(t))

    t_clk = t + jitter_clk
    t_ext = t + jitter_ext

    clk = (((t_clk % period_ns) < (period_ns / 2))).astype(int)
    ext = (((t_ext % period_ns) < (period_ns / 2))).astype(int)

    clk_edges = np.where(np.diff(clk) == 1)[0]
    ext_edges = np.where(np.diff(ext) == 1)[0]

    num_edges = min(len(clk_edges), len(ext_edges))
    if num_edges < 5:
        return np.nan

    clk_edge_times = t_clk[clk_edges[:num_edges]]
    ext_edge_times = t_ext[ext_edges[:num_edges]]

    ideal_edges = np.arange(num_edges) * period_ns

    tie_clk = clk_edge_times - ideal_edges
    tie_ext = ext_edge_times - ideal_edges

    relative_jitter = tie_ext - tie_clk

    ptp_relative = np.max(relative_jitter) - np.min(relative_jitter)
    return ptp_relative

# ------------------- MAIN SIMULATION -------------------
results = []

for freq in FREQ_LIST:
    print(f"\nRunning Peak-to-Peak jitter simulation for {freq:.2f} MHz")
    period_ns = 1000 / freq

    for sigma in sigma_values:
        ptp_vals = []
        for _ in range(NUM_TRIALS):
            ptp = compute_ptp_jitter(t, period_ns, sigma)
            ptp_vals.append(ptp)

        results.append({
            "Frequency (MHz)": freq,
            "Injected σ (ns)": sigma,
            "Measured Pk-Pk Jitter (ns)": np.nanmean(ptp_vals)
        })

# ------------------- SAVE RESULTS -------------------
df = pd.DataFrame(results)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"PkPk_Jitter_TIE_{NUM_STEPS}samples_{NUM_TRIALS}runs_{timestamp}.csv"
save_path = os.path.join(base_folder, filename)
df.to_csv(save_path, index=False)

print(f"\nPeak-to-Peak jitter data exported to:\n{save_path}")

# ------------------- PLOT -------------------
fig = go.Figure()
colors = ['blue', 'red', 'green', 'purple', 'orange', 'teal']

for i, freq in enumerate(FREQ_LIST):
    sub = df[df["Frequency (MHz)"] == freq]
    fig.add_trace(go.Scatter(
        x=sub["Injected σ (ns)"],
        y=sub["Measured Pk-Pk Jitter (ns)"],
        mode='lines+markers',
        line=dict(width=3, color=colors[i % len(colors)]),
        marker=dict(size=8),
        name=f"{freq:.2f} MHz"
    ))

fig.update_layout(
    title="Peak-to-Peak Relative Jitter vs Injected σ<br>"
          "(TIE-Based Measurement)",
    xaxis_title="Injected Jitter σ (ns)",
    yaxis_title="Measured Peak-to-Peak Relative Jitter (ns)",
    height=550,
    template="plotly_white"
)

fig.show()

# ------------------- LINEAR FIT -------------------
print("\nExtracted Peak-to-Peak Jitter Relationship:")

for freq in FREQ_LIST:
    sub = df[df["Frequency (MHz)"] == freq]
    coeffs = np.polyfit(
        sub["Injected σ (ns)"],
        sub["Measured Pk-Pk Jitter (ns)"],
        1
    )
    print(f"{freq:.2f} MHz → Pk–Pk ≈ {coeffs[0]:.3f}·σ + {coeffs[1]:.3f}")


#

# Periodic Jitter

## Case 1

### 1.1

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from datetime import datetime

# Set your Google Drive save folder
base_folder = '/content/drive/MyDrive/Jitter_Simulations'
os.makedirs(base_folder, exist_ok=True)

# USER INPUTS
NUM_FREQS = int(input("Enter number of frequencies to test: ") or 2)
FREQ_LIST = []

for i in range(NUM_FREQS):
    freq = float(input(f"Enter frequency {i+1} (MHz): ") or 40.0)
    FREQ_LIST.append(freq)

NUM_TRIALS = int(input("Enter number of simulation runs per frequency: ") or 10)

# Option to test a range of σ values
range_choice = input("Do you want to run across a σ range? (y/n): ").strip().lower()
if range_choice == 'y':
    SIGMA_START = int(input("Enter starting σ value (integer): ") or 1)
    SIGMA_END = int(input("Enter ending σ value (integer): ") or 5)
else:
    single_sigma = float(input("Enter single jitter amplitude σ (ns): ") or 1.0)
    SIGMA_START = SIGMA_END = int(single_sigma)

# SIMULATION PARAMETERS
SIM_DURATION = 100  # ns
NUM_STEPS = 100000
t = np.linspace(0, SIM_DURATION, NUM_STEPS)

# FUNCTION TO RUN ONE SIMULATION (PERIODIC JITTER)
def run_one_simulation(t, period_ns, sigma_ns):
    # Ideal pulse
    clk_ideal = ((t % period_ns) < (period_ns / 2)).astype(int)
    ext_pulse_ideal = clk_ideal.copy()

    # ---- PERIODIC JITTER ----
    jitter_freq = 1 / (10 * period_ns)  # jitter frequency = 1/10 clock frequency
    jitter_values = sigma_ns * np.sin(2 * np.pi * jitter_freq * t)

    t_jittered = t + jitter_values
    ext_pulse_jittered = (((t_jittered % period_ns) < (period_ns / 2))).astype(int)

    # Error detection
    error_signal = np.where(ext_pulse_ideal != ext_pulse_jittered, 1, 0)
    total_errors = np.sum(error_signal)

    return total_errors, ext_pulse_ideal, ext_pulse_jittered, error_signal, jitter_values

# MAIN SIMULATION LOOP
sigma_values = list(range(SIGMA_START, SIGMA_END + 1))
all_mean_errors = {}
results = []

for freq in FREQ_LIST:
    print(f"\n=== Running simulations for {freq:.2f} MHz ===")
    period_ns = 1000 / freq
    mean_errors_list = []

    for sigma in sigma_values:
        error_counts = []
        for _ in range(NUM_TRIALS):
            total_errors, ext_pulse_ideal, ext_pulse_jittered, error_signal, jitter_values = \
                run_one_simulation(t, period_ns, sigma)
            error_counts.append(total_errors)

        mean_error = np.mean(error_counts)
        mean_errors_list.append(mean_error)
        results.append({
            "Frequency (MHz)": freq,
            "Jitter Amplitude (ns)": sigma,
            "Mean Total Errors": mean_error
        })

    all_mean_errors[freq] = mean_errors_list

# SUMMARY
print("\n--- PERIODIC JITTER ERROR SUMMARY ---")
print(f"Frequencies Tested  : {', '.join([f'{f:.2f} MHz' for f in FREQ_LIST])}")
print(f"Number of Runs      : {NUM_TRIALS}")
print(f"Jitter Amplitude Range Tested : {SIGMA_START} to {SIGMA_END}")

# EXPORT RESULTS TO DRIVE
df = pd.DataFrame(results)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"PeriodicJitterSimCase1_{NUM_STEPS}samples_{NUM_TRIALS}runs_{timestamp}.csv"
save_path = os.path.join(base_folder, filename)
df.to_csv(save_path, index=False)

print(f"\nData exported successfully to:\n{save_path}")

# PLOT 1 — Last Frequency’s Waveforms
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=("Ideal External Pulse",
                                    "Periodic Jittered Pulse",
                                    "Error (Difference)"),
                    vertical_spacing=0.05)

fig.add_trace(go.Scatter(x=t, y=ext_pulse_ideal*2,
                         mode='lines', line=dict(color='blue', width=3),
                         line_shape='hv'), 1, 1)

fig.add_trace(go.Scatter(x=t, y=ext_pulse_jittered*2,
                         mode='lines', line=dict(color='orange', width=3),
                         line_shape='hv'), 2, 1)

fig.add_trace(go.Scatter(x=t, y=error_signal*2,
                         mode='lines', line=dict(color='red', width=2),
                         line_shape='hv'), 3, 1)

fig.update_layout(
    title=f"Periodic Jitter Simulation — A={sigma_values[-1]:.2f} ns @ {FREQ_LIST[-1]:.2f} MHz",
    height=650,
    hovermode='x unified',
    showlegend=False,
    xaxis3_title="Time (ns)",
    yaxis=dict(showticklabels=False)
)
fig.show()

# PLOT 2 — Mean Total Errors vs Jitter Amplitude
fig2 = go.Figure()
colors = ['purple', 'teal', 'orange', 'green', 'red', 'blue']

for idx, freq in enumerate(FREQ_LIST):
    fig2.add_trace(go.Scatter(
        x=sigma_values,
        y=all_mean_errors[freq],
        mode='lines+markers',
        line=dict(color=colors[idx % len(colors)], width=3),
        marker=dict(size=8),
        name=f'{freq:.2f} MHz'
    ))

fig2.update_layout(
    title="Mean Total Errors vs Periodic Jitter Amplitude",
    xaxis_title="Jitter Amplitude (ns)",
    yaxis_title="Mean Total Errors",
    height=550,
    template="plotly_white",
    legend_title="Frequencies"
)
fig2.show()


## Case 2

### 2.1

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from datetime import datetime

# Set your Drive save folder here
base_folder = '/content/drive/MyDrive/Jitter_Simulations'
os.makedirs(base_folder, exist_ok=True)

# USER INPUTS
NUM_FREQS = int(input("Enter number of frequencies to test: ") or 2)
FREQ_LIST = []

for i in range(NUM_FREQS):
    freq = float(input(f"Enter frequency {i+1} (MHz): ") or 40.0)
    FREQ_LIST.append(freq)

NUM_TRIALS = int(input("Enter number of simulation runs per frequency: ") or 10)

# Option to test a range of jitter amplitudes
range_choice = input("Do you want to run across a σ range? (y/n): ").strip().lower()
if range_choice == 'y':
    SIGMA_START = int(input("Enter starting σ value (integer): ") or 1)
    SIGMA_END = int(input("Enter ending σ value (integer): ") or 5)
else:
    single_sigma = float(input("Enter single jitter amplitude σ (ns): ") or 1.0)
    SIGMA_START = SIGMA_END = int(single_sigma)

# SIMULATION PARAMETERS
SIM_DURATION = 100  # ns
NUM_STEPS = 100000
t = np.linspace(0, SIM_DURATION, NUM_STEPS)

# FUNCTION TO RUN ONE SIMULATION (PERIODIC CLOCK JITTER)
def run_one_simulation(t, period_ns, sigma_ns):
    # Ideal external pulse
    ext_pulse_ideal = ((t % period_ns) < (period_ns / 2)).astype(int)

    # ---- PERIODIC CLOCK JITTER ----
    jitter_freq = 1 / (10 * period_ns)  # jitter frequency
    jitter_values = sigma_ns * np.sin(2 * np.pi * jitter_freq * t)

    t_jittered = t + jitter_values
    clk_jittered = (((t_jittered % period_ns) < (period_ns / 2))).astype(int)

    # Error detection
    error_signal = np.where(ext_pulse_ideal != clk_jittered, 1, 0)
    total_errors = np.sum(error_signal)

    return total_errors, ext_pulse_ideal, clk_jittered, error_signal, jitter_values

# MAIN SIMULATION LOOP
sigma_values = list(range(SIGMA_START, SIGMA_END + 1))
all_mean_errors = {}
results = []

for freq in FREQ_LIST:
    print(f"\n=== Running simulations for {freq:.2f} MHz ===")
    period_ns = 1000 / freq
    mean_errors_list = []

    for sigma in sigma_values:
        error_counts = []
        for _ in range(NUM_TRIALS):
            total_errors, ext_pulse_ideal, clk_jittered, error_signal, jitter_values = \
                run_one_simulation(t, period_ns, sigma)
            error_counts.append(total_errors)

        mean_error = np.mean(error_counts)
        mean_errors_list.append(mean_error)
        results.append({
            "Frequency (MHz)": freq,
            "Jitter Amplitude (ns)": sigma,
            "Mean Total Errors": mean_error
        })

    all_mean_errors[freq] = mean_errors_list

# SUMMARY
print("\n--- CASE 2: CLOCK PERIODIC JITTER / EXTERNAL PULSE IDEAL ---")
print(f"Frequencies Tested  : {', '.join([f'{f:.2f} MHz' for f in FREQ_LIST])}")
print(f"Number of Runs      : {NUM_TRIALS}")
print(f"Jitter Amplitude Range Tested : {SIGMA_START} to {SIGMA_END}")

# EXPORT TO DRIVE AS CSV WITH TIMESTAMP
df = pd.DataFrame(results)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"Case2_PeriodicClockJitter_{NUM_STEPS}samples_{NUM_TRIALS}runs_{timestamp}.csv"
save_path = os.path.join(base_folder, filename)
df.to_csv(save_path, index=False)

print(f"\nData exported successfully to:\n{save_path}")

# PLOT 1 — Last Frequency’s Waveforms
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=("Ideal External Pulse",
                                    "Periodic Jittered Clock Output",
                                    "Error (Difference)"),
                    vertical_spacing=0.05)

fig.add_trace(go.Scatter(x=t, y=ext_pulse_ideal*2,
                         mode='lines', line=dict(color='blue', width=3),
                         line_shape='hv'), 1, 1)

fig.add_trace(go.Scatter(x=t, y=clk_jittered*2,
                         mode='lines', line=dict(color='orange', width=3),
                         line_shape='hv'), 2, 1)

fig.add_trace(go.Scatter(x=t, y=error_signal*2,
                         mode='lines', line=dict(color='red', width=2),
                         line_shape='hv'), 3, 1)

fig.update_layout(
    title=f"Case 2 — Periodic Clock Jitter vs Ideal External Pulse "
          f"(A = {sigma_values[-1]:.2f} ns @ {FREQ_LIST[-1]:.2f} MHz)",
    height=650,
    hovermode='x unified',
    showlegend=False,
    xaxis3_title="Time (ns)",
    yaxis=dict(showticklabels=False)
)
fig.show()

# PLOT 2 — Mean Total Errors vs Jitter Amplitude
fig2 = go.Figure()
colors = ['purple', 'teal', 'orange', 'green', 'red', 'blue']

for idx, freq in enumerate(FREQ_LIST):
    fig2.add_trace(go.Scatter(
        x=sigma_values,
        y=all_mean_errors[freq],
        mode='lines+markers',
        line=dict(color=colors[idx % len(colors)], width=3),
        marker=dict(size=8),
        name=f'{freq:.2f} MHz'
    ))

fig2.update_layout(
    title="Case 2 — Mean Total Errors vs Periodic Clock Jitter Amplitude",
    xaxis_title="Clock Jitter Amplitude (ns)",
    yaxis_title="Mean Total Errors",
    height=550,
    template="plotly_white",
    legend_title="Frequencies"
)
fig2.show()


## Case 3

### 3.1

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from datetime import datetime

# Set your Drive save folder here
base_folder = '/content/drive/MyDrive/Jitter_Simulations'
os.makedirs(base_folder, exist_ok=True)

# USER INPUTS
NUM_FREQS = int(input("Enter number of frequencies to test: ") or 2)
FREQ_LIST = [float(input(f"Enter frequency {i+1} (MHz): ") or 40.0) for i in range(NUM_FREQS)]
NUM_TRIALS = int(input("Enter number of simulation runs per frequency: ") or 10)

# Option to test a range of jitter amplitudes
range_choice = input("Do you want to run across a σ range? (y/n): ").strip().lower()
if range_choice == 'y':
    SIGMA_START = int(input("Enter starting σ value (integer): ") or 1)
    SIGMA_END = int(input("Enter ending σ value (integer): ") or 5)
else:
    single_sigma = float(input("Enter single jitter amplitude σ (ns): ") or 1.0)
    SIGMA_START = SIGMA_END = int(single_sigma)

# SIMULATION PARAMETERS
SIM_DURATION = 100  # ns
NUM_STEPS = 100000
t = np.linspace(0, SIM_DURATION, NUM_STEPS)

# FUNCTION TO RUN ONE SIMULATION — BOTH CLOCK AND EXTERNAL PULSE PERIODIC JITTER
def run_one_simulation(t, period_ns, sigma_ns):
    jitter_freq = 1 / (10 * period_ns)

    # Independent periodic jitter (different phase)
    phase_clk = np.random.uniform(0, 2*np.pi)
    phase_ext = np.random.uniform(0, 2*np.pi)

    jitter_clk = sigma_ns * np.sin(2 * np.pi * jitter_freq * t + phase_clk)
    jitter_ext = sigma_ns * np.sin(2 * np.pi * jitter_freq * t + phase_ext)

    t_clk_jittered = t + jitter_clk
    t_ext_jittered = t + jitter_ext

    clk_jittered = (((t_clk_jittered % period_ns) < (period_ns / 2))).astype(int)
    ext_pulse_jittered = (((t_ext_jittered % period_ns) < (period_ns / 2))).astype(int)

    error_signal = np.where(ext_pulse_jittered != clk_jittered, 1, 0)
    total_errors = np.sum(error_signal)

    return total_errors, clk_jittered, ext_pulse_jittered, error_signal, jitter_clk, jitter_ext

# MAIN SIMULATION LOOP
sigma_values = list(range(SIGMA_START, SIGMA_END + 1))
all_mean_errors = {}
results = []

last_clk_jittered = None
last_ext_pulse_jittered = None
last_error_signal = None

for freq in FREQ_LIST:
    print(f"\n=== Running simulations for {freq:.2f} MHz ===")
    period_ns = 1000 / freq
    mean_errors_list = []

    for sigma in sigma_values:
        error_counts = []
        for _ in range(NUM_TRIALS):
            total_errors, clk_jittered, ext_pulse_jittered, error_signal, jitter_clk, jitter_ext = \
                run_one_simulation(t, period_ns, sigma)
            error_counts.append(total_errors)

            last_clk_jittered = clk_jittered
            last_ext_pulse_jittered = ext_pulse_jittered
            last_error_signal = error_signal

        mean_error = np.mean(error_counts)
        mean_errors_list.append(mean_error)
        results.append({
            "Frequency (MHz)": freq,
            "Jitter Amplitude (ns)": sigma,
            "Mean Total Errors": mean_error
        })

    all_mean_errors[freq] = mean_errors_list

# SUMMARY
print("\n--- CASE 3: CLOCK + EXTERNAL PULSE BOTH PERIODICALLY JITTERED ---")
print(f"Frequencies Tested  : {', '.join([f'{f:.2f} MHz' for f in FREQ_LIST])}")
print(f"Number of Runs      : {NUM_TRIALS}")
print(f"Jitter Amplitude Range Tested : {SIGMA_START} to {SIGMA_END}")

# EXPORT TO DRIVE AS CSV WITH TIMESTAMP
df = pd.DataFrame(results)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"Case3_PeriodicJitter_{NUM_STEPS}samples_{NUM_TRIALS}runs_{timestamp}.csv"
save_path = os.path.join(base_folder, filename)
df.to_csv(save_path, index=False)

print(f"\nData exported successfully to:\n{save_path}")

# PLOT 1 — Last Frequency’s Waveforms
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=("Jittered External Pulse",
                                    "Jittered Clock Output",
                                    "Error (Difference)"),
                    vertical_spacing=0.05)

fig.add_trace(go.Scatter(x=t, y=last_ext_pulse_jittered*2,
                         mode='lines', line=dict(color='blue', width=3),
                         line_shape='hv'), 1, 1)

fig.add_trace(go.Scatter(x=t, y=last_clk_jittered*2,
                         mode='lines', line=dict(color='orange', width=3),
                         line_shape='hv'), 2, 1)

fig.add_trace(go.Scatter(x=t, y=last_error_signal*2,
                         mode='lines', line=dict(color='red', width=2),
                         line_shape='hv'), 3, 1)

fig.update_layout(
    title=f"Case 3 — Clock + External Pulse Periodic Jitter "
          f"(A = {sigma_values[-1]:.2f} ns @ {FREQ_LIST[-1]:.2f} MHz)",
    height=650,
    hovermode='x unified',
    showlegend=False,
    xaxis3_title="Time (ns)",
    yaxis=dict(showticklabels=False)
)
fig.show()

# PLOT 2 — Mean Total Errors vs Jitter Amplitude
fig2 = go.Figure()
colors = ['purple', 'teal', 'orange', 'green', 'red', 'blue']

for idx, freq in enumerate(FREQ_LIST):
    fig2.add_trace(go.Scatter(
        x=sigma_values,
        y=all_mean_errors[freq],
        mode='lines+markers',
        line=dict(color=colors[idx % len(colors)], width=3),
        marker=dict(size=8),
        name=f'{freq:.2f} MHz'
    ))

fig2.update_layout(
    title="Case 3 — Mean Total Errors vs Periodic Jitter Amplitude "
          "(Clock + External Pulse)",
    xaxis_title="Jitter Amplitude (ns)",
    yaxis_title="Mean Total Errors",
    height=550,
    template="plotly_white",
    legend_title="Frequencies"
)
fig2.show()


## Metrics_ExternalPulse

### 4.1.1_RMS

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import os
from datetime import datetime

base_folder = '/content/drive/MyDrive/Jitter_Simulations'
os.makedirs(base_folder, exist_ok=True)

NUM_FREQS = int(input("Enter number of frequencies to test: ") or 2)
FREQ_LIST = [float(input(f"Enter frequency {i+1} (MHz): ") or 40.0)
             for i in range(NUM_FREQS)]
NUM_TRIALS = int(input("Enter number of simulation runs per frequency: ") or 10)

range_choice = input("Do you want to run across a σ range? (y/n): ").strip().lower()
if range_choice == 'y':
    SIGMA_START = int(input("Enter starting σ value (ns): ") or 1)
    SIGMA_END = int(input("Enter ending σ value (ns): ") or 5)
else:
    single_sigma = float(input("Enter single jitter amplitude σ (ns): ") or 1.0)
    SIGMA_START = SIGMA_END = int(single_sigma)

sigma_values = list(range(SIGMA_START, SIGMA_END + 1))

SIM_DURATION = 100
NUM_STEPS = 100000
t = np.linspace(0, SIM_DURATION, NUM_STEPS)

# ------------------- RMS JITTER FUNCTION -------------------
def compute_rms_jitter(t, period_ns, sigma_ns):
    # PERIODIC jitter applied ONLY to external pulse
    jitter_freq = 1 / (10 * period_ns)
    jitter_ext = sigma_ns * np.sin(2 * np.pi * jitter_freq * t)

    # Ideal clock
    t_clk = t
    t_ext = t + jitter_ext

    clk = (((t_clk % period_ns) < (period_ns / 2))).astype(int)
    ext = (((t_ext % period_ns) < (period_ns / 2))).astype(int)

    clk_edges = np.where(np.diff(clk) == 1)[0]
    ext_edges = np.where(np.diff(ext) == 1)[0]

    num_edges = min(len(clk_edges), len(ext_edges))
    if num_edges < 5:
        return np.nan

    clk_edge_times = t_clk[clk_edges[:num_edges]]
    ext_edge_times = t_ext[ext_edges[:num_edges]]

    ideal_edges = np.arange(num_edges) * period_ns

    tie_clk = clk_edge_times - ideal_edges
    tie_ext = ext_edge_times - ideal_edges

    relative_jitter = tie_ext - tie_clk

    rms_relative = np.sqrt(np.mean(relative_jitter ** 2))
    return rms_relative

# ------------------- SIMULATION -------------------
results = []

for freq in FREQ_LIST:
    print(f"\nRunning RMS jitter simulation for {freq:.2f} MHz")
    period_ns = 1000 / freq

    for sigma in sigma_values:
        rms_vals = []
        for _ in range(NUM_TRIALS):
            rms_vals.append(compute_rms_jitter(t, period_ns, sigma))

        results.append({
            "Frequency (MHz)": freq,
            "Injected σ (ns)": sigma,
            "Measured RMS Jitter (ns)": np.nanmean(rms_vals)
        })

# ------------------- SAVE -------------------
df = pd.DataFrame(results)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"RMS_PeriodicJitter_ExtOnly_{NUM_STEPS}samples_{NUM_TRIALS}runs_{timestamp}.csv"
save_path = os.path.join(base_folder, filename)
df.to_csv(save_path, index=False)

print(f"\nRMS jitter data exported to:\n{save_path}")

# ------------------- PLOT -------------------
fig = go.Figure()
colors = ['blue', 'red', 'green', 'purple', 'orange', 'teal']

for i, freq in enumerate(FREQ_LIST):
    sub = df[df["Frequency (MHz)"] == freq]
    fig.add_trace(go.Scatter(
        x=sub["Injected σ (ns)"],
        y=sub["Measured RMS Jitter (ns)"],
        mode='lines+markers',
        line=dict(width=3, color=colors[i % len(colors)]),
        marker=dict(size=8),
        name=f"{freq:.2f} MHz"
    ))

fig.update_layout(
    title="RMS External Pulse Jitter vs Injected σ<br>(Ideal Clock Reference)",
    xaxis_title="Injected External Jitter Amplitude (ns)",
    yaxis_title="Measured RMS Jitter (ns)",
    height=550,
    template="plotly_white"
)

fig.show()

# ------------------- LINEAR FIT -------------------
print("\nExtracted RMS Jitter Relationship:")

for freq in FREQ_LIST:
    sub = df[df["Frequency (MHz)"] == freq]
    coeffs = np.polyfit(
        sub["Injected σ (ns)"],
        sub["Measured RMS Jitter (ns)"],
        1
    )
    print(f"{freq:.2f} MHz → RMS ≈ {coeffs[0]:.3f}·σ + {coeffs[1]:.3f}")

### 4.1.2_Pk-Pk

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import os
from datetime import datetime

# ------------------- STORAGE SETUP -------------------
base_folder = '/content/drive/MyDrive/Jitter_Simulations'
os.makedirs(base_folder, exist_ok=True)

# ------------------- USER INPUT -------------------
NUM_FREQS = int(input("Enter number of frequencies to test: ") or 2)
FREQ_LIST = [float(input(f"Enter frequency {i+1} (MHz): ") or 40.0)
             for i in range(NUM_FREQS)]
NUM_TRIALS = int(input("Enter number of simulation runs per frequency: ") or 10)

range_choice = input("Do you want to run across a σ range? (y/n): ").strip().lower()
if range_choice == 'y':
    SIGMA_START = int(input("Enter starting σ value (ns): ") or 1)
    SIGMA_END = int(input("Enter ending σ value (ns): ") or 5)
else:
    single_sigma = float(input("Enter single jitter amplitude σ (ns): ") or 1.0)
    SIGMA_START = SIGMA_END = int(single_sigma)

sigma_values = list(range(SIGMA_START, SIGMA_END + 1))

# ------------------- SIMULATION BASE -------------------
SIM_DURATION = 100
NUM_STEPS = 100000
t = np.linspace(0, SIM_DURATION, NUM_STEPS)

# ------------------- PEAK-TO-PEAK JITTER FUNCTION -------------------
def compute_p2p_jitter(t, period_ns, sigma_ns):
    # PERIODIC jitter applied ONLY to external pulse
    jitter_freq = 1 / (10 * period_ns)
    jitter_ext = sigma_ns * np.sin(2 * np.pi * jitter_freq * t)

    # Ideal clock
    t_clk = t
    t_ext = t + jitter_ext

    clk = (((t_clk % period_ns) < (period_ns / 2))).astype(int)
    ext = (((t_ext % period_ns) < (period_ns / 2))).astype(int)

    clk_edges = np.where(np.diff(clk) == 1)[0]
    ext_edges = np.where(np.diff(ext) == 1)[0]

    num_edges = min(len(clk_edges), len(ext_edges))
    if num_edges < 5:
        return np.nan

    clk_edge_times = t_clk[clk_edges[:num_edges]]
    ext_edge_times = t_ext[ext_edges[:num_edges]]

    ideal_edges = np.arange(num_edges) * period_ns

    tie_clk = clk_edge_times - ideal_edges
    tie_ext = ext_edge_times - ideal_edges

    relative_jitter = tie_ext - tie_clk

    # Peak-to-Peak Metric
    peak_to_peak = np.max(relative_jitter) - np.min(relative_jitter)
    return peak_to_peak

# ------------------- SIMULATION LOOP -------------------
results = []

for freq in FREQ_LIST:
    print(f"\nRunning Peak-to-Peak jitter simulation for {freq:.2f} MHz")
    period_ns = 1000 / freq

    for sigma in sigma_values:
        p2p_vals = []
        for _ in range(NUM_TRIALS):
            p2p_vals.append(compute_p2p_jitter(t, period_ns, sigma))

        results.append({
            "Frequency (MHz)": freq,
            "Injected σ (ns)": sigma,
            "Measured Peak-to-Peak Jitter (ns)": np.nanmean(p2p_vals)
        })

# ------------------- SAVE RESULTS -------------------
df = pd.DataFrame(results)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"P2P_PeriodicJitter_ExtOnly_{NUM_STEPS}samples_{NUM_TRIALS}runs_{timestamp}.csv"
save_path = os.path.join(base_folder, filename)

df.to_csv(save_path, index=False)

print(f"\nPeak-to-Peak jitter data exported to:\n{save_path}")

# ------------------- PLOT -------------------
fig = go.Figure()
colors = ['blue', 'red', 'green', 'purple', 'orange', 'teal']

for i, freq in enumerate(FREQ_LIST):
    sub = df[df["Frequency (MHz)"] == freq]
    fig.add_trace(go.Scatter(
        x=sub["Injected σ (ns)"],
        y=sub["Measured Peak-to-Peak Jitter (ns)"],
        mode='lines+markers',
        line=dict(width=3, color=colors[i % len(colors)]),
        marker=dict(size=8),
        name=f"{freq:.2f} MHz"
    ))

fig.update_layout(
    title="Step 1: Peak-to-Peak External Pulse Jitter vs Injected σ<br>(Ideal Clock Reference)",
    xaxis_title="Injected External Jitter Amplitude (ns)",
    yaxis_title="Measured Peak-to-Peak Jitter (ns)",
    height=550,
    template="plotly_white"
)

fig.show()

# ------------------- LINEAR FIT -------------------
print("\nStep 1 Result: Peak-to-Peak Jitter Dependency")

for freq in FREQ_LIST:
    sub = df[df["Frequency (MHz)"] == freq]
    coeffs = np.polyfit(
        sub["Injected σ (ns)"],
        sub["Measured Peak-to-Peak Jitter (ns)"],
        1
    )
    print(f"{freq:.2f} MHz → P2P ≈ {coeffs[0]:.3f}·σ + {coeffs[1]:.3f}")

###4.1.3_rmsandpk-pk

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import os
from datetime import datetime

# ------------------- STORAGE SETUP -------------------
base_folder = '/content/drive/MyDrive/Jitter_Simulations'
os.makedirs(base_folder, exist_ok=True)

# ------------------- USER INPUT -------------------
NUM_FREQS = int(input("Enter number of frequencies to test: ") or 2)
FREQ_LIST = [float(input(f"Enter frequency {i+1} (MHz): ") or 40.0)
             for i in range(NUM_FREQS)]
NUM_TRIALS = int(input("Enter number of simulation runs per frequency: ") or 10)

range_choice = input("Do you want to run across a σ range? (y/n): ").strip().lower()
if range_choice == 'y':
    SIGMA_START = int(input("Enter starting σ value (ns): ") or 1)
    SIGMA_END = int(input("Enter ending σ value (ns): ") or 5)
else:
    single_sigma = float(input("Enter single jitter amplitude σ (ns): ") or 1.0)
    SIGMA_START = SIGMA_END = int(single_sigma)

sigma_values = list(range(SIGMA_START, SIGMA_END + 1))

# ------------------- SIMULATION BASE -------------------
SIM_DURATION = 100
NUM_STEPS = 100000
t = np.linspace(0, SIM_DURATION, NUM_STEPS)

# ------------------- DUAL METRIC FUNCTION -------------------
def compute_jitter_metrics(t, period_ns, sigma_ns):
    jitter_freq = 1 / (10 * period_ns)
    jitter_ext = sigma_ns * np.sin(2 * np.pi * jitter_freq * t)

    t_clk = t
    t_ext = t + jitter_ext

    clk = (((t_clk % period_ns) < (period_ns / 2))).astype(int)
    ext = (((t_ext % period_ns) < (period_ns / 2))).astype(int)

    clk_edges = np.where(np.diff(clk) == 1)[0]
    ext_edges = np.where(np.diff(ext) == 1)[0]

    num_edges = min(len(clk_edges), len(ext_edges))
    if num_edges < 5:
        return np.nan, np.nan

    clk_edge_times = t_clk[clk_edges[:num_edges]]
    ext_edge_times = t_ext[ext_edges[:num_edges]]

    ideal_edges = np.arange(num_edges) * period_ns

    tie_clk = clk_edge_times - ideal_edges
    tie_ext = ext_edge_times - ideal_edges

    relative_jitter = tie_ext - tie_clk

    # RMS
    rms = np.sqrt(np.mean(relative_jitter ** 2))

    # Peak-to-Peak
    p2p = np.max(relative_jitter) - np.min(relative_jitter)

    return rms, p2p

# ------------------- SIMULATION -------------------
results = []

for freq in FREQ_LIST:
    print(f"\nRunning dual-metric jitter simulation for {freq:.2f} MHz")
    period_ns = 1000 / freq

    for sigma in sigma_values:
        rms_vals = []
        p2p_vals = []

        for _ in range(NUM_TRIALS):
            rms, p2p = compute_jitter_metrics(t, period_ns, sigma)
            rms_vals.append(rms)
            p2p_vals.append(p2p)

        results.append({
            "Frequency (MHz)": freq,
            "Injected σ (ns)": sigma,
            "Measured RMS Jitter (ns)": np.nanmean(rms_vals),
            "Measured Peak-to-Peak Jitter (ns)": np.nanmean(p2p_vals)
        })

# ------------------- SAVE -------------------
df = pd.DataFrame(results)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"DualMetric_Jitter_{NUM_STEPS}samples_{NUM_TRIALS}runs_{timestamp}.csv"
save_path = os.path.join(base_folder, filename)

df.to_csv(save_path, index=False)

print(f"\nJitter data (RMS + P2P) exported to:\n{save_path}")

# ------------------- PLOT -------------------
fig = go.Figure()
colors = ['blue', 'red', 'green', 'purple', 'orange', 'teal']

for i, freq in enumerate(FREQ_LIST):
    sub = df[df["Frequency (MHz)"] == freq]

    # RMS curve
    fig.add_trace(go.Scatter(
        x=sub["Injected σ (ns)"],
        y=sub["Measured RMS Jitter (ns)"],
        mode='lines+markers',
        line=dict(width=3, dash='solid', color=colors[i % len(colors)]),
        name=f"{freq:.2f} MHz (RMS)"
    ))

    # P2P curve
    fig.add_trace(go.Scatter(
        x=sub["Injected σ (ns)"],
        y=sub["Measured Peak-to-Peak Jitter (ns)"],
        mode='lines+markers',
        line=dict(width=3, dash='dash', color=colors[i % len(colors)]),
        name=f"{freq:.2f} MHz (P2P)"
    ))

fig.update_layout(
    title="RMS vs Peak-to-Peak Jitter vs Injected σ<br>(External Pulse Only)",
    xaxis_title="Injected Jitter Amplitude σ (ns)",
    yaxis_title="Jitter (ns)",
    height=550,
    template="plotly_white"
)

fig.show()

# ------------------- LINEAR FIT -------------------
print("\nExtracted Jitter Relationships:")

for freq in FREQ_LIST:
    sub = df[df["Frequency (MHz)"] == freq]

    coeffs_rms = np.polyfit(sub["Injected σ (ns)"],
                            sub["Measured RMS Jitter (ns)"], 1)

    coeffs_p2p = np.polyfit(sub["Injected σ (ns)"],
                            sub["Measured Peak-to-Peak Jitter (ns)"], 1)

    print(f"\n{freq:.2f} MHz:")
    print(f"  RMS ≈ {coeffs_rms[0]:.3f}·σ + {coeffs_rms[1]:.3f}")
    print(f"  P2P ≈ {coeffs_p2p[0]:.3f}·σ + {coeffs_p2p[1]:.3f}")

## Metrics_Clock

###4.2.1_RMS

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import os
from datetime import datetime

# DRIVE SETUP
base_folder = '/content/drive/MyDrive/Jitter_Simulations'
os.makedirs(base_folder, exist_ok=True)

# USER INPUTS
NUM_FREQS = int(input("Enter number of frequencies to test: ") or 2)

FREQ_LIST = [
    float(input(f"Enter frequency {i+1} (MHz): ") or 40.0)
    for i in range(NUM_FREQS)
]

NUM_TRIALS = int(input("Enter number of simulation runs per frequency: ") or 10)

range_choice = input("Do you want to run across a σ range? (y/n): ").strip().lower()

if range_choice == 'y':
    SIGMA_START = int(input("Enter starting σ value (ns): ") or 1)
    SIGMA_END = int(input("Enter ending σ value (ns): ") or 5)
else:
    single_sigma = float(input("Enter single periodic jitter amplitude σ (ns): ") or 1.0)
    SIGMA_START = SIGMA_END = int(single_sigma)

sigma_values = list(range(SIGMA_START, SIGMA_END + 1))

# SIMULATION GRID
SIM_DURATION = 100          # ns
NUM_STEPS = 100_000
t = np.linspace(0, SIM_DURATION, NUM_STEPS)


# PERIODIC JITTER FUNCTION
def compute_pp_jitter(t, period_ns, sigma_ns):

    # Periodic jitter frequency
    jitter_freq = 1 / (10 * period_ns)

    # Periodic sinusoidal jitter applied ONLY to clock
    jitter_clk = sigma_ns * np.sin(2 * np.pi * jitter_freq * t)

    # Ideal external trigger
    t_ext = t

    # Jittered clock
    t_clk = t + jitter_clk

    # Generate signals
    ext = (((t_ext % period_ns) < (period_ns / 2))).astype(int)
    clk = (((t_clk % period_ns) < (period_ns / 2))).astype(int)

    # Rising edge extraction
    ext_edges = np.where(np.diff(ext) == 1)[0]
    clk_edges = np.where(np.diff(clk) == 1)[0]

    num_edges = min(len(ext_edges), len(clk_edges))

    if num_edges < 5:
        return np.nan

    ext_edge_times = t_ext[ext_edges[:num_edges]]
    clk_edge_times = t_clk[clk_edges[:num_edges]]

    # Ideal edge locations
    ideal_edges = np.arange(num_edges) * period_ns

    # Time Interval Error (TIE)
    tie_ext = ext_edge_times - ideal_edges
    tie_clk = clk_edge_times - ideal_edges

    # Relative jitter
    relative_jitter = tie_clk - tie_ext

    # Peak-to-Peak jitter
    pp_relative = np.max(relative_jitter) - np.min(relative_jitter)

    return pp_relative


# MAIN SIMULATION
results = []

for freq in FREQ_LIST:

    print(f"\nRunning Peak-to-Peak periodic jitter simulation for {freq:.2f} MHz")

    period_ns = 1000 / freq

    for sigma in sigma_values:

        pp_vals = []

        for _ in range(NUM_TRIALS):

            pp_vals.append(
                compute_pp_jitter(t, period_ns, sigma)
            )

        results.append({
            "Frequency (MHz)": freq,
            "Injected σ (ns)": sigma,
            "Measured Peak-to-Peak Jitter (ns)": np.nanmean(pp_vals)
        })


# SAVE RESULTS
df = pd.DataFrame(results)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

filename = (
    f"PP_PeriodicJitter_ClockOnly_"
    f"{NUM_STEPS}samples_"
    f"{NUM_TRIALS}runs_"
    f"{timestamp}.csv"
)

save_path = os.path.join(base_folder, filename)

df.to_csv(save_path, index=False)

print(f"\nPeak-to-Peak periodic jitter data exported to:\n{save_path}")


# PLOT
fig = go.Figure()

colors = ['blue', 'red', 'green', 'purple', 'orange', 'teal']

for i, freq in enumerate(FREQ_LIST):

    sub = df[df["Frequency (MHz)"] == freq]

    fig.add_trace(go.Scatter(
        x=sub["Injected σ (ns)"],
        y=sub["Measured Peak-to-Peak Jitter (ns)"],
        mode='lines+markers',
        line=dict(
            width=3,
            color=colors[i % len(colors)]
        ),
        marker=dict(size=8),
        name=f"{freq:.2f} MHz"
    ))

fig.update_layout(
    title=(
        "Peak-to-Peak Periodic Clock Jitter vs Injected σ"
        "<br>(Ideal External Trigger Reference)"
    ),
    xaxis_title="Injected Periodic Clock Jitter Amplitude σ (ns)",
    yaxis_title="Measured Peak-to-Peak Jitter (ns)",
    height=550,
    template="plotly_white"
)

fig.show()


# LINEAR FIT
print("\nExtracted Peak-to-Peak Periodic Jitter Relationship:")

for freq in FREQ_LIST:

    sub = df[df["Frequency (MHz)"] == freq]

    coeffs = np.polyfit(
        sub["Injected σ (ns)"],
        sub["Measured Peak-to-Peak Jitter (ns)"],
        1
    )

    print(
        f"{freq:.2f} MHz → "
        f"P-P ≈ {coeffs[0]:.3f}·σ + {coeffs[1]:.3f}"
    )

### 4.2.2_Pk-Pk

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import os
from datetime import datetime

# DRIVE SETUP
base_folder = '/content/drive/MyDrive/Jitter_Simulations'
os.makedirs(base_folder, exist_ok=True)

# USER INPUTS
NUM_FREQS = int(input("Enter number of frequencies to test: ") or 2)

FREQ_LIST = [
    float(input(f"Enter frequency {i+1} (MHz): ") or 40.0)
    for i in range(NUM_FREQS)
]

NUM_TRIALS = int(input("Enter number of simulation runs per frequency: ") or 10)

range_choice = input("Do you want to run across a σ range? (y/n): ").strip().lower()

if range_choice == 'y':
    SIGMA_START = int(input("Enter starting σ value (ns): ") or 1)
    SIGMA_END = int(input("Enter ending σ value (ns): ") or 5)
else:
    single_sigma = float(input("Enter single periodic jitter amplitude σ (ns): ") or 1.0)
    SIGMA_START = SIGMA_END = int(single_sigma)

sigma_values = list(range(SIGMA_START, SIGMA_END + 1))

# SIMULATION GRID
SIM_DURATION = 100        # ns
NUM_STEPS = 100_000
t = np.linspace(0, SIM_DURATION, NUM_STEPS)

# -------------------------------------------------------
# PEAK-TO-PEAK JITTER FUNCTION
# -------------------------------------------------------
def compute_pp_jitter(t, period_ns, sigma_ns):

    # Periodic jitter applied ONLY to clock
    jitter_freq = 1 / (10 * period_ns)

    # Sinusoidal periodic jitter
    jitter_clk = sigma_ns * np.sin(2 * np.pi * jitter_freq * t)

    # Ideal external trigger
    t_ext = t

    # Jittered clock
    t_clk = t + jitter_clk

    ext = (((t_ext % period_ns) < (period_ns / 2))).astype(int)
    clk = (((t_clk % period_ns) < (period_ns / 2))).astype(int)

    # Rising edges
    ext_edges = np.where(np.diff(ext) == 1)[0]
    clk_edges = np.where(np.diff(clk) == 1)[0]

    num_edges = min(len(ext_edges), len(clk_edges))

    if num_edges < 5:
        return np.nan

    ext_edge_times = t_ext[ext_edges[:num_edges]]
    clk_edge_times = t_clk[clk_edges[:num_edges]]

    # Ideal edge times
    ideal_edges = np.arange(num_edges) * period_ns

    # Time Interval Error (TIE)
    tie_ext = ext_edge_times - ideal_edges
    tie_clk = clk_edge_times - ideal_edges

    # Relative jitter (clock vs ideal reference)
    relative_jitter = tie_clk - tie_ext

    # Peak-to-Peak jitter
    pp_relative = np.max(relative_jitter) - np.min(relative_jitter)

    return pp_relative

# -------------------------------------------------------
# MAIN SIMULATION
# -------------------------------------------------------
results = []

for freq in FREQ_LIST:

    print(f"\nRunning Peak-to-Peak periodic jitter simulation for {freq:.2f} MHz")

    period_ns = 1000 / freq

    for sigma in sigma_values:

        pp_vals = []

        for _ in range(NUM_TRIALS):

            pp_vals.append(
                compute_pp_jitter(t, period_ns, sigma)
            )

        results.append({
            "Frequency (MHz)": freq,
            "Injected σ (ns)": sigma,
            "Measured Peak-to-Peak Jitter (ns)": np.nanmean(pp_vals)
        })

# -------------------------------------------------------
# SAVE RESULTS
# -------------------------------------------------------
df = pd.DataFrame(results)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

filename = (
    f"PP_PeriodicJitter_ClockOnly_"
    f"{NUM_STEPS}samples_"
    f"{NUM_TRIALS}runs_"
    f"{timestamp}.csv"
)

save_path = os.path.join(base_folder, filename)

df.to_csv(save_path, index=False)

print(f"\nPeak-to-Peak periodic jitter data exported to:\n{save_path}")

# -------------------------------------------------------
# PLOT
# -------------------------------------------------------
fig = go.Figure()

colors = ['blue', 'red', 'green', 'purple', 'orange', 'teal']

for i, freq in enumerate(FREQ_LIST):

    sub = df[df["Frequency (MHz)"] == freq]

    fig.add_trace(go.Scatter(
        x=sub["Injected σ (ns)"],
        y=sub["Measured Peak-to-Peak Jitter (ns)"],
        mode='lines+markers',
        line=dict(
            width=3,
            color=colors[i % len(colors)]
        ),
        marker=dict(size=8),
        name=f"{freq:.2f} MHz"
    ))

fig.update_layout(
    title=(
        "Peak-to-Peak Periodic Clock Jitter vs Injected σ"
        "<br>(Ideal External Trigger Reference)"
    ),
    xaxis_title="Injected Periodic Clock Jitter σ (ns)",
    yaxis_title="Measured Peak-to-Peak Jitter (ns)",
    height=550,
    template="plotly_white"
)

fig.show()

# -------------------------------------------------------
# LINEAR FIT
# -------------------------------------------------------
print("\nExtracted Peak-to-Peak Periodic Jitter Relationship:")

for freq in FREQ_LIST:

    sub = df[df["Frequency (MHz)"] == freq]

    coeffs = np.polyfit(
        sub["Injected σ (ns)"],
        sub["Measured Peak-to-Peak Jitter (ns)"],
        1
    )

    print(
        f"{freq:.2f} MHz → "
        f"P-P ≈ {coeffs[0]:.3f}·σ + {coeffs[1]:.3f}"
    )

### 4.2.3_rms&pk-pk //

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import os
from datetime import datetime

# DRIVE SETUP
base_folder = '/content/drive/MyDrive/Jitter_Simulations'
os.makedirs(base_folder, exist_ok=True)

# USER INPUTS
NUM_FREQS = int(input("Enter number of frequencies to test: ") or 2)

FREQ_LIST = [
    float(input(f"Enter frequency {i+1} (MHz): ") or 40.0)
    for i in range(NUM_FREQS)
]

NUM_TRIALS = int(input("Enter number of simulation runs per frequency: ") or 10)

range_choice = input("Do you want to run across a σ range? (y/n): ").strip().lower()

if range_choice == 'y':
    SIGMA_START = int(input("Enter starting σ value (ns): ") or 1)
    SIGMA_END = int(input("Enter ending σ value (ns): ") or 5)
else:
    single_sigma = float(input("Enter single periodic jitter amplitude σ (ns): ") or 1.0)
    SIGMA_START = SIGMA_END = int(single_sigma)

sigma_values = list(range(SIGMA_START, SIGMA_END + 1))

# SIMULATION GRID
SIM_DURATION = 100          # ns
NUM_STEPS = 100_000

t = np.linspace(0, SIM_DURATION, NUM_STEPS)

# -------------------------------------------------------
# DUAL METRIC JITTER FUNCTION
# -------------------------------------------------------
def compute_jitter_metrics(t, period_ns, sigma_ns):

    # Periodic jitter frequency
    jitter_freq = 1 / (10 * period_ns)

    # Periodic sinusoidal jitter applied ONLY to clock
    jitter_clk = sigma_ns * np.sin(2 * np.pi * jitter_freq * t)

    # Ideal external trigger
    t_ext = t

    # Jittered clock
    t_clk = t + jitter_clk

    # Generate waveforms
    ext = (((t_ext % period_ns) < (period_ns / 2))).astype(int)
    clk = (((t_clk % period_ns) < (period_ns / 2))).astype(int)

    # Rising edges
    ext_edges = np.where(np.diff(ext) == 1)[0]
    clk_edges = np.where(np.diff(clk) == 1)[0]

    num_edges = min(len(ext_edges), len(clk_edges))

    if num_edges < 5:
        return np.nan, np.nan

    ext_edge_times = t_ext[ext_edges[:num_edges]]
    clk_edge_times = t_clk[clk_edges[:num_edges]]

    # Ideal edge locations
    ideal_edges = np.arange(num_edges) * period_ns

    # Time Interval Error (TIE)
    tie_ext = ext_edge_times - ideal_edges
    tie_clk = clk_edge_times - ideal_edges

    # Relative jitter
    relative_jitter = tie_clk - tie_ext

    # ---------------------------------------------------
    # RMS JITTER
    # ---------------------------------------------------
    rms_relative = np.sqrt(np.mean(relative_jitter ** 2))

    # ---------------------------------------------------
    # PEAK-TO-PEAK JITTER
    # ---------------------------------------------------
    pp_relative = np.max(relative_jitter) - np.min(relative_jitter)

    return rms_relative, pp_relative

# -------------------------------------------------------
# MAIN SIMULATION
# -------------------------------------------------------
results = []

for freq in FREQ_LIST:

    print(f"\nRunning dual-metric periodic jitter simulation for {freq:.2f} MHz")

    period_ns = 1000 / freq

    for sigma in sigma_values:

        rms_vals = []
        pp_vals = []

        for _ in range(NUM_TRIALS):

            rms, p2p = compute_jitter_metrics(
                t,
                period_ns,
                sigma
            )

            rms_vals.append(rms)
            pp_vals.append(p2p)

        results.append({
            "Frequency (MHz)": freq,
            "Injected σ (ns)": sigma,
            "Measured RMS Jitter (ns)": np.nanmean(rms_vals),
            "Measured Peak-to-Peak Jitter (ns)": np.nanmean(pp_vals)
        })

# -------------------------------------------------------
# SAVE RESULTS
# -------------------------------------------------------
df = pd.DataFrame(results)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

filename = (
    f"DualMetric_PeriodicJitter_ClockOnly_"
    f"{NUM_STEPS}samples_"
    f"{NUM_TRIALS}runs_"
    f"{timestamp}.csv"
)

save_path = os.path.join(base_folder, filename)

df.to_csv(save_path, index=False)

print(f"\nDual-metric periodic jitter data exported to:\n{save_path}")

# -------------------------------------------------------
# PLOT
# -------------------------------------------------------
fig = go.Figure()

colors = ['blue', 'red', 'green', 'purple', 'orange', 'teal']

for i, freq in enumerate(FREQ_LIST):

    sub = df[df["Frequency (MHz)"] == freq]

    # RMS Plot
    fig.add_trace(go.Scatter(
        x=sub["Injected σ (ns)"],
        y=sub["Measured RMS Jitter (ns)"],
        mode='lines+markers',
        line=dict(
            width=3,
            color=colors[i % len(colors)]
        ),
        marker=dict(size=8),
        name=f"{freq:.2f} MHz (RMS)"
    ))

    # Peak-to-Peak Plot
    fig.add_trace(go.Scatter(
        x=sub["Injected σ (ns)"],
        y=sub["Measured Peak-to-Peak Jitter (ns)"],
        mode='lines+markers',
        line=dict(
            width=3,
            dash='dash',
            color=colors[i % len(colors)]
        ),
        marker=dict(size=8),
        name=f"{freq:.2f} MHz (P-P)"
    ))

fig.update_layout(
    title=(
        "Dual Metric Periodic Clock Jitter vs Injected σ"
        "<br>(Ideal External Trigger Reference)"
    ),
    xaxis_title="Injected Periodic Clock Jitter σ (ns)",
    yaxis_title="Measured Jitter (ns)",
    height=600,
    template="plotly_white"
)

fig.show()

# -------------------------------------------------------
# LINEAR FIT
# -------------------------------------------------------
print("\nExtracted Dual-Metric Periodic Jitter Relationships:")

for freq in FREQ_LIST:

    sub = df[df["Frequency (MHz)"] == freq]

    # RMS fit
    coeffs_rms = np.polyfit(
        sub["Injected σ (ns)"],
        sub["Measured RMS Jitter (ns)"],
        1
    )

    # Peak-to-Peak fit
    coeffs_pp = np.polyfit(
        sub["Injected σ (ns)"],
        sub["Measured Peak-to-Peak Jitter (ns)"],
        1
    )

    print(f"\n{freq:.2f} MHz")

    print(
        f"RMS ≈ "
        f"{coeffs_rms[0]:.3f}·σ + {coeffs_rms[1]:.3f}"
    )

    print(
        f"P-P ≈ "
        f"{coeffs_pp[0]:.3f}·σ + {coeffs_pp[1]:.3f}"
    )

## Metrics_relativejitter(Clock+extpulse)

### 4.3.1_RMS //

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import os
from datetime import datetime

base_folder = '/content/drive/MyDrive/Jitter_Simulations'
os.makedirs(base_folder, exist_ok=True)

NUM_FREQS = int(input("Enter number of frequencies to test: ") or 2)
FREQ_LIST = [float(input(f"Enter frequency {i+1} (MHz): ") or 40.0) for i in range(NUM_FREQS)]
NUM_TRIALS = int(input("Enter number of simulation runs per frequency: ") or 10)

range_choice = input("Do you want to run across a jitter amplitude range? (y/n): ").strip().lower()

if range_choice == 'y':
    JITTER_START = int(input("Enter starting periodic jitter amplitude (ns): ") or 1)
    JITTER_END = int(input("Enter ending periodic jitter amplitude (ns): ") or 5)
else:
    single_jitter = float(input("Enter single periodic jitter amplitude (ns): ") or 1.0)
    JITTER_START = JITTER_END = int(single_jitter)

jitter_values = list(range(JITTER_START, JITTER_END + 1))

SIM_DURATION = 100        # ns
NUM_STEPS = 100000
t = np.linspace(0, SIM_DURATION, NUM_STEPS)


def compute_rms_jitter(t, period_ns, jitter_amp_ns):
    # Periodic jitter frequency
    jitter_freq = 0.1  # cycles/ns

    # Periodic sinusoidal jitter
    jitter_clk = jitter_amp_ns * np.sin(2 * np.pi * jitter_freq * t)

    # Independent periodic jitter with phase offset
    phase_offset = np.pi / 3
    jitter_ext = jitter_amp_ns * np.sin(
        2 * np.pi * jitter_freq * t + phase_offset
    )

    t_clk = t + jitter_clk
    t_ext = t + jitter_ext

    clk = (((t_clk % period_ns) < (period_ns / 2))).astype(int)
    ext = (((t_ext % period_ns) < (period_ns / 2))).astype(int)

    # Rising edges
    clk_edges = np.where(np.diff(clk) == 1)[0]
    ext_edges = np.where(np.diff(ext) == 1)[0]

    num_edges = min(len(clk_edges), len(ext_edges))

    if num_edges < 5:
        return np.nan

    clk_edge_times = t_clk[clk_edges[:num_edges]]
    ext_edge_times = t_ext[ext_edges[:num_edges]]

    # Ideal edge times
    ideal_edges = np.arange(num_edges) * period_ns

    # Time Interval Error (TIE)
    tie_clk = clk_edge_times - ideal_edges
    tie_ext = ext_edge_times - ideal_edges

    # Relative jitter
    relative_jitter = tie_ext - tie_clk

    rms_relative = np.sqrt(np.mean(relative_jitter ** 2))

    return rms_relative


results = []

for freq in FREQ_LIST:
    print(f"\nRunning RMS jitter simulation for {freq:.2f} MHz")

    period_ns = 1000 / freq

    for jitter_amp in jitter_values:

        rms_vals = []

        for _ in range(NUM_TRIALS):
            rms = compute_rms_jitter(
                t,
                period_ns,
                jitter_amp
            )
            rms_vals.append(rms)

        results.append({
            "Frequency (MHz)": freq,
            "Injected Periodic Jitter Amplitude (ns)": jitter_amp,
            "Measured RMS Relative Jitter (ns)": np.nanmean(rms_vals)
        })


df = pd.DataFrame(results)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

filename = (
    f"RMS_Periodic_Jitter_TIE_"
    f"{NUM_STEPS}samples_"
    f"{NUM_TRIALS}runs_"
    f"{timestamp}.csv"
)

save_path = os.path.join(base_folder, filename)

df.to_csv(save_path, index=False)

print(f"\nRMS jitter data exported to:\n{save_path}")


fig = go.Figure()

colors = ['blue', 'red', 'green', 'purple', 'orange', 'teal']

for i, freq in enumerate(FREQ_LIST):

    sub = df[df["Frequency (MHz)"] == freq]

    fig.add_trace(go.Scatter(
        x=sub["Injected Periodic Jitter Amplitude (ns)"],
        y=sub["Measured RMS Relative Jitter (ns)"],
        mode='lines+markers',
        line=dict(
            width=3,
            color=colors[i % len(colors)]
        ),
        marker=dict(size=8),
        name=f"{freq:.2f} MHz"
    ))

fig.update_layout(
    title="RMS Relative Jitter vs Periodic Jitter Amplitude<br>"
          "(TIE-Based Measurement — Periodic Jitter)",
    xaxis_title="Injected Periodic Jitter Amplitude (ns)",
    yaxis_title="Measured RMS Relative Jitter (ns)",
    height=550,
    template="plotly_white"
)

fig.show()

print("\nExtracted RMS Jitter Relationship:")

for freq in FREQ_LIST:

    sub = df[df["Frequency (MHz)"] == freq]

    coeffs = np.polyfit(
        sub["Injected Periodic Jitter Amplitude (ns)"],
        sub["Measured RMS Relative Jitter (ns)"],
        1
    )

    print(
        f"{freq:.2f} MHz → "
        f"RMS ≈ {coeffs[0]:.3f}·PJ + {coeffs[1]:.3f}"
    )

### 4.3.2_Pk-Pk //

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import os
from datetime import datetime

# DRIVE SETUP
base_folder = '/content/drive/MyDrive/Jitter_Simulations'
os.makedirs(base_folder, exist_ok=True)

# USER INPUTS
NUM_FREQS = int(input("Enter number of frequencies to test: ") or 2)

FREQ_LIST = [
    float(input(f"Enter frequency {i+1} (MHz): ") or 40.0)
    for i in range(NUM_FREQS)
]

NUM_TRIALS = int(
    input("Enter number of simulation runs per frequency: ") or 10
)

range_choice = input(
    "Do you want to run across a periodic jitter amplitude range? (y/n): "
).strip().lower()

if range_choice == 'y':

    SIGMA_START = int(
        input("Enter starting periodic jitter amplitude (ns): ") or 1
    )

    SIGMA_END = int(
        input("Enter ending periodic jitter amplitude (ns): ") or 5
    )

else:

    single_sigma = float(
        input("Enter single periodic jitter amplitude (ns): ") or 1.0
    )

    SIGMA_START = SIGMA_END = int(single_sigma)

sigma_values = list(range(SIGMA_START, SIGMA_END + 1))

# SIMULATION GRID
SIM_DURATION = 100        # ns
NUM_STEPS = 100000       # high resolution

t = np.linspace(0, SIM_DURATION, NUM_STEPS)

# PEAK-TO-PEAK JITTER FUNCTION
def compute_ptp_jitter(t, period_ns, sigma_ns):

    # Periodic jitter parameters
    jitter_freq = 0.1      # cycles/ns
    phase_offset = np.pi / 3

    # PERIODIC JITTER
    jitter_clk = sigma_ns * np.sin(
        2 * np.pi * jitter_freq * t
    )

    jitter_ext = sigma_ns * np.sin(
        2 * np.pi * jitter_freq * t + phase_offset
    )

    # Apply jitter to time axis
    t_clk = t + jitter_clk
    t_ext = t + jitter_ext

    # Generate clocks
    clk = (
        ((t_clk % period_ns) < (period_ns / 2))
    ).astype(int)

    ext = (
        ((t_ext % period_ns) < (period_ns / 2))
    ).astype(int)

    # Detect rising edges
    clk_edges = np.where(np.diff(clk) == 1)[0]
    ext_edges = np.where(np.diff(ext) == 1)[0]

    num_edges = min(len(clk_edges), len(ext_edges))

    if num_edges < 5:
        return np.nan

    clk_edge_times = t_clk[clk_edges[:num_edges]]
    ext_edge_times = t_ext[ext_edges[:num_edges]]

    # Ideal edge locations
    ideal_edges = np.arange(num_edges) * period_ns

    # Time Interval Error (TIE)
    tie_clk = clk_edge_times - ideal_edges
    tie_ext = ext_edge_times - ideal_edges

    # Relative jitter
    relative_jitter = tie_ext - tie_clk

    # Peak-to-Peak jitter
    ptp_relative = (
        np.max(relative_jitter) -
        np.min(relative_jitter)
    )

    return ptp_relative


# MAIN SIMULATION
results = []

for freq in FREQ_LIST:

    print(
        f"\nRunning Peak-to-Peak periodic jitter simulation "
        f"for {freq:.2f} MHz"
    )

    period_ns = 1000 / freq

    for sigma in sigma_values:

        ptp_vals = []

        for _ in range(NUM_TRIALS):

            ptp = compute_ptp_jitter(
                t,
                period_ns,
                sigma
            )

            ptp_vals.append(ptp)

        results.append({
            "Frequency (MHz)": freq,
            "Injected Periodic Jitter Amplitude (ns)": sigma,
            "Measured Pk-Pk Jitter (ns)": np.nanmean(ptp_vals)
        })


# SAVE RESULTS
df = pd.DataFrame(results)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

filename = (
    f"PkPk_Periodic_Jitter_TIE_"
    f"{NUM_STEPS}samples_"
    f"{NUM_TRIALS}runs_"
    f"{timestamp}.csv"
)

save_path = os.path.join(base_folder, filename)

df.to_csv(save_path, index=False)

print(f"\nPeak-to-Peak jitter data exported to:\n{save_path}")


# PLOT RESULTS
fig = go.Figure()

colors = [
    'blue',
    'red',
    'green',
    'purple',
    'orange',
    'teal'
]

for i, freq in enumerate(FREQ_LIST):

    sub = df[df["Frequency (MHz)"] == freq]

    fig.add_trace(go.Scatter(
        x=sub["Injected Periodic Jitter Amplitude (ns)"],
        y=sub["Measured Pk-Pk Jitter (ns)"],
        mode='lines+markers',

        line=dict(
            width=3,
            color=colors[i % len(colors)]
        ),

        marker=dict(size=8),

        name=f"{freq:.2f} MHz"
    ))

fig.update_layout(
    title=(
        "Peak-to-Peak Relative Jitter vs "
        "Injected Periodic Jitter Amplitude<br>"
        "(TIE-Based Measurement)"
    ),

    xaxis_title="Injected Periodic Jitter Amplitude (ns)",

    yaxis_title="Measured Peak-to-Peak Relative Jitter (ns)",

    height=550,

    template="plotly_white"
)

fig.show()


# LINEAR FIT
print("\nExtracted Peak-to-Peak Jitter Relationship:")

for freq in FREQ_LIST:

    sub = df[df["Frequency (MHz)"] == freq]

    coeffs = np.polyfit(
        sub["Injected Periodic Jitter Amplitude (ns)"],
        sub["Measured Pk-Pk Jitter (ns)"],
        1
    )

    print(
        f"{freq:.2f} MHz → "
        f"Pk–Pk ≈ {coeffs[0]:.3f}·PJ + "
        f"{coeffs[1]:.3f}"
    )

### 4.3.3_rms&pkpk //

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from datetime import datetime

# =========================================================
# DRIVE SETUP
# =========================================================

base_folder = '/content/drive/MyDrive/Jitter_Simulations'
os.makedirs(base_folder, exist_ok=True)

# =========================================================
# USER INPUTS
# =========================================================

NUM_FREQS = int(
    input("Enter number of frequencies to test: ") or 2
)

FREQ_LIST = [
    float(input(f"Enter frequency {i+1} (MHz): ") or 40.0)
    for i in range(NUM_FREQS)
]

NUM_TRIALS = int(
    input("Enter number of simulation runs per frequency: ") or 10
)

range_choice = input(
    "Do you want to run across a periodic jitter amplitude range? (y/n): "
).strip().lower()

if range_choice == 'y':

    JITTER_START = int(
        input("Enter starting periodic jitter amplitude (ns): ") or 1
    )

    JITTER_END = int(
        input("Enter ending periodic jitter amplitude (ns): ") or 5
    )

else:

    single_jitter = float(
        input("Enter single periodic jitter amplitude (ns): ") or 1.0
    )

    JITTER_START = JITTER_END = int(single_jitter)

jitter_values = list(range(JITTER_START, JITTER_END + 1))

# =========================================================
# SIMULATION GRID
# =========================================================

SIM_DURATION = 100        # ns
NUM_STEPS = 100000

t = np.linspace(0, SIM_DURATION, NUM_STEPS)

# =========================================================
# JITTER MEASUREMENT FUNCTION
# =========================================================

def compute_jitter_metrics(t, period_ns, jitter_amp_ns):

    # -----------------------------------------------------
    # PERIODIC JITTER PARAMETERS
    # -----------------------------------------------------

    jitter_freq = 0.1      # cycles/ns
    phase_offset = np.pi / 3

    # -----------------------------------------------------
    # PERIODIC JITTER GENERATION
    # -----------------------------------------------------

    jitter_clk = jitter_amp_ns * np.sin(
        2 * np.pi * jitter_freq * t
    )

    jitter_ext = jitter_amp_ns * np.sin(
        2 * np.pi * jitter_freq * t + phase_offset
    )

    # -----------------------------------------------------
    # APPLY JITTER
    # -----------------------------------------------------

    t_clk = t + jitter_clk
    t_ext = t + jitter_ext

    # -----------------------------------------------------
    # CLOCK GENERATION
    # -----------------------------------------------------

    clk = (
        ((t_clk % period_ns) < (period_ns / 2))
    ).astype(int)

    ext = (
        ((t_ext % period_ns) < (period_ns / 2))
    ).astype(int)

    # -----------------------------------------------------
    # RISING EDGE DETECTION
    # -----------------------------------------------------

    clk_edges = np.where(np.diff(clk) == 1)[0]
    ext_edges = np.where(np.diff(ext) == 1)[0]

    num_edges = min(len(clk_edges), len(ext_edges))

    if num_edges < 5:
        return np.nan, np.nan

    clk_edge_times = t_clk[clk_edges[:num_edges]]
    ext_edge_times = t_ext[ext_edges[:num_edges]]

    # -----------------------------------------------------
    # IDEAL EDGE LOCATIONS
    # -----------------------------------------------------

    ideal_edges = np.arange(num_edges) * period_ns

    # -----------------------------------------------------
    # TIME INTERVAL ERROR (TIE)
    # -----------------------------------------------------

    tie_clk = clk_edge_times - ideal_edges
    tie_ext = ext_edge_times - ideal_edges

    # -----------------------------------------------------
    # RELATIVE JITTER
    # -----------------------------------------------------

    relative_jitter = tie_ext - tie_clk

    # -----------------------------------------------------
    # RMS JITTER
    # -----------------------------------------------------

    rms_relative = np.sqrt(
        np.mean(relative_jitter ** 2)
    )

    # -----------------------------------------------------
    # PEAK-TO-PEAK JITTER
    # -----------------------------------------------------

    ptp_relative = (
        np.max(relative_jitter) -
        np.min(relative_jitter)
    )

    return rms_relative, ptp_relative

# =========================================================
# MAIN SIMULATION
# =========================================================

results = []

for freq in FREQ_LIST:

    print(
        f"\nRunning periodic jitter simulation "
        f"for {freq:.2f} MHz"
    )

    period_ns = 1000 / freq

    for jitter_amp in jitter_values:

        rms_vals = []
        ptp_vals = []

        for _ in range(NUM_TRIALS):

            rms, ptp = compute_jitter_metrics(
                t,
                period_ns,
                jitter_amp
            )

            rms_vals.append(rms)
            ptp_vals.append(ptp)

        results.append({

            "Frequency (MHz)": freq,

            "Injected Periodic Jitter Amplitude (ns)": jitter_amp,

            "Measured RMS Jitter (ns)": np.nanmean(rms_vals),

            "Measured Pk-Pk Jitter (ns)": np.nanmean(ptp_vals)
        })

# =========================================================
# SAVE RESULTS
# =========================================================

df = pd.DataFrame(results)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

filename = (
    f"Periodic_Jitter_RMS_and_PkPk_"
    f"{NUM_STEPS}samples_"
    f"{NUM_TRIALS}runs_"
    f"{timestamp}.csv"
)

save_path = os.path.join(base_folder, filename)

df.to_csv(save_path, index=False)

print(f"\nSimulation data exported to:\n{save_path}")

# =========================================================
# PLOTS
# =========================================================

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "RMS Relative Jitter",
        "Peak-to-Peak Relative Jitter"
    )
)

colors = [
    'blue',
    'red',
    'green',
    'purple',
    'orange',
    'teal'
]

for i, freq in enumerate(FREQ_LIST):

    sub = df[df["Frequency (MHz)"] == freq]

    # -----------------------------------------------------
    # RMS PLOT
    # -----------------------------------------------------

    fig.add_trace(

        go.Scatter(

            x=sub["Injected Periodic Jitter Amplitude (ns)"],

            y=sub["Measured RMS Jitter (ns)"],

            mode='lines+markers',

            line=dict(
                width=3,
                color=colors[i % len(colors)]
            ),

            marker=dict(size=8),

            name=f"RMS {freq:.2f} MHz"

        ),

        row=1,
        col=1
    )

    # -----------------------------------------------------
    # Pk-Pk PLOT
    # -----------------------------------------------------

    fig.add_trace(

        go.Scatter(

            x=sub["Injected Periodic Jitter Amplitude (ns)"],

            y=sub["Measured Pk-Pk Jitter (ns)"],

            mode='lines+markers',

            line=dict(
                width=3,
                color=colors[i % len(colors)]
            ),

            marker=dict(size=8),

            name=f"PkPk {freq:.2f} MHz"

        ),

        row=1,
        col=2
    )

# =========================================================
# LAYOUT
# =========================================================

fig.update_layout(

    title=(
        "Periodic Relative Jitter Analysis<br>"
        "(TIE-Based RMS and Peak-to-Peak Measurement)"
    ),

    height=600,

    width=1200,

    template="plotly_white"
)

fig.update_xaxes(
    title_text="Injected Periodic Jitter Amplitude (ns)",
    row=1,
    col=1
)

fig.update_xaxes(
    title_text="Injected Periodic Jitter Amplitude (ns)",
    row=1,
    col=2
)

fig.update_yaxes(
    title_text="Measured RMS Jitter (ns)",
    row=1,
    col=1
)

fig.update_yaxes(
    title_text="Measured Peak-to-Peak Jitter (ns)",
    row=1,
    col=2
)

fig.show()

# =========================================================
# LINEAR FIT RESULTS
# =========================================================

print("\nExtracted Jitter Relationships:\n")

for freq in FREQ_LIST:

    sub = df[df["Frequency (MHz)"] == freq]

    # -----------------------------------------------------
    # RMS FIT
    # -----------------------------------------------------

    rms_coeffs = np.polyfit(
        sub["Injected Periodic Jitter Amplitude (ns)"],
        sub["Measured RMS Jitter (ns)"],
        1
    )

    # -----------------------------------------------------
    # Pk-Pk FIT
    # -----------------------------------------------------

    ptp_coeffs = np.polyfit(
        sub["Injected Periodic Jitter Amplitude (ns)"],
        sub["Measured Pk-Pk Jitter (ns)"],
        1
    )

    print(f"{freq:.2f} MHz")

    print(
        f"  RMS   ≈ "
        f"{rms_coeffs[0]:.3f}·PJ + "
        f"{rms_coeffs[1]:.3f}"
    )

    print(
        f"  PkPk  ≈ "
        f"{ptp_coeffs[0]:.3f}·PJ + "
        f"{ptp_coeffs[1]:.3f}"
    )

    print()

#

# Data Dependent Jitter